# MADRL CityLearn v3 — Tutorial Completo (Google Colab · A100 / H100)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/codex/fix-madrl-traceability-docs/examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb)

**Proyecto:** Multi-Agente de Aprendizaje por Refuerzo Profundo para gestion coordinada
de flexibilidad energetica, emisiones de CO2 y eficiencia economica en comunidades inteligentes.

**Caso de estudio:** 17 edificios reales de Iquitos, Peru · Dataset 2023-2025 · 26 304 pasos horarios.

| Parametro | Valor |
|---|---|
| Algoritmos | HAPPO · MASAC · MATD3 · MAAC |
| Escenarios | E1 (Flexibilidad / OE1) · E2 (CO2 / OE2) · E3 (Costos / OE3) |
| Entrenamiento | **Desde cero**, `LAUNCH_FULL_TRAINING=True`, `N_EPISODES=50` típico |
| Protocolo | `two_phase_happo_masac`: **Fase1** HAPPO+MASAC → **Fase2** MATD3+MAAC (6+6) |
| Episodios | 50 por corrida (reanudable con `--skip-completed`) · 8 760 pasos/episodio |
| Total steps | 438 000 por corrida · 5 256 000 en los 12 jobs |
| KPIs post-train | **OE completos** (`objectives.py`): OE1=36 flex · OE2=7 CO2 · OE3=11 costo (**54** total); distrito + edificio |
| Ranking | *signed relative gain* vs baseline; **OG** = media de scores E1/E2/E3 |
| Baseline | `evaluate_v2` embebido en corridas MADRL; benchmarks SB3 (7.6) opcionales/off |
| GPU objetivo | NVIDIA H100 (~26 vCPU) primario · A100-SXM4-80GB compatible · 167 GiB RAM (Colab Pro+ High-RAM) |

> **Requisito:** Runtime A100/H100 High-RAM. El notebook falla temprano si Colab entrega GPU insuficiente.

### Fuentes cientificas y de tesis usadas para el diseno

- CityLearn estandariza la evaluacion RL/MARL para demanda respuesta urbana: https://arxiv.org/abs/2012.10504
- CityLearn v2 y CityLearn Challenge documentan KPIs de flexibilidad, carbono y costo: https://escholarship.org/content/qt5t48x8xk/qt5t48x8xk.pdf y https://proceedings.mlr.press/v220/nweye23a.html
- HAPPO/HATRPO justifica actualizacion secuencial y trust region en MARL: https://openreview.net/forum?id=EcGGFkNTxdJ
- MAAC usa criticos centralizados con atencion para escalar agentes: https://proceedings.mlr.press/v97/iqbal19a.html
- MATD3 reduce sobreestimacion mediante doble critico centralizado: https://arxiv.org/abs/1910.01465
- MASAC se apoya en SAC y mezcla QMIX/CTDE: https://arxiv.org/abs/1812.05905 y https://arxiv.org/abs/1803.11485
- PyTorch CUDA y reproducibilidad: https://docs.pytorch.org/docs/stable/notes/cuda.html y https://docs.pytorch.org/docs/stable/notes/randomness.html
- Colab no garantiza tipo de GPU ni duracion; por eso se requiere checkpoint/estado recuperable: https://research.google.com/colaboratory/faq.html
- A100 40/80 GB, HBM y TF32: https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/a100/pdf/nvidia-a100-datasheet-us-nvidia-1758950-r4-web.pdf
- Tesis consultadas sobre RL/MARL energetico: Ross May PhD Dalarna 2023, Oxford residential flexibility thesis, Politecnico di Torino MARL building-energy thesis.


## Guia Rapida de Lanzamiento en Colab A100 / H100

> **Tiempo estimado:** 50 episodios/corrida en **H100 (~26 vCPU, primario)**; ~2× más lento en A100 (12 vCPU). Reanudable con `--skip-completed`. **two_phase:** Fase1 HAPPO+MASAC → Fase2 MATD3+MAAC; 6 jobs/fase; cuello = CPU de CityLearn.
> **Prerequisito:** Runtime A100/H100 antes de **1.1**. Sección 6 auto-ajusta hilos a las vCPU.
> **Defaults:** `LAUNCH_FULL_TRAINING=True`, `N_EPISODES=50`, `2.2/2.3` en `skip` (solo recuperación).

### Mapa de navegacion (orden obligatorio)

| Bloque | Celdas | Funcion |
|--------|--------|---------|
| Setup | **0.verify** → **1.1–1.5** | GPU A100/H100, clone/sync, venv 3.9, Drive |
| Rutas | **2.1** → **2.1b** [→ **2.1c** limpieza] | `OUTPUT_ROOT` auto (nuevo o resume). **2.2/2.3** opcionales (`skip` por defecto) |
| Validacion | **3.1** → **4.1** → **5.1** | Dataset, env, pesos recompensa |
| Config | **6.1** [→ **6.2** si `QUICK_TEST`] | Fuente unica de HPs + `EXECUTION_MODE=two_phase_happo_masac` |
| Lanzamiento | **7.0** → **7.1** → **7.2** | Helpers, dry-run 12 jobs, entrenamiento (`LAUNCH_FULL_TRAINING=True`) |
| Post | **7.3–7.7**, **8.x**, **9.x** | Monitor, auditoria, KPIs OE completos (54), ranking OG |

**Sync git (celda 1.2):** padre `codex/fix-madrl-traceability-docs` · CityLearn `codex/iquitos-distillation-madrl-docs` · external/MAAC `codex/integrar-limpieza-diagnosticos` · protocolo `two_phase_happo_masac_v3` only. La celda **1.2** verifica parches (MAAC cuda-sync, validacion de corrida completa, ImportError del launcher) y aborta si falta alguno.

**Flujo unico (desde cero o reanudar):** `1.1–1.5` → `2.1` → `2.1b` → `6.1` → `7.0` → `7.1` → `7.2`. La celda **2.1** decide si crea run nuevo o reanuda; **2.1b** solo verifica el plan. **No ejecutes 2.2/2.3** en el flujo desde cero.

**Garantias de persistencia al reanudar (Drive):**
- **No se crea carpeta nueva** al reconectar: `2.1` reutiliza el ultimo `madrl_v3_*`.
- **Jobs completos** (`results.json`) se **omiten** (`--skip-completed`).
- **Jobs interrumpidos** reanudan desde checkpoint `.pt`.
- **`timeseries.csv` / `trace.csv`**: append incremental + preload al reanudar.
- **Al completar 50 ep:** `results.json` + KPIs OE (distrito y edificio) + `artifact_audit.json`.
- **Escritura atomica** (tmp+replace) y checkpoints `.pt` intactos en Drive.

**Outputs en Drive:** carpeta canónica compartida (`1ihH6RqL2KpevfCQEUXj7PP1aS2QYssAX`) y workspace `MyDrive/MADRLCitytleranflexresdr/outputs/madrl_v3_<timestamp>/` (`{MADRL}/E{1,2,3}/`).

---

## Flujo A — Lanzamiento DESDE CERO (Colab, primera vez)

**Cuando usarlo:** Drive vacio (sin `madrl_v3_*`) o run completamente nuevo.

**No toques banderas** — defaults de `2.1` (`AUTO_RESUME_LATEST=True`, `FORCE_NEW_RUN=False`) y `LAUNCH_FULL_TRAINING=True` en **7.2**.

| Paso | Celda | Que hace | Que debes ver |
|------|-------|----------|---------------|
| 1 | **Runtime Colab** | Entorno > A100 o H100 | GPU detectada |
| 2 | **0.verify** | Verifica conexion al runtime | OK |
| 3 | **1.1** | GPU, CUDA, Python | `NVIDIA A100` o `H100` |
| 4 | **1.2** | Clone repo + submodulos | codigo en `/content/MADRLCitytleranflexresdr` |
| 5 | **1.2b** | Valida ramas git | OK |
| 6 | **1.3** | Instala deps en `.venv39-citylearn-v3` | pip OK |
| 7 | **1.4** | sys.path + smoke imports | imports OK |
| 8 | **1.5** | Monta Google Drive | Drive montado |
| 9 | **2.1** | Crea `OUTPUT_ROOT` nuevo | `MODO RUN : NUEVO run` |
| 10 | **2.1b** | Verifica plan 12 jobs | 12 × `PENDIENTE (fresh)` |
| 11 | **3.1** → **4.1** → **5.1** | Dataset, env, pesos (recomendado 1ª vez) | OK |
| 12 | **6.1** | HPs (`N_EPISODES=50`, `SIX_JOB_*`) | config impresa |
| 13 | **7.0** | Helpers launcher/monitor | OK |
| 14 | **7.1** | Preflight + dry-run 12 jobs | `dry-run OK` / `PASSED` |
| 15 | **7.2** | Entrenamiento two_phase (`LAUNCH_FULL_TRAINING=True`) | monitor por MADRL |
| 16 | **7.3** (auto si `AUTO_RUN_POST_TRAINING`) | Monitor + auditoria | progreso |
| 17 | **8.x / 9.x** | KPIs OE completos + ranking OG | `results.json` × 12 |

**Secuencia minima:**
```
0.verify -> 1.1 -> 1.2 -> 1.2b -> 1.3 -> 1.4 -> 1.5
-> 2.1 -> 2.1b -> 6.1 -> 7.0 -> 7.1 -> 7.2
-> 8.x -> 9.x
```

**Que se crea en Drive:**
```
outputs/madrl_v3_<timestamp>/
  HAPPO/E1/data/timeseries.csv   (crece episodio a episodio)
  HAPPO/E1/data/results.json     (KPIs OE: distrito + edificio)
  HAPPO/E1/checkpoints/*.pt
  MASAC/E1/ ... MATD3/ ... MAAC/ ...
  official_full_status.json
```

---

## Flujo B — REINICIO y REANUDACION (Colab se desconecto)

**Cuando usarlo:** Colab cerro sesion o quieres continuar un entrenamiento ya iniciado.

**No toques banderas** — `2.1` hace AUTO-RESUME del ultimo `madrl_v3_*`.

| Paso | Celda | Que hace | Que debes ver |
|------|-------|----------|---------------|
| 1 | **Runtime Colab** | Nuevo runtime A100/H100 | GPU detectada |
| 2 | **1.1 → 1.5** | Re-sync codigo + remontar Drive | mismo setup que Flujo A |
| 3 | **2.1** | Reutiliza ultimo run | `MODO RUN : AUTO-RESUME` |
| 4 | **2.1b** | Reporte por job (12 corridas) | ver tabla abajo |
| 5 | **6.1 → 7.0 → 7.1 → 7.2** | Resume two_phase con `--skip-completed` | mismo `OUTPUT_ROOT` |
| 6 | **2.2 / 2.3** *(opcional, `skip` por defecto)* | Rescue HAPPO / salvage KPI 49→50 | solo si aplica |
| 7 | **8.x / 9.x** | Analisis KPIs OE + ranking | 4 MADRL × 3 escenarios |

**Secuencia minima:**
```
1.1 -> 1.2 -> 1.2b -> 1.3 -> 1.4 -> 1.5
-> 2.1 -> 2.1b -> 6.1 -> 7.0 -> 7.1 -> 7.2
-> 8.x -> 9.x
```

**Atajo tras reconectar (mismo kernel setup ya hecho):** solo **7.2** (bootstrap: git sync + Drive + mismo `OUTPUT_ROOT` + `--skip-completed`).

> **2.2 / 2.3 (opcionales):** fuera del flujo desde cero. `RESCUE_MODE='skip'` y `HAPPO_KPI_MODE='skip'` por defecto. Activa solo si necesitas rescatar checkpoints HAPPO o completar salvage 49/50 sin KPIs. En reanudacion normal, **7.2** cubre tails.

**Que significa el reporte de 2.1b (por cada MADRL/E1..E3):**

| Estado en 2.1b | Que hace 7.2 | Archivos en Drive |
|----------------|--------------|-------------------|
| `OK COMPLETO 50/50 ep` | **OMITE** (`--skip-completed`) | Intactos (`results.json`, CSV, figuras) |
| `REANUDA ep X/50` | Carga `.pt`, continua ep X+1..50 | CSV se **continua appendeando** |
| `REANUDA 49/50` HAPPO salvage sin KPIs | Preferible **2.3** (`execute`); 7.2 tambien puede cubrir | Checkpoints intactos; falta KPI en `results.json` |
| `PENDIENTE (fresh)` | Arranca de cero ese job | Crea archivos nuevos en su carpeta |

**Garantias al reanudar:**
- Mismo `OUTPUT_ROOT` (sin carpeta `madrl_v3_*` duplicada).
- Episodios ya entrenados no se pierden (CSV incremental + preload).
- Jobs completos no se repiten (`--skip-completed`).
- A 50 ep: `results.json` + `artifact_audit.json` listos para **8.x/9.x** (OE1/OE2/OE3 completos).

**Opcional:** **2.1c** reporta runs duplicados (`DELETE_DUPLICATE_RUNS=True` para borrar).

**Casos especiales:**

```python
# Reanudar un run concreto (antes de 2.1):
RESUME_OUTPUT_ROOT = '/content/drive/MyDrive/MADRLCitytleranflexresdr/outputs/madrl_v3_20260626_004846'

# Empezar de cero ignorando runs viejos (antes de 2.1):
FORCE_NEW_RUN = True
```

---

### Recordatorio (sin duplicar el flujo)

- Runtime: **H100** (primario) o **A100** · High-RAM.
- HPs: celda **6.1** (`N_EPISODES=50`, `SIX_JOB_*`, `EXECUTION_MODE=two_phase_happo_masac`).
- Entrenamiento: **7.0 → 7.1 → 7.2** con `LAUNCH_FULL_TRAINING=True`.
- Analisis: **8.x / 9.x** — KPIs OE completos (54), niveles distrito + edificio; ranking por *signed relative gain*; OG = media E1/E2/E3.
- Baseline: `evaluate_v2` en cada corrida MADRL; **7.6** SB3 opcional (`RUN_CITYLEARN_V2_SB3_BENCHMARKS=False`).
- Artefactos: `{OUTPUT_ROOT}/{HAPPO|MASAC|MATD3|MAAC}/E{1,2,3}/` + `official_full_status.json`.


## Paso 0: Conectar VS Code al runtime A100/H100 de Google Colab

> Haz este paso **UNA SOLA VEZ** antes de ejecutar cualquier celda.
> No se necesita ngrok ni tunnels: la extension `google.colab` de VS Code
> maneja la conexion directamente con tu cuenta de Google.

---

### 0.1  Seleccionar el kernel Colab en VS Code

1. Abre este notebook en VS Code
   (`examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb`)
2. Haz clic en **"Select Kernel"** (esquina superior derecha del notebook)
3. En la lista elige **"Google Colab"**
4. Si pide autenticacion → cuenta Google con Colab Pro+ y espacio en Drive
5. Runtime: **"New Colab Server"** → acelerador **A100** (o H100 si disponible) → forma **High-RAM**
   *(NO elijas "Standard": ~40 GiB VRAM + ~83 GiB RAM no alcanza para 6 jobs/fase)*

> Si no ves "Google Colab": `Ctrl+Shift+P` → **"Colab: Sign In"**, autentica, repite.

---

### 0.2  Verificar la conexion

Ejecuta la celda de codigo siguiente (**0.verify**). Debe mostrar GPU A100/H100 con VRAM suficiente (~80 GiB) y RAM ~167 GiB en Colab High-RAM.
Si muestra otra GPU, menos VRAM o error → vuelve al paso 0.1.

---

### 0.3  Flujo de trabajo diario

```
VS Code (editor local)
       │
       │  google.colab extension
       ▼
Colab A100/H100 runtime
  /content/MADRLCitytleranflexresdr/   ← repo clonado en celda 1.2
  /content/drive/MyDrive/MADRLCitytleranflexresdr/outputs/  ← checkpoints en Drive
```

- El **codigo se ejecuta en el runtime Colab**.
- Los **outputs y graficas** aparecen en VS Code.
- Si Colab desconecta: repetir 0.1, luego Flujo B de la guia rapida (o atajo **7.2**).

> **Apertura directa en Colab (sin VS Code):** badge del titulo o
> [`madrl_citylearn_v3_tutorial.ipynb` en GitHub](https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/codex/fix-madrl-traceability-docs/examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb)
> · rama `codex/iquitos-distillation-madrl-docs`.


In [ ]:
# ── 0.verify  Verificar conexion al runtime (A100/H100 en Colab; local con advertencias) ────
import subprocess
import os
import sys
import platform

MIN_VRAM_GIB = 38.0   # config conservadora 6-jobs corre desde ~40 GiB (A100-40/80, H100, RTX PRO 6000)
MIN_RAM_GIB  = 64.0   # buffers conservadores: MASAC 3x12 + MATD3 3x14 GiB en RAM (replay CPU)
# GPUs datacenter conocidas que soportan TF32 + expandable_segments (perfil 'aws').
# La validacion real es por VRAM suficiente; el nombre solo informa.
_KNOWN_GPUS = ('A100', 'H100', 'H200', 'RTX PRO 6000', 'BLACKWELL', 'A40', 'L40', 'L4')

import importlib.util
IN_COLAB = importlib.util.find_spec('google.colab') is not None

def check_connection():
    _errors = []
    _warnings = []
    gpu_mem_gib = None
    mem_gib = None

    # 1. GPU — hard fail en Colab si no A100; advertencia local
    try:
        result = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
            text=True, stderr=subprocess.DEVNULL
        ).strip()
        gpu_name, gpu_mem = result.split(',')
        gpu_mem_gib = int(gpu_mem.strip()) / 1024.0
        vram_ok = gpu_mem_gib >= MIN_VRAM_GIB
        _name_known = any(k in gpu_name.upper() for k in _KNOWN_GPUS)
        gpu_ok = vram_ok  # requisito real = VRAM suficiente (cualquier GPU datacenter capaz)
        status = '[OK]' if gpu_ok else ('[WARN]' if not IN_COLAB else '[FAIL]')
        print(f"{status} GPU    : {gpu_name.strip()}  ({gpu_mem_gib:.1f} GiB VRAM)")
        if gpu_ok and not _name_known:
            print(f"     (GPU no listada pero {gpu_mem_gib:.0f} GiB VRAM >= {MIN_VRAM_GIB:.0f} -> apta; TF32 perfil 'aws')")
        if gpu_ok and any(k in gpu_name.upper() for k in ('BLACKWELL', 'RTX PRO 6000')):
            print('     [!] Blackwell sm_120: celda 1.3 instalara PyTorch cu128 (cu126 falla con no kernel image)')
        if not vram_ok:
            msg = f"VRAM insuficiente: {gpu_mem_gib:.1f} GiB < {MIN_VRAM_GIB:.0f} GiB minimos."
            if IN_COLAB:
                _errors.append(msg + " Selecciona A100/H100/RTX PRO 6000 (80 GiB) en Colab Pro+.")
            else:
                _warnings.append(msg + " Entorno local: se usara la GPU disponible o CPU.")
    except Exception as e:
        status = '[FAIL]' if IN_COLAB else '[--]'
        print(f"{status} GPU    : nvidia-smi no disponible ({e})")
        if IN_COLAB:
            _errors.append("nvidia-smi no disponible: no hay GPU o driver NVIDIA en Colab.")
        else:
            _warnings.append("nvidia-smi no disponible: entorno local sin GPU NVIDIA detectada.")

    # 2. RAM — hard fail en Colab si < 64 GiB (buffers conservadores); advertencia local
    try:
        if sys.platform.startswith('linux'):
            with open('/proc/meminfo') as f:
                for line in f:
                    if 'MemTotal' in line:
                        mem_gib = int(line.split()[1]) / (1024 * 1024)
                        ram_ok = mem_gib >= MIN_RAM_GIB
                        status = '[OK]' if ram_ok else ('[WARN]' if not IN_COLAB else '[FAIL]')
                        print(f"{status} RAM    : ~{mem_gib:.0f} GiB")
                        if not ram_ok:
                            msg = f"RAM insuficiente: {mem_gib:.0f} GiB < {MIN_RAM_GIB:.0f} GiB recomendados."
                            if IN_COLAB:
                                _errors.append(msg + " Activa 'A100 High-RAM' en Colab.")
                            else:
                                _warnings.append(msg + " MASAC puede requerir reducir replay_buffer_size.")
                        break
        else:
            import psutil
            mem_gib = psutil.virtual_memory().total / (1024**3)
            ram_ok = mem_gib >= MIN_RAM_GIB
            status = '[OK]' if ram_ok else '[WARN]'
            print(f"{status} RAM    : ~{mem_gib:.0f} GiB  (psutil, entorno local)")
    except Exception:
        print("[--] RAM    : No se pudo leer memoria del sistema")

    # 3. Python y plataforma
    print(f"[OK] Python : {sys.version.split()[0]}  ({platform.system()} {platform.machine()})")
    print(f"[OK] Entorno: {'Google Colab' if IN_COLAB else 'Local / otro'}")

    # 4. Google Drive (solo Colab)
    if IN_COLAB:
        drive_ok = os.path.exists('/content/drive/MyDrive')
        print(f"{'[OK]' if drive_ok else '[--]'} Drive  : {'montado en /content/drive/MyDrive' if drive_ok else 'no montado (ejecuta celda 1.5)'}")

    # 5. CUDA y PyTorch
    try:
        import torch
        cuda_ok = torch.cuda.is_available()
        if cuda_ok:
            print(f"[OK] CUDA   : {getattr(getattr(torch, 'version', None), 'cuda', None)}  device={torch.cuda.get_device_name(0)}")
        else:
            print("[INFO] CUDA : torch disponible pero CUDA no detectado — se usara CPU")
    except ImportError:
        print("[--] CUDA   : torch no instalado aun (normal antes de celda 1.3)")

    # ── Resultado final ──────────────────────────────────────────────────────
    for w in _warnings:
        print(f"  ⚠️  {w}")
    if _errors:
        # Diagnóstico: Pro+ puede entregar A100 Standard (~40 GiB VRAM, ~83 GiB RAM)
        _std_a100 = (
            IN_COLAB
            and gpu_mem_gib is not None and mem_gib is not None
            and gpu_mem_gib < MIN_VRAM_GIB
            and mem_gib < MIN_RAM_GIB
            and gpu_mem_gib >= 35 and mem_gib >= 75
        )
        if _std_a100:
            print()
            print("  ℹ️  Diagnóstico: estás en A100 *Standard*, no en A100 *High-RAM*.")
            print("      Detectado : ~40 GiB VRAM + ~83 GiB RAM")
            print("      Requerido : ~80 GiB VRAM + ~167 GiB RAM (MASAC buffer en CPU)")
            print("      VS Code   : Select Kernel → Colab → New Colab Server → A100 → High-RAM")
            print("      Colab web : Runtime → Cambiar tipo → A100 → activar High RAM")
            print("      Luego desconecta el runtime actual y vuelve a conectar.")
        print()
        for err in _errors:
            print(f"  ❌  {err}")
        raise RuntimeError(
            f"Pre-vuelo A100 fallo ({len(_errors)} error(es)). "
            "Corrige los problemas anteriores antes de continuar en Colab."
        )
    if IN_COLAB:
        print("\n✅  Runtime GPU + High-RAM listo para entrenamiento MADRL.")
    else:
        print("\n✅  Entorno local verificado. Advertencias anteriores son normales fuera de Colab.")

check_connection()


## Sección 0: Arquitectura del Proyecto — 9 Diagramas de Defensa

> Fuente canónica de arquitectura del experimento MADRL CityLearn v3.
> Renderizado: **mermaid.ink** (SVG estático) con fallback **Mermaid@10 CDN**. Ejecutar primero la celda **0.0**.

| Celda | Diagrama |
|-------|----------|
| 0.0 | Helper Mermaid + carga CDN |
| 0.1 | Visión general (inicio → resultado) |
| 0.2 | Pipeline del dataset Iquitos 2023–2025 |
| 0.3 | Arquitectura Dec-POMDP y CTDE — 17 agentes |
| 0.4 | Los 4 algoritmos MADRL: taxonomía |
| 0.5 | Flujo de entrenamiento: 12 corridas (two_phase: Fase1 HAPPO+MASAC → Fase2 MATD3+MAAC) |
| 0.6 | Recompensa multiobjetivo por escenario |
| 0.7 | Pipeline de evaluación: KPIs OE completos (54) + ranking OG |
| 0.8 | Infraestructura de cómputo: Local → Colab A100/H100 → AWS |
| 0.9 | Estructura de capas del repositorio |


In [ ]:
# ── 0.0  Helper Mermaid — renderiza los 9 diagramas de arquitectura ──────────
# Estrategia: mermaid.ink API (SVG estatico guardado en notebook) con fallback CDN
import json
import base64
import urllib.request
from IPython.display import display, HTML

_diagram_idx = [0]

def render_mermaid(title, code, height=520):
    """Renderiza diagrama Mermaid via mermaid.ink (estatico) o CDN (fallback)."""
    _diagram_idx[0] += 1
    uid = f"mmd_{_diagram_idx[0]}"

    # ── Intento 1: mermaid.ink API → SVG embebido en el output (offline despues) ──
    try:
        encoded = base64.urlsafe_b64encode(code.strip().encode("utf-8")).decode("utf-8")
        url = f"https://mermaid.ink/svg/{encoded}"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            svg = resp.read().decode("utf-8")
        html = f"""<div style="margin:16px 0;border:1px solid #e2e8f0;border-radius:10px;
padding:20px;background:#f8fafc;font-family:sans-serif;">
  <h4 style="margin:0 0 14px 0;color:#0f172a;font-size:14px;">{title}</h4>
  <div style="overflow:auto;max-height:{height + 80}px;">{svg}</div>
</div>"""
        display(HTML(html))
        return
    except Exception as _e:
        pass  # fallback below

    # ── Intento 2: CDN Mermaid@10 (requiere JS habilitado en el navegador) ──────
    code_js = json.dumps(code, ensure_ascii=False)
    html = f"""<div style="margin:16px 0;border:1px solid #e2e8f0;border-radius:10px;
padding:20px;background:#f8fafc;font-family:sans-serif;">
  <h4 style="margin:0 0 14px 0;color:#0f172a;font-size:14px;">{title}</h4>
  <div id="{uid}" style="min-height:{height}px;"></div>
  <script>
  (function(){{
    var el=document.getElementById("{uid}");
    el.textContent={code_js};
    el.className="mermaid";
    function tryR(){{
      if(window._mermaidReady&&typeof mermaid!=="undefined"){{
        try{{mermaid.run({{nodes:[el]}});}}catch(e){{console.error(e);}}
      }}else{{
        if(!window._mermaidCDNLoading){{
          window._mermaidCDNLoading=true;
          var s=document.createElement("script");
          s.src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js";
          s.onload=function(){{mermaid.initialize({{startOnLoad:false,theme:"default",securityLevel:"loose"}});window._mermaidReady=true;}};
          document.head.appendChild(s);
        }}
        setTimeout(tryR,400);
      }}
    }}
    tryR();
  }})();
  </script>
</div>"""
    display(HTML(html))

print("✅  Helper Mermaid listo (mermaid.ink + CDN fallback). Ejecuta celdas 0.1-0.9.")


In [ ]:
# ── 0.1  Diagrama 1 ────────────────────────────────────────
render_mermaid("Diagrama 1 — Vision General del Proyecto (inicio a fin)", r"""
flowchart LR
    subgraph ORIGEN["① Origen del proyecto"]
        direction TB
        PROB(["Problema de investigacion\n¿Que MADRL optimiza mejor\nflexibilidad + CO2 + costos\nen comunidades inteligentes?"])
        OBJ["Objetivos especificos\nOE1 Flexibilidad\nOE2 Emisiones CO2\nOE3 Costos energeticos"]
        PROB --> OBJ
    end

    subgraph DATOS["② Dataset Iquitos"]
        direction TB
        RAW["Facturas electricas reales\nCityLearn/data/buildingcsv/\n17 edificios B01-B17"]
        PIPE["Pipeline destilacion\nNSL residual + EV + BESS\ngenerate_iquitos_dataset.py"]
        DS[("citylearn_iquitos_2023_2025\nschema.json\n26 304 pasos horarios\n222 CSV activos")]
        RAW --> PIPE --> DS
    end

    subgraph SIM["③ Simulador"]
        direction TB
        V2["CityLearn v2 base\nFisica edificios\nBESS + PV + EV\nKPIs oficiales"]
        V3["CityLearn v3 propuesto\nDec-POMDP 17 agentes\nCTDE: critic centralizado\nRecompensa multiobjetivo"]
        V2 -->|"se extiende con\ncapa experimental"| V3
    end

    subgraph MADRL_BLOQUE["④ Entrenamiento MADRL"]
        direction TB
        ALGS["4 algoritmos\nHAPPO · MASAC\nMATD3 · MAAC"]
        EJES["3 escenarios\nE1 Flex · E2 CO2 · E3 Costo"]
        GPU["Colab A100 oficial (two_phase_happo_masac_v3)\n50 ep x 8 760 pasos = 438 000 steps/corrida\n12 jobs: Fase1 HAPPO+MASAC · Fase2 MATD3+MAAC\n~20 h wall (6 paralelos/fase, sin stagger)"]
        ALGS --> EJES --> GPU
    end

    subgraph EVAL["⑤ Evaluacion y seleccion"]
        direction TB
        KPIS["KPIs CityLearn v3 completos\n54 KPIs OE: OE1=36 · OE2=7 · OE3=11\ndistrito+edificio · citylearn/v3/objectives.py"]
        STAT["Pruebas estadisticas\nShapiro-Wilk\nKruskal-Wallis\nMann-Whitney U\nWilcoxon SR"]
        RANK["Ranking\nScore Ek = mean signed relative gain\nOG = media(E1,E2,E3)\nCliff delta + Hedges g"]
        KPIS --> STAT --> RANK
    end

    subgraph RESULT["⑥ Resultado"]
        direction TB
        MEJOR(["Ganador OG (ejemplo piloto v4)\nMATD3 · E1/E2/E3 ~0.75\nKW p=0.0459 — confirmar en Seccion 9"])
        TESIS["Evidencia para tesis\nTablas + graficas + conclusiones\ngenerate_thesis_objective_evidence.py"]
        MEJOR --> TESIS
    end

    ORIGEN --> DATOS
    DATOS --> SIM
    SIM --> MADRL_BLOQUE
    MADRL_BLOQUE --> EVAL
    EVAL --> RESULT

    classDef origen fill:#fef3c7,stroke:#d97706,color:#1c1917,stroke-width:2px
    classDef datos fill:#dbeafe,stroke:#2563eb,color:#1c1917,stroke-width:2px
    classDef sim fill:#e0e7ff,stroke:#4f46e5,color:#1c1917,stroke-width:2px
    classDef madrl fill:#fae8ff,stroke:#a21caf,color:#1c1917,stroke-width:2px
    classDef eval fill:#dcfce7,stroke:#16a34a,color:#1c1917,stroke-width:2px
    classDef result fill:#ffedd5,stroke:#ea580c,color:#1c1917,stroke-width:3px

    class ORIGEN origen
    class DATOS datos
    class SIM sim
    class MADRL_BLOQUE madrl
    class EVAL eval
    class RESULT result
""", height=520)


In [ ]:
# ── 0.2  Diagrama 2 ────────────────────────────────────────
render_mermaid("Diagrama 2 — Pipeline del Dataset Iquitos 2023-2025", r"""
flowchart TD
    subgraph INSUMOS["Insumos primarios (reales)"]
        FAC["Facturas electricas\nB02-B17.csv\nkWh punta / fuera punta\nGastos reales 2023-2025"]
        MET["Datos meteorologicos\nOpen-Meteo API\nGHI, T_amb, HR\nIquitos -3.74 lat"]
        AUD["Auditoria tecnica\nAreas techadas\nTipos HVAC\nFlota EV por edificio"]
    end

    subgraph PIPELINE["Pipeline de generacion (tools/)"]
        DEST["distill_building_loads.py\nNSL residual = E_medido - cooling/COP - DHW/COP\nBalance mensual < 0.1 percent error"]
        GEN["generate_iquitos_dataset.py\nInterpolacion horaria\nPronostico meses faltantes\ncalendar_month_mean_overlap_scaled"]
        SCHEMA["fix_schema_cooling.py\nAutosize safety factor\nchiller agua / multi-chiller\nprecision AC / ultra-freezers -80C"]
        VALID["orchestrate_citylearn_dataset.py\nIntegridad 222 CSV\n26 304 filas x edificio\ncharger NaN check"]
    end

    subgraph DATASET["Dataset final (CityLearn/data/datasets/)"]
        direction LR
        BUILD["Building_X.csv x17\nNSL + cooling + DHW\n26 304 pasos horarios"]
        WEATH["weather.csv\nGHI, T, HR, presion\nIquitos tropical"]
        CARBON["carbon_intensity.csv\n0.671-0.790 kgCO2/kWh\nMINAM RAGEI 2019"]
        PRICE["pricing.csv\nPunta 18-22h: 0.38 USD/kWh\nFuera punta: 0.26 USD/kWh"]
        EV["charger_X_Y.csv x185\n96 equipos Modo 3\n1 850 EV en pool"]
        SC["schema.json\n17 edificios registrados\nBESS + PV + EV por edificio"]
        BUILD --- WEATH --- CARBON --- PRICE --- EV --- SC
    end

    subgraph EDIFICIOS["17 edificios reales de Iquitos"]
        B["B01 ELECTRO ORIENTE 6747 kWh BESS\nB03 AEROPUERTO 2363 kWh BESS\nB06 MALL AVENTURA 2541 kWh BESS\nB07 UNAP BIOLOGIA 984 kWh BESS\nB11 HOSPITAL REGIONAL 1901 kWh BESS\n... 12 edificios mas"]
    end

    FAC --> DEST
    MET --> GEN
    AUD --> SCHEMA
    DEST --> VALID
    GEN --> VALID
    SCHEMA --> VALID
    VALID --> DATASET
    DATASET --> EDIFICIOS

    classDef ins fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef pipe fill:#e0f2fe,stroke:#0284c7,color:#1c1917
    classDef ds fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef edif fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    class INSUMOS ins
    class PIPELINE pipe
    class DATASET ds
    class EDIFICIOS edif
""", height=700)


In [ ]:
# ── 0.3  Diagrama 3 ────────────────────────────────────────
render_mermaid("Diagrama 3 — Arquitectura Dec-POMDP y CTDE de los 17 Agentes", r"""
flowchart TD
    subgraph ENV["Entorno CityLearn v3 (simulacion horaria)"]
        direction LR
        B1["Edificio 1\nBESS + PV + EV"]
        B2["Edificio 2\nBESS + PV + EV"]
        BN["... Edificio 17\nBESS + PV + EV"]
        GRID(["Red electrica\nElectro Oriente\nSistema aislado diesel"])
        B1 --- B2 --- BN
        B1 & B2 & BN --> GRID
    end

    subgraph OBS["Observaciones locales oᵢ(t) — 19 dimensiones"]
        direction LR
        TIME["Tiempo\nmes hora tipo_dia"]
        PHYS["Fisica edificio\nT_interior DHW\ncarga_no_desplazable\ngeneracion_solar"]
        BESS_OBS["Estado BESS\nSOC accion_previa"]
        EV_OBS["Estado EV\nSOC_k salida_k\nSOC_req_k llegada_k"]
        SIG["Senales globales\ncarbono precio\nGHI T_amb HR"]
    end

    subgraph POLICY["Politicas descentralizadas (ejecucion)"]
        P1["π₁(a₁|o₁)\nred neuronal\nedificio 1"]
        P2["π₂(a₂|o₂)\nred neuronal\nedificio 2"]
        PN["π₁₇(a₁₇|o₁₇)\nred neuronal\nedificio 17"]
    end

    subgraph ACTIONS["Acciones locales aᵢ(t)"]
        A1["a₁: BESS carga/descarga\nEV carga\nLavadora on/off"]
        A2["a₂: BESS · EV · Lavadora"]
        AN["a₁₇: BESS · EV · Lavadora"]
    end

    subgraph CRITIC["Critico centralizado (solo en entrenamiento CTDE)"]
        STATE["Estado global s = concat(o₁,...,o₁₇)\nV(s) o Q(s,a) centralizado"]
        REWARD["Recompensa mixta por agente\nr_i_mix = 0.30 * r_i + 0.70 * team_reward\nteam_reward = mean(r₁,...,r₁₇)"]
        STATE --> REWARD
    end

    subgraph UPDATE["Actualizacion de politicas (CTDE)"]
        GRAD["Gradiente con informacion global\nHAPPO: secuencial con trust region\nMASAC: Q-mix + SAC discreto\nMATD3: TD3 con critico centralizado\nMAC: attention sobre Q"]
    end

    ENV -->|"emite oᵢ(t)"| OBS
    OBS --> POLICY
    POLICY -->|"accion aᵢ"| ACTIONS
    ACTIONS -->|"aplica en entorno"| ENV
    ENV -->|"estado global\nsolo entrenamiento"| CRITIC
    CRITIC --> UPDATE
    UPDATE -->|"actualiza pesos"| POLICY

    classDef env fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    classDef obs fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef pol fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef crit fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef act fill:#ffedd5,stroke:#ea580c,color:#1c1917
    class ENV env
    class OBS obs
    class POLICY,ACTIONS pol
    class CRITIC,UPDATE crit
""", height=720)


In [ ]:
# ── 0.4  Diagrama 4 ────────────────────────────────────────
render_mermaid("Diagrama 4 — Los 4 Algoritmos MADRL: Taxonomia y Diferencias", r"""
flowchart LR
    subgraph HAPPO_BOX["HAPPO — Heterogeneous-Agent PPO"]
        direction TB
        HAPPO_T["Tipo: On-policy\nActualizacion secuencial\nTrust region por agente"]
        HAPPO_C["Critico: Centralizado V(s)\nActor: π(a|o) local\nBackend: HARL/external/HARL"]
        HAPPO_P["Parametros A100/H100 Colab\nhidden_size=512\nn_rollout_threads=auto (2 A100, 4 H100)\nlog_interval=1"]
    end

    subgraph MASAC_BOX["MASAC — Multi-Agent SAC Discreto"]
        direction TB
        MASAC_T["Tipo: Off-policy\nEntropy regularization\nAcciones discretas por eje"]
        MASAC_C["Critico: Q-mix centralizado\nActor: π(a|o) + temperatura\nBackend: MARL/external/MARL/src"]
        MASAC_P["Parametros SIX_JOB_* (celda 6.1)\naction_bins=3 axis mode\nbuffer=4 ep · max 8 GiB\ncritic_batch=1 · rnn/qmix/hyper 64/32/64\nreplay CPU · cuda_frac=0.16–0.22"]
    end

    subgraph MATD3_BOX["MATD3 — Multi-Agent TD3"]
        direction TB
        MATD3_T["Tipo: Off-policy\nDoble critico (anti-overest)\nPolicy delay + target noise"]
        MATD3_C["Critico: Par Q₁ Q₂ centralizado\nActor: μ(o) deterministico\nBackend: off-policy/external"]
        MATD3_P["Parametros SIX_JOB_* (celda 6.1)\nbatch=256 · buffer=4096\nhidden=256 (fase 2 estable)"]
    end

    subgraph MAAC_BOX["MAAC — Multi-Agent Attention Critic"]
        direction TB
        MAAC_T["Tipo: Off-policy\nAtencion sobre agentes\nSAC con Q de atencion"]
        MAAC_C["Critico: Attention SAC Q(s,a)\nActor: π(a|o) estocastico\nBackend: MAAC/external/MAAC"]
        MAAC_P["Parametros SIX_JOB_* (celda 6.1)\nbatch=768 · buffer=450K\nhidden=768 · num_updates=12"]
    end

    HAPPO_BOX -->|"12 jobs · 50 ep"| COMP(["Comparacion\n54 KPIs OE (objectives.py)\nvs baseline evaluate_v2"])
    MASAC_BOX --> COMP
    MATD3_BOX --> COMP
    MAAC_BOX --> COMP

    COMP -->|"OG = media E1/E2/E3"| WINNER(["Ganador OG (piloto v4: MATD3)\nScore Ek = mean signed relative gain\nconfirmar en Seccion 9"])

    classDef happo fill:#dbeafe,stroke:#2563eb,color:#1c1917,stroke-width:2px
    classDef masac fill:#fae8ff,stroke:#a21caf,color:#1c1917,stroke-width:2px
    classDef matd3 fill:#dcfce7,stroke:#16a34a,color:#1c1917,stroke-width:2px
    classDef maac fill:#fef3c7,stroke:#d97706,color:#1c1917,stroke-width:2px
    classDef winner fill:#ffedd5,stroke:#ea580c,color:#1c1917,stroke-width:3px
    class HAPPO_BOX happo
    class MASAC_BOX masac
    class MATD3_BOX matd3
    class MAAC_BOX maac
    class WINNER winner
""", height=640)


In [ ]:
# ── 0.5  Diagrama 5 ────────────────────────────────────────
render_mermaid("Diagrama 5 — Flujo de Entrenamiento: 12 Corridas (two_phase_happo_masac_v3)", r"""
flowchart TD
    START(["Colab A100-SXM4-80GB · protocolo two_phase_happo_masac_v3\ncolab_a100_official_launcher.py --execution-mode two_phase_happo_masac\n--scenario ALL --episodes 50 --episode-time-steps 8760\n438 000 steps/corrida · --skip-completed · OOM retry"])

    subgraph PHASE1["FASE 1 — HAPPO + MASAC (6 jobs en paralelo, sin stagger) · ~10 h"]
        direction LR
        subgraph P1_H["HAPPO x3 (on-policy)"]
            H_E1["E1 flex\nw: 0.70/0.15/0.15"]
            H_E2["E2 CO2\nw: 0.15/0.70/0.15"]
            H_E3["E3 costo\nw: 0.25/0.15/0.60"]
        end
        subgraph P1_M["MASAC x3 (off-policy, replay CPU)"]
            M_E1["E1\nbuf 4 ep · 8 GiB cap"]
            M_E2["E2\ncuda_frac 0.16–0.22"]
            M_E3["E3\nreplay CPU · critic_batch=1"]
        end
    end

    subgraph PHASE2["FASE 2 — MATD3 + MAAC (backfill dinámico: cada job entra al terminar uno de Fase 1, lightest-first) · ~10 h"]
        direction LR
        subgraph P2_T["MATD3 x3"]
            T_E1["E1\nbatch 256"]
            T_E2["E2\nbuf 4096"]
            T_E3["E3\nhidden 256"]
        end
        subgraph P2_A["MAAC x3"]
            A_E1["E1\nbatch 768"]
            A_E2["E2\nbuf 450K"]
            A_E3["E3\nupdates 12"]
        end
    end

    subgraph ARTEFACTOS["Artefactos por corrida (algorithm-first layout)"]
        direction LR
        PATH["outputs/<ts>/<MADRL>/<Escenario>/"]
        CHK["checkpoints/\nmodelos .pt"]
        DATA["data/\nresults.json · timeseries.csv\ntrace.csv · training_summary.json"]
        FIG["figures/\n13 graficas PNG"]
        LOG["logs/\nE?_algo.log · stderr"]
        PATH --- CHK --- DATA --- FIG --- LOG
    end

    subgraph STATUS["Protocolo, estado y monitoreo"]
        G1["colab_protocol_guard.py\nbloquea layout legacy 9+3"]
        S1["official_full_status.json\nofficial_full_manifest.json"]
        S2["live_progress.json\nepisodio · paso · reward · GPU"]
        MON["colab_a100_live_monitor.py\nprotocol=two_phase_happo_masac_v3"]
        G1 --- S1 --- S2 --- MON
    end

    START --> PHASE1
    PHASE1 -->|"backfill dinamico\n(Fase2 al liberar slot)"| PHASE2
    PHASE2 --> ARTEFACTOS
    PHASE1 & PHASE2 -->|"escribe en tiempo real"| STATUS

    classDef p1 fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef p2 fill:#dcfce7,stroke:#16a34a,color:#1c1917
    classDef art fill:#f8fafc,stroke:#64748b,color:#1c1917
    classDef stat fill:#fff7ed,stroke:#ea580c,color:#1c1917
    class PHASE1,P1_H,P1_M p1
    class PHASE2,P2_T,P2_A p2
    class ARTEFACTOS art
    class STATUS stat
""", height=780)


In [ ]:
# ── 0.6  Diagrama 6 ────────────────────────────────────────
render_mermaid("Diagrama 6 — Recompensa Multiobjetivo por Escenario", r"""
flowchart LR
    subgraph REW_FUNC["CityLearnV3MADRLRewardFunction"]
        direction TB
        COMP1["Componente FLEX\npeak_penalty + ramping_penalty\n+ load_factor + ev_service"]
        COMP2["Componente CO2\ncarbon_emissions\n* carbon_intensity_signal"]
        COMP3["Componente COSTO\nelectricity_cost\n* price_signal"]
        COMP4["Componente EV\nEV SOC deficit\n* urgencia_salida\n1 / horas_restantes"]
        COMP5["Componente BESS v4\nC-rate penalty\nArrhenius LiFePO4\ndegradacion ciclica"]
    end

    subgraph PESOS["Pesos por escenario (w_eje)"]
        direction TB
        PE1["E1 Flexibilidad\nflex=0.70 co2=0.15 cost=0.15"]
        PE2["E2 CO2\nflex=0.15 co2=0.70 cost=0.15"]
        PE3["E3 Costos\nflex=0.25 co2=0.15 cost=0.60"]
    end

    subgraph MIX["Recompensa mixta (CTDE)"]
        TEAM["team_reward = mean(r₁...r₁₇)\ncooperacion distrital"]
        MIXED["r_i_mix = 0.30 * r_i + 0.70 * team_reward\nteam_ratio=0.70"]
        TEAM --> MIXED
    end

    subgraph PERFILES["Perfiles por algoritmo (v4)"]
        PH["happo_unified_comparable_v4\npeak_weight=0.45 ramp_weight=0.35\nev_weight=0.25 reward_scale=1.00"]
        PM["masac_unified_comparable_v4"]
        PT["matd3_unified_comparable_v4"]
        PA["maac_unified_comparable_v4"]
    end

    REW_FUNC --> PESOS
    PESOS --> MIX
    MIX --> PERFILES

    PERFILES -->|"mismos pesos base\ndiferente backend"| TRAIN(["Entrenamiento\nuniforme y comparable\npara todos los algoritmos"])

    classDef rw fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef pe fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef mx fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef pf fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    class REW_FUNC rw
    class PESOS pe
    class MIX mx
    class PERFILES pf
""", height=620)


In [ ]:
# ── 0.7  Diagrama 7 ────────────────────────────────────────
render_mermaid("Diagrama 7 — Pipeline de Evaluacion y Seleccion del Mejor MADRL", r"""
flowchart TD
    subgraph ARTIFACTS_IN["Entrada: artefactos de 12 corridas (50 ep · two_phase)"]
        direction LR
        R_J["results.json\npor cada algo/escenario"]
        TS["timeseries.csv\npor cada algo/escenario"]
        TR["trace.csv\npor cada algo/escenario"]
    end

    subgraph BENCHMARK["Baseline MADRL (evaluate_v2)"]
        direction TB
        RBC["RBC CityLearn v2\nembebido en cada corrida\n(evaluate_v2 / axis_baseline)"]
        SB3["SB3 PPO/SAC/A2C\ncelda 7.6 opcional/off\nNO es baseline del ranking"]
        BENCH_OUT["objective_kpis +\nbaseline_gain_by_kpi\nen results.json"]
        RBC --> BENCH_OUT
        SB3 -.->|"comparativo extra"| BENCH_OUT
    end

    subgraph KPIS_CALC["KPIs OE completos (54)"]
        direction TB
        E1_KPI["E1 → OE1 flex\n36 KPIs · distrito+edificio\nOE1_FLEXIBILITY_KPIS"]
        E2_KPI["E2 → OE2 CO2\n7 KPIs · distrito+edificio\nOE2_EMISSIONS_KPIS"]
        E3_KPI["E3 → OE3 costo\n11 KPIs · distrito+edificio\nOE3_COST_KPIS"]
    end

    subgraph DELTA["Signed relative gain vs baseline"]
        GAIN["gain = signed relative gain\npositivo = mejora\nmedia sobre TODOS los KPIs del OE"]
    end

    subgraph STAT_TEST["Suite de pruebas estadisticas (4 tests)"]
        direction LR
        SW["Shapiro-Wilk\n¿Distribucion normal?\npor algoritmo y eje"]
        KW["Kruskal-Wallis\n¿Diferencia global?\n4 grupos simultaneos"]
        MWU["Mann-Whitney U\n¿Cual par difiere?\nmuestras independientes\n+ Cliff delta + Hedges g"]
        WC["Wilcoxon SR\n¿Diferencia sistematica?\nmuestras pareadas"]
        SW --> KW --> MWU --> WC
    end

    subgraph RANKING["Seleccion OG / OE.1 / OE.2 / OE.3"]
        direction TB
        SCORE_E1["Ganador OE.1\nmejor score E1 (36 KPIs)"]
        SCORE_E2["Ganador OE.2\nmejor score E2 (7 KPIs)"]
        SCORE_E3["Ganador OE.3\nmejor score E3 (11 KPIs)"]
        GLOBAL["Ganador OG\nmedia(E1,E2,E3)"]
        SCORE_E1 & SCORE_E2 & SCORE_E3 --> GLOBAL
    end

    subgraph RESULT_BOX["Resultado (ejemplo piloto v4 — no fija el run actual)"]
        direction TB
        MATD3_WIN["Piloto v4: MATD3 lider OG\nE1~0.75 E2~0.75 E3~0.73\nKW p=0.0459 — confirmar en 9.x"]
        PAIRS["Pruebas: Shapiro + KW + MWU\nCliff delta / Hedges g\nsobre scores de escenario"]
        MATD3_WIN --> PAIRS
    end

    ARTIFACTS_IN --> KPIS_CALC
    BENCHMARK --> KPIS_CALC
    KPIS_CALC --> DELTA
    DELTA --> STAT_TEST
    STAT_TEST --> RANKING
    RANKING --> RESULT_BOX

    subgraph OUTPUT_FILES["Archivos de salida (outputs/thesis_objective_evidence/)"]
        OF1["analisis_estadistico_madrl.csv\nSW + KW por eje"]
        OF2["comparaciones_mwu_madrl.csv\nMWU + effect sizes"]
        OF3["comparaciones_wilcoxon_madrl.csv\nWilcoxon SR pareado"]
        OF4["hipotesis_estadisticas_madrl.csv\n4 tests unificados"]
        OF5["scores_kpi_algoritmo_madrl.csv\nranking final"]
    end

    RESULT_BOX --> OUTPUT_FILES

    classDef inp fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef bench fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    classDef kpi fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef stat fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef rank fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef res fill:#ffedd5,stroke:#ea580c,color:#1c1917,stroke-width:3px
    classDef out fill:#f8fafc,stroke:#64748b,color:#1c1917
    class ARTIFACTS_IN inp
    class BENCHMARK bench
    class KPIS_CALC,DELTA kpi
    class STAT_TEST stat
    class RANKING rank
    class RESULT_BOX res
    class OUTPUT_FILES out
""", height=700)


In [ ]:
# ── 0.8  Diagrama 8 ────────────────────────────────────────
render_mermaid("Diagrama 8 — Infraestructura de Despliegue: Local, Colab A100 y AWS EC2", r"""
flowchart LR
    subgraph COLAB["Colab A100 Pro+ (canal oficial de entrenamiento)"]
        direction TB
        BADGE["Open in Colab\nMac-Tapia/CityLearn\ncodex/iquitos-distillation-madrl-docs"]
        SYNC["Celda 1.2 hard sync\n/content/MADRLCitytleranflexresdr"]
        LAUNCH_C["colab_a100_official_launcher.py\ntwo_phase_happo_masac_v3\n50 ep · 12 jobs"]
        DRIVE["Google Drive outputs\nMADRLCitytleranflexresdr/outputs"]
        MON_C["colab_a100_live_monitor.py\ncolab_protocol_guard.py"]
        BADGE --> SYNC --> LAUNCH_C --> DRIVE
        LAUNCH_C --> MON_C
    end

    subgraph DEV["Desarrollo local (Windows — RTX 4060)"]
        direction TB
        CODE["Codigo fuente\nCityLearn/ + uc3m/\nscripts/ + tools/"]
        VENV["Entorno .venv39-citylearn-v3\nPyTorch cu126 (A100/H100)\ncu128 auto en Blackwell"]
        PS["Launcher local PS1\nsmoke test rapido"]
        MON_L["monitor_citylearn_v3_official_training.ps1"]
        CODE --> VENV --> PS --> MON_L
    end

    subgraph GIT["Repositorio GitHub"]
        REPO_P["Mac-Tapia/MADRLCitytleranflexresdr\ncodex/fix-madrl-traceability-docs"]
        REPO_CL["Mac-Tapia/CityLearn\ncodex/iquitos-distillation-madrl-docs"]
        REPO_P --- REPO_CL
    end

    subgraph AWS["Produccion AWS EC2 (opcional)"]
        direction TB
        DOCKER["Docker madrl-training\nrun_aws_training.sh"]
        VOL["bind mount outputs/"]
        DOCKER --> VOL
    end

    DEV -->|"git push"| GIT
    GIT -->|"clone + submodule"| COLAB
    GIT -->|"clone --recurse-submodules"| AWS
    COLAB -->|"artefactos Drive"| DEV

    classDef colab fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef dev fill:#e0f2fe,stroke:#0284c7,color:#1c1917
    classDef git fill:#f0fdf4,stroke:#16a34a,color:#1c1917
    classDef aws fill:#fef3c7,stroke:#d97706,color:#1c1917
    class COLAB colab
    class DEV dev
    class GIT git
    class AWS aws
""", height=720)


In [ ]:
# ── 0.9  Diagrama 9 ────────────────────────────────────────
render_mermaid("Diagrama 9 — Estructura de Capas del Software", r"""
flowchart TD
    subgraph L1["Capa 1: Simulador base (CityLearn v2)"]
        direction LR
        CL2["CityLearn/citylearn/*.py\nFisica edificios + BESS + PV + EV\nKPIs oficiales del challenge"]
    end

    subgraph L2["Capa 2: Extension experimental (CityLearn v3 propuesto)"]
        direction LR
        ENV3["CityLearn/citylearn/v3/\nDec-POMDP environment\nObjectives + Config + Reward"]
        COMM["CityLearn/scripts/\ncitylearn_v3_training_common.py\nresolve_output_dir() + ensure_artifact_layout()"]
        ENV3 --- COMM
    end

    subgraph L3["Capa 3: Framework UC3M (wrapper universal)"]
        direction LR
        UC3M_E["uc3m/env/uc3m_env.py\nUC3MEnv: Dec-POMDP 11-aria\nCompatible HARL + MARLlib + RLlib"]
        UC3M_B["uc3m/env/bact.py\nBACTTensor 29D\nClima(7)+Geo(8)+Fisico(14)"]
        UC3M_R["uc3m/reward/axes.py\nRewardAxes 7 ejes\nflex+co2+cost+ev+bess+resil+acs"]
        UC3M_H["uc3m/reward/hphi.py\nHPHI: Holistic Pareto\nHypervolume Index 7D"]
        UC3M_K["uc3m/kpis/evaluator.py\nKPIEvaluator\nnormalizados contra RBC"]
        UC3M_E --- UC3M_B --- UC3M_R --- UC3M_H --- UC3M_K
    end

    subgraph L4["Capa 4: Backends MADRL externos"]
        direction LR
        HARL["external/HARL/\nHAPPO: on-policy\nsequential trust region"]
        MARL["external/MARL/src/\nMASAC: Q-mix + SAC discreto"]
        OFFP["external/off-policy/\nMATD3: doble critico TD3"]
        MAAC_B["external/MAAC/\nMAC: attention critic SAC"]
        HARL --- MARL --- OFFP --- MAAC_B
    end

    subgraph L5["Capa 5: Launchers y orquestacion"]
        direction LR
        TRAIN_S["CityLearn/scripts/train_citylearn_v3_*.py\n4 scripts de entrenamiento\nuno por algoritmo"]
        COLAB_L["CityLearn/scripts/colab_a100_official_launcher.py\ncolab_a100_live_monitor.py\ncolab_protocol_guard.py · two_phase_happo_masac_v3"]
        LAUNCH["scripts/ + deploy/aws/training/\nLaunchers locales PS + AWS bash\nMonitoreo + checkpointing"]
        TRAIN_S --- COLAB_L --- LAUNCH
    end

    subgraph L6["Capa 6: Evaluacion y evidencia"]
        direction LR
        GEN["CityLearn/scripts/generate_thesis_objective_evidence.py\nKPIs + estadisticas + figuras"]
        BENCH["CityLearn/scripts/benchmark_citylearn_v2_agents.py\nLinea base RBC + SAC v2"]
        COMP["CityLearn/scripts/compare_citylearn_v2_vs_v3_madrl.py\nDelta + ranking + HPHI"]
        GEN --- BENCH --- COMP
    end

    L1 -->|"extiende"| L2
    L2 -->|"wrap universal"| L3
    L3 -->|"conecta"| L4
    L4 -->|"invocado por"| L5
    L5 -->|"genera artefactos para"| L6

    classDef l1 fill:#f1f5f9,stroke:#64748b,color:#1c1917
    classDef l2 fill:#e0e7ff,stroke:#4f46e5,color:#1c1917
    classDef l3 fill:#dbeafe,stroke:#2563eb,color:#1c1917
    classDef l4 fill:#fae8ff,stroke:#a21caf,color:#1c1917
    classDef l5 fill:#fef3c7,stroke:#d97706,color:#1c1917
    classDef l6 fill:#dcfce7,stroke:#16a34a,color:#1c1917
    class L1 l1
    class L2 l2
    class L3 l3
    class L4 l4
    class L5 l5
    class L6 l6
""", height=700)


## Sección 1: Configuración inicial

Celdas **1.1–1.5**: GPU/CUDA, clone+sync git (protocolo `two_phase_happo_masac_v3`), venv Python 3.9, `sys.path`/imports y montaje de Google Drive. Obligatorias antes de **2.1** y del entrenamiento.


In [ ]:
# ── 1.1  Verificar entorno: IN_COLAB, GPU, CUDA, Python 3.9 ─────────────────
import sys
import subprocess
import importlib.util

# ── Deteccion automatica de entorno ──────────────────────────────────────────
IN_COLAB = importlib.util.find_spec('google.colab') is not None

print(f"Ejecutando en Google Colab : {IN_COLAB}")
print(f"Python version             : {sys.version.split()[0]}")
print(f"Plataforma                 : {sys.platform}")

if not sys.version_info[:2] == (3, 9):
    msg = (
        f"Python {sys.version.split()[0]} detectado; el proyecto usa Python 3.9.25. "
        "El venv .venv39-citylearn-v3 garantiza la version correcta."
    )
    if IN_COLAB:
        print(f"[WARN] {msg}")
    else:
        print(f"[INFO] {msg} (normal si el kernel de VS Code usa otra version)")

# ── GPU via nvidia-smi ───────────────────────────────────────────────────────
res = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
if res.returncode == 0:
    print(f"GPU                        : {res.stdout.strip()}")
else:
    if IN_COLAB:
        raise RuntimeError(
            "nvidia-smi fallo. Habilita el runtime GPU A100 antes de ejecutar esta celda."
        )
    else:
        print("GPU                        : nvidia-smi no disponible — ejecuta en Colab A100-SXM4-80GB")

# ── Verificacion PyTorch + CUDA ───────────────────────────────────────────────
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"PyTorch version            : {torch.__version__}")
    print(f"CUDA disponible            : {cuda_ok}")
    if cuda_ok:
        name = torch.cuda.get_device_name(0)
        mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
        vram_free = torch.cuda.mem_get_info(0)[0] / 1024**3
        print(f"Dispositivo GPU            : {name}")
        print(f"VRAM total                 : {mem:.1f} GiB")
        print(f"VRAM libre inicial         : {vram_free:.1f} GiB")
        torch.cuda.empty_cache()
        _known_gpu = any(k in name.upper() for k in ('A100','H100','H200','RTX PRO 6000','BLACKWELL','A40','L40','L4'))
        if _known_gpu or mem >= 38.0:
            print(f"[OK] {name} detectado — TF32 + expandable_segments activos")
            if mem >= 78.0:
                print(f"     {mem:.0f} GiB VRAM: holgado para 6 jobs/fase. La Seccion 6 auto-ajusta hilos a las vCPU.")
        else:
            if IN_COLAB:
                raise RuntimeError(
                    f"GPU detectada: {name} ({mem:.0f} GiB). Se requiere >=38 GiB VRAM (A100/H100/RTX PRO 6000). "
                    "Cambia el runtime: Entorno de ejecucion > Cambiar tipo > A100 / H100."
                )
            else:
                print(f"[WARN] GPU: {name} ({mem:.0f} GiB < 38). Este notebook esta optimizado para GPUs >=40 GiB.")
    else:
        if IN_COLAB:
            raise RuntimeError(
                "CUDA no disponible. Selecciona runtime A100 en Colab y vuelve a ejecutar."
            )
        else:
            print("[INFO] CUDA no disponible — se usara CPU (entorno local). El entrenamiento sera lento.")
except ImportError:
    print("[INFO] torch no disponible en kernel Python. La celda 1.3 lo instala en .venv39.")
    print("       La verificacion GPU (nvidia-smi) confirma que el hardware esta presente.")


In [ ]:
# ── 1.1b  Diagnóstico de recursos: vCPU / RAM / GPU + recomendación de hilos ──────
# El cuello de botella de MADRL+CityLearn es la CPU (env.step de 17 edificios),
# NO la GPU (las redes son pequeñas para un A100). Esta celda mide los recursos
# reales del runtime y recomienda torch_threads / n_rollout_threads para el
# esquema de 6 jobs en paralelo por fase. Es de solo lectura: puedes correrla
# mientras el entrenamiento avanza sin afectarlo.
import os
import subprocess
import shutil

PHASE_JOBS      = 6   # Fase 1 = 3 HAPPO + 3 MASAC ; Fase 2 = 3 MATD3 + 3 MAAC
CUR_TORCH       = 2   # --torch-threads / --two-phase-torch-threads (default actual)
CUR_HAPPO_ROLL  = 2   # --happo-n-rollout-threads (SubprocVecEnv, default actual)

# ── vCPUs realmente utilizables ──────────────────────────────────────────────
try:
    _sched_affinity = getattr(os, 'sched_getaffinity', None)
    if _sched_affinity is not None:
        usable_cpus = len(_sched_affinity(0))
    else:
        usable_cpus = os.cpu_count() or 1
except Exception:
    usable_cpus = os.cpu_count() or 1
total_cpus = os.cpu_count() or usable_cpus
print(f"vCPUs utilizables          : {usable_cpus}  (os.cpu_count={total_cpus})")

# ── RAM total / disponible ────────────────────────────────────────────────────
ram_total = ram_avail = None
try:
    import psutil
    vm = psutil.virtual_memory()
    ram_total, ram_avail = vm.total / 1024**3, vm.available / 1024**3
except Exception:
    try:
        with open("/proc/meminfo") as fh:
            info = {line.split(":")[0]: line.split()[1] for line in fh if ":" in line}
        ram_total = int(info["MemTotal"]) / 1024**2
        ram_avail = int(info.get("MemAvailable", info["MemFree"])) / 1024**2
    except Exception:
        pass
if ram_total is not None:
    print(f"RAM total / disponible     : {ram_total:.0f} GiB / {ram_avail:.0f} GiB")
else:
    print("RAM total / disponible     : (psutil/proc no disponibles)")

# ── GPU en vivo (nombre, VRAM usada/total, utilización) ───────────────────────
if shutil.which("nvidia-smi"):
    res = subprocess.run(
        ["nvidia-smi",
         "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    if res.returncode == 0:
        for line in res.stdout.strip().splitlines():
            name, used, tot, util = [x.strip() for x in line.split(",")]
            frac = 100 * float(used) / float(tot) if float(tot) else 0
            print(f"GPU                        : {name}")
            print(f"VRAM usada / total         : {used} / {tot} MiB ({frac:.1f}%)")
            print(f"Utilización GPU            : {util}%")
else:
    print("GPU                        : nvidia-smi no disponible")

# ── Demanda de CPU del esquema actual y recomendación ─────────────────────────
# HAPPO (on-policy): torch_threads + n_rollout_threads (cada worker SubprocVecEnv ~1 core)
# MASAC (off-policy): torch_threads (+ replay en RAM)  → fase 1 = 3 HAPPO + 3 MASAC
demand_now = 3 * (CUR_TORCH + CUR_HAPPO_ROLL) + 3 * CUR_TORCH
ratio = demand_now / usable_cpus if usable_cpus else float("inf")
print("\n── Análisis de paralelismo (Fase 1: 3 HAPPO + 3 MASAC) ──")
print(f"Demanda de hilos actual    : {demand_now}  "
      f"(HAPPO 3×{CUR_TORCH + CUR_HAPPO_ROLL} + MASAC 3×{CUR_TORCH})")
print(f"vCPUs utilizables          : {usable_cpus}")
print(f"Ratio demanda/CPU          : {ratio:.2f}×  "
      + ("→ SOBRE-SUSCRITO (la CPU es el cuello)" if ratio > 1.15
         else "→ equilibrado" if ratio >= 0.85
         else "→ CPU OCIOSA (puedes subir hilos)"))

# Recomendación por fase: la demanda total de cada fase debe caber en las vCPUs.
# Fase 1 (HAPPO+MASAC): HAPPO gasta torch+rollout, MASAC solo torch ->
#   3*(p1_torch + rollout) + 3*p1_torch <= vCPUs.  Con rollout fijo=2 buscamos p1_torch.
# Fase 2 (MATD3+MAAC): single-env sin rollout -> 6*p2_torch <= vCPUs.
ROLLOUT_FIXED = CUR_HAPPO_ROLL
p1 = 1
while 3 * ((p1 + 1) + ROLLOUT_FIXED) + 3 * (p1 + 1) <= usable_cpus:
    p1 += 1
p2 = max(1, usable_cpus // PHASE_JOBS)
d1 = 3 * (p1 + ROLLOUT_FIXED) + 3 * p1
d2 = PHASE_JOBS * p2
print("\n── Recomendación robusta por fase (sin riesgo de OOM) ──")
print(f"Fase 1  --two-phase-p1-torch-threads : {p1}  "
      f"(rollout={ROLLOUT_FIXED}) -> demanda {d1}/{usable_cpus} vCPU")
print(f"Fase 2  --two-phase-p2-torch-threads : {p2}  "
      f"(sin rollout)  -> demanda {d2}/{usable_cpus} vCPU")
print(f"HAPPO   --happo-n-rollout-threads    : {ROLLOUT_FIXED}")
print("RAM holgada → buffers off-policy actuales son seguros; "
      "GPU subutilizada es NORMAL en MADRL (redes pequeñas).")
print("\nEstos son los defaults ya fijados en Sección 6 (TWO_PHASE_P1/P2_TORCH).")


In [ ]:
# ── 1.2  Clonar repositorio completo + todos los submódulos ─────────────────
# Repo padre: scripts/, tools/, uc3m/, docs/, outputs/, deploy/
# Submódulos fijados (pinned commits en .gitmodules):
#   CityLearn        → Mac-Tapia/CityLearn (Colab rama viva: codex/iquitos-distillation-madrl-docs; .gitmodules pin: citylearn-v3-madrl)
#   external/HARL    → github.com/Mac-Tapia/HARL
#   external/MAAC    → github.com/Mac-Tapia/MAAC (rama viva codex/integrar-limpieza-diagnosticos: fix cuda/cpu Adam)
#   external/MARL    → github.com/Mac-Tapia/MARL
#   external/MARLlib → github.com/Mac-Tapia/MARLlib
#   external/MATD3implementation → github.com/Mac-Tapia/MATD3implementation
#   external/MicroGrids  → github.com/Mac-Tapia/MicroGrids
#   external/evcc        → github.com/evcc-io/evcc
#   external/prosumpy    → github.com/Mac-Tapia/prosumpy
# CityLearn se lleva a su rama viva (sale del detached HEAD del clone).
import os
import subprocess
from pathlib import Path

REPO_URL         = 'https://github.com/Mac-Tapia/MADRLCitytleranflexresdr.git'
REPO_BRANCH      = 'codex/fix-madrl-traceability-docs'  # rama de trabajo Colab
REPO             = '/content/MADRLCitytleranflexresdr'

CITYLEARN_URL    = 'https://github.com/Mac-Tapia/CityLearn.git'
CITYLEARN_BRANCH = 'codex/iquitos-distillation-madrl-docs'  # two_phase_happo_masac
CITYLEARN_DIR    = f'{REPO}/CityLearn'

# external/MAAC vive en su propia rama viva (fix cuda/cpu del optimizador Adam al
# reanudar desde checkpoint). Igual que CityLearn, se saca del commit fijado por el
# padre y se lleva a la punta de su rama para garantizar el parche en Colab.
MAAC_URL         = 'https://github.com/Mac-Tapia/MAAC.git'
MAAC_BRANCH      = 'codex/integrar-limpieza-diagnosticos'
MAAC_DIR         = f'{REPO}/external/MAAC'


def git_check(args, cwd=None):
    cmd = ['git'] + [str(a) for a in args]
    print('+', ' '.join(cmd))
    kw = {'cwd': cwd} if cwd else {}
    subprocess.check_call(cmd, **kw)


def git_out(args, cwd=None) -> str:
    kw = {'cwd': cwd} if cwd else {}
    return subprocess.check_output(
        ['git'] + [str(a) for a in args], text=True, **kw
    ).strip()


# ── A: Clonar repo padre con submódulos (si no existe) ───────────────────────
if not os.path.exists(f'{REPO}/.git'):
    if os.path.exists(REPO):
        raise RuntimeError(
            f'{REPO} existe pero sin .git. Elimina la carpeta y vuelve a ejecutar.'
        )
    print(f'Clonando {REPO_URL} (rama {REPO_BRANCH}) con submódulos ...')
    git_check([
        'clone',
        '--branch', REPO_BRANCH,
        '--depth', '1',
        '--recurse-submodules',
        '--shallow-submodules',
        REPO_URL, REPO,
    ])
    print('[OK] Repo padre clonado con todos los submódulos')

# ── B: Repo padre ya existe — refrescar ──────────────────────────────────────
else:
    current_origin = git_out(['config', '--get', 'remote.origin.url'], cwd=REPO)
    if current_origin != REPO_URL:
        raise RuntimeError(
            f'Repo apunta a {current_origin}; esperado {REPO_URL}. '
            'Elimina /content/MADRLCitytleranflexresdr y vuelve a ejecutar.'
        )
    print(f'Repo existente — HARD SYNC a origin/{REPO_BRANCH} ...')
    git_check(['fetch', 'origin', REPO_BRANCH], cwd=REPO)
    git_check(['reset', '--hard', f'origin/{REPO_BRANCH}'], cwd=REPO)
    git_check(['clean', '-fd'], cwd=REPO)
    # Actualizar submódulos fijados (todo excepto CityLearn que se trata aparte)
    git_check(['submodule', 'sync', '--recursive'], cwd=REPO)
    git_check([
        'submodule', 'update', '--init', '--recursive',
        '--force',
    ], cwd=REPO)
    parent_head = git_out(['rev-parse', '--short', 'HEAD'], cwd=REPO)
    print(f'[OK] Rama {REPO_BRANCH} @ {parent_head} (hard reset)')

# ── C: Hacer que CityLearn viva en su rama propia (no detached HEAD) ─────────
# Después de --recurse-submodules CityLearn queda en el commit fijado por el
# padre (detached HEAD). Lo llevamos a la punta de codex/iquitos-distillation-madrl-docs
# para que el notebook, badge Open in Colab y scripts esten actualizados.
print()
print(f'Activando CityLearn en rama viva: {CITYLEARN_BRANCH} ...')

# Asegurar que el remote mac-tapia apunte al fork correcto
existing_remotes = git_out(['remote'], cwd=CITYLEARN_DIR).splitlines()
if 'mac-tapia' not in existing_remotes:
    git_check(['remote', 'add', 'mac-tapia', CITYLEARN_URL], cwd=CITYLEARN_DIR)
else:
    git_check(['remote', 'set-url', 'mac-tapia', CITYLEARN_URL], cwd=CITYLEARN_DIR)

# Fetch la rama y checkout (rama viva, no detached HEAD)
git_check(['fetch', 'mac-tapia', CITYLEARN_BRANCH], cwd=CITYLEARN_DIR)
git_check(['checkout', '-B', CITYLEARN_BRANCH, f'mac-tapia/{CITYLEARN_BRANCH}'], cwd=CITYLEARN_DIR)
git_check(['clean', '-fd'], cwd=CITYLEARN_DIR)

cl_commit = git_out(['rev-parse', '--short', 'HEAD'], cwd=CITYLEARN_DIR)
cl_branch = git_out(['rev-parse', '--abbrev-ref', 'HEAD'], cwd=CITYLEARN_DIR)
if cl_branch == 'HEAD':
    # Fallback: algunos runtimes dejan detached HEAD tras fetch; forzar rama local
    git_check(['checkout', '-B', CITYLEARN_BRANCH], cwd=CITYLEARN_DIR)
    cl_commit = git_out(['rev-parse', '--short', 'HEAD'], cwd=CITYLEARN_DIR)
    cl_branch = git_out(['rev-parse', '--abbrev-ref', 'HEAD'], cwd=CITYLEARN_DIR)
if cl_branch != CITYLEARN_BRANCH:
    raise RuntimeError(
        f'CityLearn sigue en detached HEAD ({cl_branch!r}). '
        f'Esperado {CITYLEARN_BRANCH!r}. Revisa permisos git en {CITYLEARN_DIR}.'
    )
print(f'[OK] CityLearn activo en rama: {cl_branch} @ {cl_commit}')

# ── C2: Hacer que external/MAAC viva en su rama propia (no detached HEAD) ─────
# El backend MAAC necesita el parche _sync_optimizer_state (estado Adam a GPU al
# reanudar). Lo llevamos a la punta de MAAC_BRANCH igual que CityLearn.
print()
print(f'Activando external/MAAC en rama viva: {MAAC_BRANCH} ...')
_maac_remotes = git_out(['remote'], cwd=MAAC_DIR).splitlines()
if 'mac-tapia' not in _maac_remotes:
    git_check(['remote', 'add', 'mac-tapia', MAAC_URL], cwd=MAAC_DIR)
else:
    git_check(['remote', 'set-url', 'mac-tapia', MAAC_URL], cwd=MAAC_DIR)
git_check(['fetch', 'mac-tapia', MAAC_BRANCH], cwd=MAAC_DIR)
git_check(['checkout', '-B', MAAC_BRANCH, f'mac-tapia/{MAAC_BRANCH}'], cwd=MAAC_DIR)
git_check(['clean', '-fd'], cwd=MAAC_DIR)
maac_commit = git_out(['rev-parse', '--short', 'HEAD'], cwd=MAAC_DIR)
maac_branch = git_out(['rev-parse', '--abbrev-ref', 'HEAD'], cwd=MAAC_DIR)
if maac_branch != MAAC_BRANCH:
    raise RuntimeError(
        f'external/MAAC en rama incorrecta: {maac_branch!r} != {MAAC_BRANCH!r}.'
    )
print(f'[OK] external/MAAC activo en rama: {maac_branch} @ {maac_commit}')

# ── D: Verificar submódulos restantes (excluir CityLearn/MAAC que están adelante) ─
status_lines = git_out(['submodule', 'status', '--recursive'], cwd=REPO).splitlines()
bad = [
    ln for ln in status_lines
    if ln and ln[0] in {'-', 'U'}            # '-' = no inicializado, 'U' = conflicto
    # '+' para CityLearn es ESPERADO (está adelante del commit fijado)
]
if bad:
    print('[ERROR] Submódulos sin inicializar o en conflicto:')
    for ln in bad:
        print(f'  {ln}')
    raise RuntimeError('Repara los submódulos antes de continuar.')

print()
print('═' * 60)
print('  Repositorio listo')
print(f'  Padre    : {REPO_BRANCH} @ {git_out(["rev-parse", "--short", "HEAD"], cwd=REPO)}')
print(f'  CityLearn: {cl_branch} @ {cl_commit}  ← RAMA VIVA')
print('═' * 60)

os.chdir(REPO)

COLAB_OPEN_URL = (
    f'https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/'
    f'{REPO_BRANCH}/examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb'
)
print(f'Open in Colab (GitHub): {COLAB_OPEN_URL}')

# ── E: Bloqueo protocolo en disco (no continuar con scripts legacy) ───────────
import sys as _sys_guard
_guard_py = f'{REPO}/CityLearn/scripts/colab_protocol_guard.py'
if not os.path.isfile(_guard_py):
    raise FileNotFoundError(f'Falta colab_protocol_guard.py: {_guard_py}')
subprocess.check_call([_sys_guard.executable, _guard_py, 'verify-repo', '--repo', REPO])
print('[OK] protocol-guard: launcher/monitor two_phase_happo_masac_v3 en /content')

# ── F: Verificación de parches (script en disco — inmune a notebook/kernel en caché)
# El hard sync de 1.2 actualiza este .py; no depende del texto de la celda en Colab.
_verify_patches_py = f'{REPO}/CityLearn/scripts/colab_verify_critical_patches.py'
if not os.path.isfile(_verify_patches_py):
    raise FileNotFoundError(
        f'Falta {_verify_patches_py}. Re-ejecuta celda 1.2 (hard sync).'
    )
subprocess.check_call([_sys_guard.executable, _verify_patches_py, '--repo', REPO])


In [ ]:
# ── 1.2b  Validar repo Colab: padre + CityLearn en rama viva ───────────────
import glob
import os
import subprocess

PROJECT_NAME = 'MADRLCitytleranflexresdr'
REPO_URL     = globals().get('REPO_URL', 'https://github.com/Mac-Tapia/MADRLCitytleranflexresdr.git')
REPO_BRANCH  = globals().get('REPO_BRANCH', 'codex/fix-madrl-traceability-docs')

# ── Detección automática de REPO (Colab o local; independiente de celda 1.2) ─
import importlib.util
_in_colab_12b = importlib.util.find_spec('google.colab') is not None

_repo_from_ctx = globals().get('REPO', None)
if _repo_from_ctx and Path(_repo_from_ctx).exists() and (Path(_repo_from_ctx) / 'CityLearn').exists():
    REPO = _repo_from_ctx
elif _in_colab_12b:
    REPO = '/content/MADRLCitytleranflexresdr'
    if not Path(REPO).exists():
        raise RuntimeError(
            f'{REPO} no existe. Ejecuta celda 1.2 (clone + hard sync) antes de 1.2b.'
        )
else:
    _start = Path.cwd()
    _candidates = [
        _start,
        _start.parent,
        _start.parent.parent,
        Path('d:/MADRLCitytleranflexresdr'),
        Path.home() / 'MADRLCitytleranflexresdr',
    ]
    REPO = next(
        (
            str(p)
            for p in _candidates
            if (p / 'CityLearn').exists() and (p / '.git').exists()
        ),
        None,
    )
    if REPO is None:
        raise RuntimeError(
            'No se encontró el repo MADRLCitytleranflexresdr. '
            'Ejecuta celda 1.2 en Colab o abre el notebook desde la raíz del proyecto.'
        )

DATASET_DIR        = f'{REPO}/CityLearn/data/datasets/citylearn_iquitos_2023_2025'
SCHEMA_FOR_CONTEXT = f'{DATASET_DIR}/schema.json'
CITYLEARN_URL      = globals().get('CITYLEARN_URL', 'https://github.com/Mac-Tapia/CityLearn.git')
CITYLEARN_BRANCH   = globals().get('CITYLEARN_BRANCH', 'codex/iquitos-distillation-madrl-docs')
MAAC_URL           = globals().get('MAAC_URL', 'https://github.com/Mac-Tapia/MAAC.git')
MAAC_BRANCH        = globals().get('MAAC_BRANCH', 'codex/integrar-limpieza-diagnosticos')


def sh(args, cwd=REPO) -> str:
    return subprocess.check_output([str(a) for a in args], cwd=cwd, text=True).strip()


# 1. Repo padre en la rama correcta
repo_root = sh(['git', 'rev-parse', '--show-toplevel'])
branch    = sh(['git', 'rev-parse', '--abbrev-ref', 'HEAD'])
head      = sh(['git', 'rev-parse', 'HEAD'])
origin    = sh(['git', 'config', '--get', 'remote.origin.url'])

assert Path(repo_root).resolve() == Path(REPO).resolve(), f'Repo root: {repo_root}'
assert branch == REPO_BRANCH, (
    f'Rama incorrecta: {branch!r} != {REPO_BRANCH!r}. '
    f'Ejecuta la celda 1.2 para sincronizar.'
)
assert origin == REPO_URL, f'Origin: {origin!r} != {REPO_URL!r}'
print(f'[OK] Repo padre: {branch} @ {head[:12]}')

# 2. CityLearn en su rama viva (NO detached HEAD, NO commit fijado antiguo)
cl_dir    = f'{REPO}/CityLearn'
cl_branch = sh(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=cl_dir)
cl_commit = sh(['git', 'rev-parse', 'HEAD'], cwd=cl_dir)

if cl_branch == 'HEAD':
    print('[FIX] CityLearn en detached HEAD — activando rama viva ...')
    _remotes = sh(['git', 'remote'], cwd=cl_dir).splitlines()
    if 'mac-tapia' not in _remotes:
        subprocess.check_call(
            ['git', 'remote', 'add', 'mac-tapia', CITYLEARN_URL], cwd=cl_dir
        )
    subprocess.check_call(
        ['git', 'fetch', 'mac-tapia', CITYLEARN_BRANCH], cwd=cl_dir
    )
    subprocess.check_call(
        ['git', 'checkout', '-B', CITYLEARN_BRANCH, f'mac-tapia/{CITYLEARN_BRANCH}'],
        cwd=cl_dir,
    )
    cl_branch = sh(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=cl_dir)
    cl_commit = sh(['git', 'rev-parse', 'HEAD'], cwd=cl_dir)
    print(f'[OK] CityLearn reparado: {cl_branch} @ {cl_commit[:12]}')

assert cl_branch == CITYLEARN_BRANCH, (
    f'CityLearn en rama incorrecta: {cl_branch!r} != {CITYLEARN_BRANCH!r}. '
    f'Ejecuta la celda 1.2 para activar la rama viva.'
)
print(f'[OK] CityLearn: {cl_branch} @ {cl_commit[:12]}  ← rama viva')

# 2b. external/MAAC en su rama viva (parche cuda/cpu del optimizador Adam)
maac_dir    = f'{REPO}/external/MAAC'
maac_branch = sh(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=maac_dir)
maac_commit = sh(['git', 'rev-parse', 'HEAD'], cwd=maac_dir)
if maac_branch != MAAC_BRANCH:
    print(f'[FIX] external/MAAC en {maac_branch!r} — activando rama viva ...')
    _mremotes = sh(['git', 'remote'], cwd=maac_dir).splitlines()
    if 'mac-tapia' not in _mremotes:
        subprocess.check_call(['git', 'remote', 'add', 'mac-tapia', MAAC_URL], cwd=maac_dir)
    subprocess.check_call(['git', 'fetch', 'mac-tapia', MAAC_BRANCH], cwd=maac_dir)
    subprocess.check_call(
        ['git', 'checkout', '-B', MAAC_BRANCH, f'mac-tapia/{MAAC_BRANCH}'], cwd=maac_dir,
    )
    maac_branch = sh(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=maac_dir)
    maac_commit = sh(['git', 'rev-parse', 'HEAD'], cwd=maac_dir)
assert maac_branch == MAAC_BRANCH, (
    f'external/MAAC en rama incorrecta: {maac_branch!r} != {MAAC_BRANCH!r}. '
    f'Ejecuta la celda 1.2 para activar la rama viva.'
)
print(f'[OK] external/MAAC: {maac_branch} @ {maac_commit[:12]}  ← rama viva')

# 3. Submódulos dependencia en estado correcto (sin '-' ni 'U')
submodule_status = sh(['git', 'submodule', 'status', '--recursive'])
bad_submodules = [
    ln for ln in submodule_status.splitlines()
    if ln and ln[0] in {'-', 'U'}    # '+' para CityLearn es esperado y aceptado
]
if bad_submodules:
    raise RuntimeError(
        'Submódulos no inicializados o en conflicto:\n' + '\n'.join(bad_submodules)
    )
print('[OK] Todos los submódulos de dependencia inicializados')

# 4. Rutas críticas del proyecto
required_paths = [
    'examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb',
    'CityLearn/scripts/colab_a100_official_launcher.py',
    'CityLearn/scripts/colab_a100_live_monitor.py',
    'CityLearn/scripts/colab_protocol_guard.py',
    'CityLearn/scripts/colab_verify_critical_patches.py',
    'CityLearn/scripts/train_citylearn_v3_happo.py',
    'CityLearn/scripts/train_citylearn_v3_masac.py',
    'CityLearn/scripts/train_citylearn_v3_matd3.py',
    'CityLearn/scripts/train_citylearn_v3_maac.py',
    'CityLearn/citylearn/v3/environment.py',
    'external/HARL',
    'external/MARL/src',
    'external/off-policy',
    'external/MAAC',
    'uc3m',
    'tools',
]
missing = [p for p in required_paths if not (Path(REPO) / p).exists()]
if missing:
    raise FileNotFoundError('Rutas requeridas no encontradas: ' + ', '.join(missing))
print(f'[OK] {len(required_paths)} rutas críticas presentes')

# 5. Dataset Iquitos 2023-2025
csv_count = len(glob.glob(f'{DATASET_DIR}/*.csv'))
with open(SCHEMA_FOR_CONTEXT) as f:
    schema_context = json.load(f)
assert csv_count == 222, f'Dataset incompleto: {csv_count}/222 CSV'
assert len(schema_context.get('buildings', {})) == 17, 'Schema: se esperan 17 edificios'
assert schema_context.get('simulation_end_time_step') == 26303
print(f'[OK] Dataset: {csv_count} CSV, 17 edificios, 26304 pasos')

# Protocolo y parches ya verificados en celda 1.2 (sections E/F).

# 5c. Badge Open in Colab alineado con REPO_BRANCH (push en Mac-Tapia/MADRLCitytleranflexresdr)
_nb_file = Path(REPO) / 'examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb'
_badge_needle = (
    f'github/Mac-Tapia/MADRLCitytleranflexresdr/blob/{REPO_BRANCH}/'
    'examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb'
)
_nb_raw = _nb_file.read_text(encoding='utf-8')
if _badge_needle not in _nb_raw:
    _colab_url = (
        f'https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/'
        f'{REPO_BRANCH}/examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb'
    )
    raise RuntimeError(
        'Badge Open in Colab desactualizado en el notebook.\n'
        f'  Debe apuntar a: {_colab_url}\n'
        '  Actualiza la celda markdown del titulo y haz push a CityLearn.'
    )
print(f'[OK] Open in Colab badge -> Mac-Tapia/CityLearn @ {CITYLEARN_BRANCH}')

# 6. Guardar contexto del proyecto para celdas siguientes
COLAB_PROJECT_CONTEXT = {
    'project_name': PROJECT_NAME,
    'repo_url': REPO_URL,
    'repo_branch': branch,
    'repo_commit': head,
    'repo_root': REPO,
    'citylearn_branch': cl_branch,
    'citylearn_commit': cl_commit,
    'citylearn_live': True,           # confirma que CityLearn está en rama viva
    'launcher_protocol': 'two_phase_happo_masac_v3',
    'dataset_dir': DATASET_DIR,
    'dataset_csv_count': csv_count,
    'buildings': len(schema_context.get('buildings', {})),
    'simulation_steps': schema_context.get('simulation_end_time_step') + 1,
}
os.makedirs(f'{REPO}/outputs', exist_ok=True)
with open(f'{REPO}/outputs/colab_project_context.json', 'w') as f:
    json.dump(COLAB_PROJECT_CONTEXT, f, indent=2)

print()
print('═' * 60)
print('  Repo Colab VALIDADO')
print(f'  Repo padre : {branch} @ {head[:12]}')
print(f'  CityLearn  : {cl_branch} @ {cl_commit[:12]}  ← VIVA')
print(f'  Dataset    : {csv_count} CSV · 17 edificios · 26304 pasos')
print('═' * 60)


In [ ]:
# 1.3 Instalar dependencias del proyecto de forma reproducible
# Usa Python 3.9 del proyecto. Si Colab entrega kernel 3.11, crea/usa .venv39-citylearn-v3.
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = '/content/MADRLCitytleranflexresdr'
PROJECT_ROOT = Path(PROJECT_DIR)
if not PROJECT_ROOT.exists() and Path.cwd().name == 'MADRLCitytleranflexresdr':
    PROJECT_ROOT = Path.cwd()
    PROJECT_DIR = str(PROJECT_ROOT)

VENV_DIR = PROJECT_ROOT / '.venv39-citylearn-v3'
SETUP_LOG = Path('/tmp/madrl_py39_setup.log')
CONSTRAINTS = Path('/tmp/madrl_compat.txt')
PYTHON_REQUIRED = (3, 9)
PYTHON_MIN = PYTHON_REQUIRED
PYTHON_MAX_EXCLUSIVE = (3, 10)
PYTORCH_INDEX_CU126 = 'https://download.pytorch.org/whl/cu126'
PYTORCH_INDEX_CU128 = 'https://download.pytorch.org/whl/cu128'
TORCH_PACKAGES = ('torch', 'torchvision')

def detect_pytorch_cuda_wheel():
    """Blackwell (sm_120, RTX PRO 6000) requiere wheels cu128; A100/H100 usan cu126."""
    try:
        name = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True, stderr=subprocess.DEVNULL,
        ).strip().splitlines()[0].strip()
    except Exception:
        return PYTORCH_INDEX_CU126, None
    upper = name.upper()
    blackwell = any(k in upper for k in ('BLACKWELL', 'RTX PRO 6000', 'RTX 50'))
    index = PYTORCH_INDEX_CU128 if blackwell else PYTORCH_INDEX_CU126
    return index, name

PYTORCH_INDEX_URL, _DETECTED_GPU = detect_pytorch_cuda_wheel()
print(f'[torch] GPU: {_DETECTED_GPU or "(sin nvidia-smi)"} -> index {PYTORCH_INDEX_URL.split("/")[-1]}')

COMPAT_WHEELS = [
    'numpy==1.23.5',
    'pandas==2.0.3',
    'scipy==1.10.1',
    'scikit-learn==1.2.2',
    'matplotlib==3.7.5',
    'seaborn==0.12.2',
]
PINNED = {
    'numpy': '1.23.5',
    'pandas': '2.0.3',
    'scipy': '1.10.1',
    'scikit-learn': '1.2.2',
    'matplotlib': '3.7.5',
    'seaborn': '0.12.2',
    'gymnasium': '0.28.1',
    'pettingzoo': '1.12.0',
}
BASE_DEPS = [
    *COMPAT_WHEELS,
    'pyyaml',
    'requests>=2.28',
    'tqdm>=4.65',
    'psutil>=5.9',
    'platformdirs>=3.0',
    'protobuf==3.20.3',
    'gymnasium==0.28.1',
    'pettingzoo==1.12.0',
    'gym==0.20.0',
    'tensorboard',
    'tensorboardX',
    'setproctitle',
    'simplejson',
    'absl-py',
    'dm-tree',
    'importlib-metadata>=6.0,<9',
]
NO_DEPS_UTILS = [
    'supersuit==3.2.0',
    'icecream==2.1.3',
]
EDITABLES = [
    'CityLearn/',
    'external/HARL/',
    'external/off-policy/',
    'external/MAAC/',
    'external/MARL/src/',
]
BINARY_DEPS = ('numpy', 'pandas', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn')

ABI_CHECK = """
import importlib
import json
import sys

modules = {
    'torch': 'torch',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'scikit-learn': 'sklearn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'gym': 'gym',
    'gymnasium': 'gymnasium',
    'pettingzoo': 'pettingzoo',
    'citylearn.v3.environment': 'citylearn.v3.environment',
}
CRITICAL = {'numpy', 'scipy', 'sklearn', 'gymnasium', 'pettingzoo', 'citylearn.v3.environment'}
versions = {'python': sys.version.split()[0], 'executable': sys.executable}
failures = {}
for label, module_name in modules.items():
    try:
        module = importlib.import_module(module_name)
        versions[label] = getattr(module, '__version__', 'importado')
    except Exception as exc:
        failures[label] = repr(exc)
        versions[label] = f'ERROR: {exc}'
try:
    import torch
    versions['torch_cuda_available'] = bool(torch.cuda.is_available())
    versions['torch_cuda'] = getattr(torch.version, 'cuda', None)
except Exception as exc:
    versions['torch_cuda_error'] = repr(exc)
print(json.dumps(versions, indent=2, sort_keys=True))
critical_failures = {k: v for k, v in failures.items() if k in CRITICAL}
if critical_failures:
    print('CRITICAL_FAILURES: ' + json.dumps(critical_failures), file=sys.stderr)
    sys.exit(1)
elif failures:
    print('NON_CRITICAL_FAILURES: ' + json.dumps(failures), file=sys.stderr)
"""


def write_log(text):
    SETUP_LOG.parent.mkdir(parents=True, exist_ok=True)
    with SETUP_LOG.open('a', encoding='utf-8') as f:
        f.write(text)
        if not text.endswith('\n'):
            f.write('\n')


def print_log_tail(lines=80):
    if not SETUP_LOG.exists():
        return
    tail = SETUP_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-lines:]
    print(f'\n[TAIL {SETUP_LOG}]')
    print('\n'.join(tail))


def run(cmd, *, cwd=None, env=None, check=True):
    cmd = [str(part) for part in cmd]
    message = '+ ' + ' '.join(cmd)
    print(message)
    write_log('\n' + message)
    proc = subprocess.run(
        cmd,
        cwd=str(cwd or PROJECT_ROOT),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if proc.stdout:
        write_log(proc.stdout)
    if check and proc.returncode != 0:
        print_log_tail()
        raise RuntimeError(
            f'Comando fallo con exit={proc.returncode}: {message}. '
            f'Log completo: {SETUP_LOG}'
        )
    return proc


def run_shell(script, *, cwd=None, env=None, check=True):
    return run(['bash', '-lc', script], cwd=cwd, env=env, check=check)


def venv_python_path():
    if os.name == 'nt':
        return VENV_DIR / 'Scripts' / 'python.exe'
    return VENV_DIR / 'bin' / 'python'


def python_info(python):
    python = str(python)
    if not Path(python).exists() and python != sys.executable:
        return None
    code = """
import json
import sys
print(json.dumps({
    'executable': sys.executable,
    'version': sys.version.split()[0],
    'version_info': list(sys.version_info[:3]),
}))
"""
    result = subprocess.run([python, '-c', code], capture_output=True, text=True)
    if result.returncode != 0:
        return None
    return json.loads(result.stdout)


def same_executable(left, right):
    try:
        return Path(left).resolve() == Path(right).resolve()
    except Exception:
        return str(left) == str(right)


def setup_env():
    env = os.environ.copy()
    home_bin = str(Path.home() / '.local' / 'bin')
    env['PATH'] = home_bin + os.pathsep + env.get('PATH', '')
    return env


def ensure_uv(env):
    uv = shutil.which('uv', path=env.get('PATH'))
    if uv:
        return uv
    run_shell('curl -LsSf https://astral.sh/uv/install.sh | sh', env=env)
    uv = shutil.which('uv', path=env.get('PATH'))
    if uv:
        return uv
    candidate = Path.home() / '.local' / 'bin' / 'uv'
    if candidate.exists():
        return str(candidate)
    raise RuntimeError('uv no quedo disponible en PATH despues de instalarlo.')


def ensure_project_python39():
    current_info = python_info(sys.executable)
    if current_info and tuple(current_info['version_info'][:2]) == PYTHON_REQUIRED:
        return sys.executable

    project_python = venv_python_path()
    project_info = python_info(project_python)
    if project_info and tuple(project_info['version_info'][:2]) == PYTHON_REQUIRED:
        return str(project_python)

    if platform.system() == 'Windows':
        raise RuntimeError(
            'El kernel actual no es Python 3.9. En Windows selecciona '
            '.venv39-citylearn-v3 como kernel o recrea el entorno con scripts/setup.'
        )

    env = setup_env()
    uv = ensure_uv(env)
    print(
        f'Kernel actual: Python {sys.version.split()[0]} ({sys.executable}). '
        f'Creando entorno de proyecto Python 3.9 en {VENV_DIR}.'
    )
    run([uv, 'python', 'install', '3.9'], cwd=PROJECT_ROOT, env=env)
    run([uv, 'venv', '--python', '3.9', str(VENV_DIR)], cwd=PROJECT_ROOT, env=env)

    project_info = python_info(project_python)
    if not project_info or tuple(project_info['version_info'][:2]) != PYTHON_REQUIRED:
        raise RuntimeError(f'No se pudo crear un Python 3.9 valido en {project_python}')
    return str(project_python)


def pip_install(*args):
    cmd = [PROJECT_PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check', *args]
    run(cmd)


def installed_version(package):
    code = """
import importlib.metadata as im
import sys
package = sys.argv[1]
names = (package, package.replace('-', '_'), package.replace('_', '-'))
for name in dict.fromkeys(names):
    try:
        print(im.version(name))
        raise SystemExit(0)
    except im.PackageNotFoundError:
        pass
raise SystemExit(1)
"""
    result = subprocess.run([PROJECT_PYTHON, '-c', code, package], capture_output=True, text=True)
    return result.stdout.strip() if result.returncode == 0 else None


def torch_gpu_ready():
    """True solo si PyTorch puede ejecutar un kernel real en la GPU (detecta sm_120 sin cu128)."""
    code = """
import json, sys
try:
    import torch
    info = {
        'version': torch.__version__,
        'cuda_runtime': getattr(torch.version, 'cuda', None),
        'cuda_available': bool(torch.cuda.is_available()),
    }
    if not info['cuda_available']:
        print(json.dumps({**info, 'ready': False, 'reason': 'cuda_unavailable'}))
        raise SystemExit(0)
    cap = torch.cuda.get_device_capability(0)
    info['device'] = torch.cuda.get_device_name(0)
    info['capability'] = list(cap)
    upper = info['device'].upper()
    needs_cu128 = cap[0] >= 12 or any(k in upper for k in ('BLACKWELL', 'RTX PRO 6000', 'RTX 50'))
    info['needs_cu128'] = needs_cu128
    cuda_rt = str(info['cuda_runtime'] or '')
    if needs_cu128 and not cuda_rt.startswith('12.8'):
        print(json.dumps({**info, 'ready': False, 'reason': 'blackwell_needs_cu128'}))
        raise SystemExit(0)
    x = torch.zeros(1, device='cuda')
    _ = (x + 1).item()
    torch.cuda.synchronize()
    print(json.dumps({**info, 'ready': True, 'kernel_ok': True}))
except Exception as exc:
    print(json.dumps({'ready': False, 'reason': 'kernel_failed', 'error': repr(exc)}))
"""
    result = subprocess.run([PROJECT_PYTHON, '-c', code], capture_output=True, text=True)
    line = (result.stdout or '').strip().splitlines()
    if not line:
        return False, {}
    try:
        data = json.loads(line[-1])
    except Exception:
        return False, {}
    if result.stdout.strip():
        print('[torch]', line[-1])
    return bool(data.get('ready')), data


def torch_cuda_available():
    ready, _ = torch_gpu_ready()
    return ready


def restart_runtime(reason):
    print(f'\n[RESTART REQUERIDO] {reason}')
    print('Reinicia el runtime y vuelve a ejecutar desde la celda 1.2b.')
    try:
        from IPython.core.getipython import get_ipython
        import time

        print('Colab detectado: reiniciando kernel automaticamente...')
        _ip = get_ipython()
        _kernel = getattr(_ip, 'kernel', None) if _ip is not None else None
        if _kernel is not None:
            _kernel.do_shutdown(restart=True)
        time.sleep(10)
    except Exception:
        pass
    raise RuntimeError(reason)


def repair_binary_abi():
    print('[ABI] Reinstalando ruedas binarias con versiones fijadas (numpy 1.23.5)...')
    pip_install('-q', '--force-reinstall', '--no-cache-dir', *COMPAT_WHEELS)


def verify_subprocess_imports():
    result = subprocess.run([PROJECT_PYTHON, '-c', ABI_CHECK], capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout, end='')
    if result.returncode != 0:
        stderr = result.stderr.strip()
        abi_mismatch = any(
            token in stderr
            for token in ('numpy.dtype size changed', 'binary incompatibility', 'numpy.core', 'numpy.strings')
        )
        if abi_mismatch:
            repair_binary_abi()
            result = subprocess.run([PROJECT_PYTHON, '-c', ABI_CHECK], capture_output=True, text=True)
            if result.stdout.strip():
                print(result.stdout, end='')
        if result.returncode != 0:
            if result.stderr.strip():
                print('[STDERR verificacion ABI:]')
                print(result.stderr[-4000:], end='')
            print_log_tail()
            raise RuntimeError('ABI fallo: ' + result.stderr.strip()[-2000:])
    elif result.stderr.strip():
        print('[advertencias ABI (no criticas):]')
        print(result.stderr.strip())


def verify_current_kernel_imports_if_needed():
    if not same_executable(PROJECT_PYTHON, sys.executable):
        print(
            f'Kernel notebook: Python {sys.version.split()[0]} ({sys.executable}). '
            f'Entrenamiento: {PROJECT_PYTHON}. No se importan paquetes del proyecto en el kernel.'
        )
        return

    modules = ('numpy', 'scipy', 'sklearn', 'pandas', 'citylearn.v3.environment')
    failures = {}
    for module_name in modules:
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[module_name] = repr(exc)
    if failures:
        restart_runtime(
            'El kernel actual tiene imports binarios inconsistentes: '
            f'{failures}. Esto ocurre si pip cambio numpy/scipy sin reiniciar.'
        )


if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'PROJECT_DIR no existe: {PROJECT_ROOT}. Ejecuta primero la celda 1.2.')

SETUP_LOG.write_text('', encoding='utf-8')
PROJECT_PYTHON = ensure_project_python39()
PYTHON = PROJECT_PYTHON
project_info = python_info(PROJECT_PYTHON)
if not project_info or tuple(project_info['version_info'][:2]) != PYTHON_REQUIRED:
    raise RuntimeError(f'Python de proyecto invalido: {project_info}')

os.chdir(PROJECT_DIR)
CONSTRAINTS.write_text('\n'.join(f'{p}=={v}' for p, v in PINNED.items()) + '\n')
print(f"Python proyecto: {project_info['version']} ({PROJECT_PYTHON})")
print(f"Python kernel  : {sys.version.split()[0]} ({sys.executable})")
print(f'Log setup      : {SETUP_LOG}')

# Pip compatible con gym/ray legacy del proyecto.
run([PROJECT_PYTHON, '-m', 'ensurepip', '--upgrade'], check=False)
pip_install('--force-reinstall', 'pip==21.3.1', 'setuptools==65.5.0', 'wheel==0.38.0')

pip_install('-q', *BASE_DEPS)
for package in NO_DEPS_UTILS:
    if installed_version(package.split('==')[0]) is None:
        pip_install('-q', '--no-deps', package)

ready, torch_info = torch_gpu_ready() if installed_version('torch') else (False, {})
if not ready:
    reason = torch_info.get('reason', 'missing_or_incompatible')
    print(f'[torch] Instalando PyTorch ({PYTORCH_INDEX_URL.split("/")[-1]}) — motivo: {reason}')
    pip_install('--force-reinstall', '-q', *TORCH_PACKAGES, '--index-url', PYTORCH_INDEX_URL)
    ready, torch_info = torch_gpu_ready()
if not ready:
    raise RuntimeError(
        'PyTorch no puede ejecutar kernels en esta GPU. '
        f'info={torch_info}. Blackwell (RTX PRO 6000) requiere cu128: '
        'pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128'
    )

for package_dir in EDITABLES:
    pip_install('-q', '--no-deps', '-c', str(CONSTRAINTS), '-e', package_dir)

# Reinstalacion final: evita ruedas pandas/scipy compiladas para numpy 2.x.
pip_install('-q', '--force-reinstall', '--no-cache-dir', *COMPAT_WHEELS)

binary_after = {package: installed_version(package) for package in BINARY_DEPS}
print('Paquetes binarios:', binary_after)

print('\nVerificando ABI en Python 3.9 del proyecto...')
verify_subprocess_imports()
verify_current_kernel_imports_if_needed()
print('\nCelda 1.3 OK: Python 3.9 del proyecto listo y backends en modo editable.')


In [ ]:
# 1.4 Configurar sys.path, CUDA y smoke imports
# Vinculada con 1.3: usa PROJECT_PYTHON aunque el kernel Colab sea Python 3.11.
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_DIR', '/content/MADRLCitytleranflexresdr'))
if not PROJECT_ROOT.exists() and Path.cwd().name == 'MADRLCitytleranflexresdr':
    PROJECT_ROOT = Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.resolve()
REPO = str(PROJECT_ROOT)

PYTHON_MIN = globals().get('PYTHON_MIN', (3, 9))
PYTHON_MAX_EXCLUSIVE = globals().get('PYTHON_MAX_EXCLUSIVE', (3, 10))
PROJECT_PYTHON = globals().get('PROJECT_PYTHON', globals().get('PYTHON', sys.executable))
PYTHON = PROJECT_PYTHON
EDITABLES = globals().get('EDITABLES', [
    'CityLearn/',
    'external/HARL/',
    'external/off-policy/',
    'external/MAAC/',
    'external/MARL/src/',
])


def repo_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def _python_info_strict(python):
    code = """
import json
import sys
print(json.dumps({
    'executable': sys.executable,
    'version': sys.version.split()[0],
    'version_info': list(sys.version_info[:3]),
}))
"""
    result = subprocess.run([str(python), '-c', code], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr or f'No se pudo ejecutar {python}')
    return json.loads(result.stdout)


python_info = _python_info_strict


if 'same_executable' not in globals():
    def same_executable(left, right):
        try:
            return Path(left).resolve() == Path(right).resolve()
        except Exception:
            return str(left) == str(right)

if 'restart_runtime' not in globals():
    def restart_runtime(reason):
        print(f'\n[RESTART REQUERIDO] {reason}')
        print('Reinicia el runtime y vuelve a ejecutar desde la celda 1.2b.')
        try:
            from IPython.core.getipython import get_ipython
            import time

            print('Colab detectado: reiniciando kernel automaticamente...')
            _ip = get_ipython()
            _kernel = getattr(_ip, 'kernel', None) if _ip is not None else None
            if _kernel is not None:
                _kernel.do_shutdown(restart=True)
            time.sleep(10)
        except Exception:
            pass
        raise RuntimeError(reason)


project_python_info = python_info(PROJECT_PYTHON)
if project_python_info is None:
    raise RuntimeError(f'No se pudo leer Python de proyecto: {PROJECT_PYTHON}')
if not (PYTHON_MIN <= tuple(project_python_info['version_info'][:2]) < PYTHON_MAX_EXCLUSIVE):
    raise RuntimeError(
        f"Python de proyecto {project_python_info['version']} no soportado. "
        'Ejecuta primero la celda 1.3 para crear/validar .venv39-citylearn-v3.'
    )

PATHS = list(dict.fromkeys(str(path) for path in [
    PROJECT_ROOT,
    PROJECT_ROOT / 'CityLearn',
    PROJECT_ROOT / 'CityLearn' / 'scripts',
    *(repo_path(path) for path in EDITABLES),
]))
missing = [path for path in PATHS if not Path(path).exists()]
if missing:
    raise FileNotFoundError(f'Rutas requeridas no encontradas: {missing}')

for path in reversed(PATHS):
    while path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)

old_pythonpath = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p]
old_pythonpath = [p for p in old_pythonpath if p not in PATHS]
os.environ['PYTHONPATH'] = os.pathsep.join(PATHS + old_pythonpath)
os.environ['CITYLEARN_PROJECT_ROOT'] = REPO
os.environ.setdefault('CUDA_DEVICE_ORDER', 'PCI_BUS_ID')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('PYTHONHASHSEED', '0')

SMOKE_IMPORTS = {
    'torch': 'torch',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'sklearn': 'sklearn',
    'citylearn': 'citylearn',
    'citylearn.v3.environment': 'citylearn.v3.environment',
    'harl': 'harl',
    'runner_msac': 'runner_msac',
    'offpolicy': 'offpolicy',
    'algorithms.attention_sac': 'algorithms.attention_sac',
}
OPTIONAL_IMPORTS = {'harl', 'runner_msac', 'offpolicy', 'algorithms.attention_sac'}

smoke_code = f"""
import importlib, json, sys
paths = {PATHS!r}
modules = {SMOKE_IMPORTS!r}
optional = set({sorted(OPTIONAL_IMPORTS)!r})
for path in reversed(paths):
    if path not in sys.path:
        sys.path.insert(0, path)
imports, versions = {{}}, {{'python': sys.version.split()[0], 'executable': sys.executable}}
for label, module_name in modules.items():
    try:
        module = importlib.import_module(module_name)
        imports[label] = 'ok'
        version = getattr(module, '__version__', None)
        if version:
            versions[label] = version
    except Exception as exc:
        imports[label] = f'FAILED: {{exc}}'
print(json.dumps({{'imports': imports, 'versions': versions}}, indent=2, sort_keys=True))
failed = {{k: v for k, v in imports.items() if v.startswith('FAILED')}}
critical_failed = {{k: v for k, v in failed.items() if k not in optional}}
if failed.keys() - critical_failed.keys():
    print(f'[WARN] Modulos opcionales no disponibles: {{sorted(failed.keys() - critical_failed.keys())}}')
if critical_failed:
    abi = any('numpy.dtype size changed' in v or 'numpy.core' in v or 'numpy.strings' in v or '_center' in v for v in critical_failed.values())
    hint = 'Reinicia el runtime y ejecuta 1.1-1.4 en orden.' if abi else 'Ejecuta primero la celda 1.3 y repite 1.4.'
    raise SystemExit(f'ERROR: imports criticos fallaron: {{critical_failed}}. {{hint}}')
"""

result = subprocess.run([PROJECT_PYTHON, '-c', smoke_code], capture_output=True, text=True, env=os.environ.copy())
if result.stdout.strip():
    print(result.stdout, end='')
if result.returncode != 0:
    if result.stderr.strip():
        print('[STDERR smoke check:]')
        print(result.stderr, end='')
    raise RuntimeError('Smoke imports criticos fallaron en Python 3.9 del proyecto. Revisa el JSON anterior.')

if same_executable(PROJECT_PYTHON, sys.executable):
    current_failures = {}
    for label, module_name in SMOKE_IMPORTS.items():
        if label in OPTIONAL_IMPORTS:
            continue
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            current_failures[label] = repr(exc)
    if current_failures:
        restart_runtime(
            'El subprocess importa bien, pero el kernel actual esta inconsistente: '
            f'{current_failures}.'
        )
else:
    print(f'Kernel notebook en {sys.version.split()[0]}; smoke imports ejecutados con {PROJECT_PYTHON}.')

print(f'Celda 1.4 OK: sys.path, CUDA env y smoke imports configurados para {PROJECT_PYTHON}.')


### Persistencia obligatoria en Google Drive

Para entrenar **50 episodios/corrida** (reanudable con `--skip-completed`) en Colab, los artefactos deben vivir fuera de `/content`. La celda siguiente monta Drive por defecto; fuera de Colab usa el fallback local del repo.


In [ ]:
# ── 1.5  Montar Google Drive (workspace canonico; verificacion en 2.1) ─────
import os
import sys
import shutil
import subprocess
import time
from pathlib import Path

USE_GOOGLE_DRIVE = True
REQUIRE_GOOGLE_DRIVE = True
DRIVE_MOUNT_POINT = '/content/drive'
PROJECT_NAME = globals().get('PROJECT_NAME', 'MADRLCitytleranflexresdr')
GDRIVE_ROOT = None
GDRIVE_OUTPUT_PARENT = None
CANONICAL_DRIVE_URL = 'https://drive.google.com/drive/folders/1ihH6RqL2KpevfCQEUXj7PP1aS2QYssAX'

MIN_DRIVE_FREE_GIB = 30.0  # A100-80GB HAPPO hidden=512: checkpoints + timeseries persistentes
DRIVE_MOUNT_MAX_ATTEMPTS = 3

def _drive_mydrive_ready(mount_point=DRIVE_MOUNT_POINT):
    mydrive = f'{mount_point}/MyDrive'
    return os.path.isdir(mydrive) and os.access(mydrive, os.R_OK | os.W_OK)

def _mount_google_drive_colab(mount_point=DRIVE_MOUNT_POINT, max_attempts=DRIVE_MOUNT_MAX_ATTEMPTS):
    """Monta Drive con un solo popup OAuth (drive.mount). Cualquier cuenta Google sirve."""
    from google.colab import drive  # type: ignore[import-not-found,import-untyped]

    if _drive_mydrive_ready(mount_point):
        print(f'[OK] Google Drive ya montado en {mount_point}')
        return

    # No llamar auth.authenticate_user(): dispara un segundo OAuth efimero y rompe
    # credentials-propagation de drive.mount (400 Bad Request, authuser=0).
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        force = attempt > 1 and os.path.isdir(mount_point)
        try:
            print(f'Montando Google Drive (intento {attempt}/{max_attempts}, force_remount={force})...')
            drive.mount(mount_point, force_remount=force)
            if not _drive_mydrive_ready(mount_point):
                raise RuntimeError(f'Mount OK pero {mount_point}/MyDrive no es accesible')
            print(f'[OK] Google Drive montado en {mount_point}')
            return
        except Exception as exc:
            last_exc = exc
            print(f'[WARN] Intento {attempt} fallo: {type(exc).__name__}: {exc}')
            if attempt < max_attempts:
                time.sleep(5)

    help_msg = (
        'Google Drive no se pudo montar.\n'
        '1) Acepta el popup "Permitir acceso" con la cuenta Google que prefieras\n'
        '2) Si falla: Runtime -> Disconnect and delete runtime (o Factory reset)\n'
        '3) Abre el notebook en colab.research.google.com (no solo VS Code + extension)\n'
        '4) Re-ejecuta solo la celda 1.5'
    )
    raise RuntimeError(help_msg) from last_exc

if USE_GOOGLE_DRIVE:
    try:
        _mount_google_drive_colab()
        _repo15 = Path(globals().get('REPO', '/content/MADRLCitytleranflexresdr'))
        _scripts15 = str(_repo15 / 'CityLearn' / 'scripts')
        if _scripts15 not in sys.path:
            sys.path.insert(0, _scripts15)
        import importlib as _il15
        _cm15 = _il15.import_module('citylearn_v3_training_common')
        _ctx15 = _cm15.prepare_colab_drive_mount_context(
            Path(DRIVE_MOUNT_POINT),
            project_name=PROJECT_NAME,
            repo=_repo15,
        )
        GDRIVE_ROOT = _ctx15['gdrive_root']
        GDRIVE_OUTPUT_PARENT = _ctx15['outputs_parent']
        os.makedirs(GDRIVE_OUTPUT_PARENT, exist_ok=True)
        _cm15.print_colab_drive_mount_report(_ctx15)
        print(f'[OK] Carpeta canonica: {CANONICAL_DRIVE_URL}')
        print('[INFO] 2.1 lee checkpoints en outputs/ (sin mirror ni copia).')

        # ── Verificar espacio libre en Drive ────────────────────────────────
        try:
            usage = shutil.disk_usage('/content/drive/MyDrive')
            free_gib = usage.free / (1024 ** 3)
            total_gib = usage.total / (1024 ** 3)
            if free_gib < MIN_DRIVE_FREE_GIB:
                raise RuntimeError(
                    f"Espacio insuficiente en Google Drive: {free_gib:.1f} GiB libre "
                    f"(total {total_gib:.0f} GiB). Se necesitan >= {MIN_DRIVE_FREE_GIB} GiB. "
                    "Libera espacio antes de entrenar."
                )
            print(f"[OK] Drive espacio libre: {free_gib:.1f} GiB / {total_gib:.0f} GiB")
        except RuntimeError:
            raise
        except Exception as _de:
            print(f"[WARN] No se pudo verificar espacio en Drive: {_de}")

        # ── Cuarentena clone legacy en Drive (scripts 9+3 no deben ejecutarse) ──
        _LEGACY_DRIVE_ROOT = '/content/drive/MyDrive/MADRL_CityLearn_v3/MADRLCitytleranflexresdr'
        _legacy_launcher = f'{_LEGACY_DRIVE_ROOT}/CityLearn/scripts/colab_a100_official_launcher.py'
        _guard_py15 = f'{globals().get("REPO", "/content/MADRLCitytleranflexresdr")}/CityLearn/scripts/colab_protocol_guard.py'
        if os.path.isdir(_LEGACY_DRIVE_ROOT):
            print(f'[WARN] Clone legacy en Drive detectado: {_LEGACY_DRIVE_ROOT}')
            if os.path.isfile(_legacy_launcher):
                _leg_src = open(_legacy_launcher, encoding='utf-8').read()
                if (
                    'FASE 1: HAPPO + MATD3' in _leg_src
                    or 'run_two_phase_jobs' in _leg_src
                    or 'two_phase_happo_masac_v3' not in _leg_src
                ):
                    if os.path.isfile(_guard_py15):
                        subprocess.check_call(
                            [sys.executable, _guard_py15, 'quarantine-legacy-drive']
                        )
                    else:
                        raise RuntimeError(
                            'Launcher legacy 9+3 en Drive. Borra o renombra '
                            f'{_LEGACY_DRIVE_ROOT}/CityLearn/scripts antes de entrenar.'
                        )
            print('  Codigo SOLO desde /content/MADRLCitytleranflexresdr (celda 1.2).')

    except Exception as exc:
        if REQUIRE_GOOGLE_DRIVE:
            raise RuntimeError(
                'Google Drive es obligatorio para este entrenamiento largo. '
                'Sigue los pasos del mensaje anterior (cualquier cuenta Google con '
                'Colab Pro+ y espacio en Drive) y vuelve a ejecutar 1.5.'
            ) from exc
        print('Drive no disponible; usando outputs local del runtime:', exc)
        GDRIVE_ROOT = None
        GDRIVE_OUTPUT_PARENT = None

## Sección 2: Configuración del proyecto

### 2.1 `OUTPUT_ROOT` en Drive (MyDrive + carpeta compartida, sin mirror)

La celda **2.1** audita `outputs/` en MyDrive y en la carpeta compartida canónica ([`1ihH6RqL2KpevfCQEUXj7PP1aS2QYssAX`](https://drive.google.com/drive/folders/1ihH6RqL2KpevfCQEUXj7PP1aS2QYssAX)) en **solo lectura** y elige `OUTPUT_ROOT` sin copiar ni mirror.

| Flag (opcional) | Default | Efecto |
|---|---|---|
| `AUTO_RESUME_LATEST` | `True` | Reanuda el run más completo por artefactos |
| `FORCE_NEW_RUN` | `False` | Ignora runs previos y crea `madrl_v3_<timestamp>` |
| `RESUME_OUTPUT_ROOT` | `None` | Ruta exacta de un run concreto (override manual) |
| `ENABLE_FUSE_MIRROR` | `False` | Solo si MyDrive está vacío: bootstrap opcional desde carpeta compartida (1ª vez) |

Flujo mínimo: **1.5** → **2.1** → **2.1b** (preview SKIP/REANUDA). **2.2/2.3** quedan en `skip` salvo recuperación.


In [ ]:
# ── 2.1  OUTPUT_ROOT (nuevo o resume automatico; 2.2/2.3 quedan en skip) ──
import json
import os
import sys
from datetime import datetime
from pathlib import Path

# ── Deteccion automatica de REPO (Colab o local) ─────────────────────────────
import importlib.util
IN_COLAB = importlib.util.find_spec('google.colab') is not None

if IN_COLAB:
    # Codigo SIEMPRE en /content (hard sync celda 1.2). Outputs van a Drive.
    REPO = '/content/MADRLCitytleranflexresdr'
    CODE_ROOT = REPO
else:
    # Buscar repo root desde el directorio del notebook hacia arriba
    _start = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
    _candidates = [
        _start,
        _start.parent,
        _start.parent.parent,
        Path('d:/MADRLCitytleranflexresdr'),
        Path.home() / 'MADRLCitytleranflexresdr',
    ]
    REPO = next(
        (str(p) for p in _candidates if (p / 'CityLearn').exists()),
        str(_start)
    )
    CODE_ROOT = REPO

PROJECT_NAME = globals().get('PROJECT_NAME', 'MADRLCitytleranflexresdr')
TIMESTAMP    = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_LABEL    = f'madrl_v3_{TIMESTAMP}'
SCHEMA_PATH  = str(Path(REPO) / 'CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json')
PYTHON       = globals().get('PROJECT_PYTHON', globals().get('PYTHON', sys.executable))

GDRIVE_OUTPUT_PARENT = globals().get('GDRIVE_OUTPUT_PARENT', None)
GDRIVE_ROOT          = globals().get('GDRIVE_ROOT', None)
REQUIRE_GOOGLE_DRIVE = globals().get('REQUIRE_GOOGLE_DRIVE', False)

# Outputs en Drive: SOLO ruta canonica (nunca MADRL_CityLearn_v3 legacy)
_DRIVE_OUTPUT_CANDIDATES = []
if IN_COLAB:
    if GDRIVE_OUTPUT_PARENT:
        _DRIVE_OUTPUT_CANDIDATES.append(GDRIVE_OUTPUT_PARENT)
    else:
        _DRIVE_OUTPUT_CANDIDATES.append(
            '/content/drive/MyDrive/MADRLCitytleranflexresdr/outputs'
        )

BASE_OUTPUT_PARENT = None
for _cand in _DRIVE_OUTPUT_CANDIDATES:
    if _cand and Path(_cand).parent.exists():
        BASE_OUTPUT_PARENT = _cand
        break
if BASE_OUTPUT_PARENT is None:
    BASE_OUTPUT_PARENT = str(Path(REPO) / 'outputs')
# 2.1: audita outputs/ en Drive (MyDrive + carpeta compartida); reanuda o run nuevo.
AUTO_RESUME_LATEST = globals().get('AUTO_RESUME_LATEST', True)
FORCE_NEW_RUN      = globals().get('FORCE_NEW_RUN', False)
ENABLE_FUSE_MIRROR = globals().get('ENABLE_FUSE_MIRROR', False)  # opt-in: bootstrap solo si MyDrive vacio

RESUME_OUTPUT_ROOT = globals().get('RESUME_OUTPUT_ROOT', None)

_scripts_dir = str(Path(REPO) / 'CityLearn' / 'scripts')
if _scripts_dir not in sys.path:
    sys.path.insert(0, _scripts_dir)
sys.modules.pop('citylearn_v3_training_common', None)
import importlib
_common21 = importlib.import_module('citylearn_v3_training_common')
_pick = getattr(_common21, 'pick_colab_output_root', None)
if _pick is None:
    raise RuntimeError(
        'citylearn_v3_training_common desactualizado (falta pick_colab_output_root).\n'
        '  Ejecuta celda 1.2 (hard sync CityLearn) y vuelve a 2.1.'
    )

_n_ep_pick = int(globals().get('N_EPISODES', globals().get('EPISODES', 50)))
_ep_steps_pick = int(globals().get('EPISODE_STEPS', 8760))
_happo_roll_pick = int(globals().get('HAPPO_ROLLOUT_THREADS', 0)) or None
_gdrive_root_path = Path(GDRIVE_ROOT) if globals().get('GDRIVE_ROOT') else None
_mount_point_21 = Path(globals().get('DRIVE_MOUNT_POINT', '/content/drive'))

_picked = _pick(
    Path(BASE_OUTPUT_PARENT),
    run_label=RUN_LABEL,
    resume_output_root=RESUME_OUTPUT_ROOT,
    auto_resume_latest=AUTO_RESUME_LATEST,
    force_new_run=FORCE_NEW_RUN,
    target_episodes=_n_ep_pick,
    episode_time_steps=_ep_steps_pick,
    happo_rollout_threads=_happo_roll_pick,
    repo=Path(REPO),
    in_colab=IN_COLAB,
    gdrive_root=_gdrive_root_path,
    mount_point=_mount_point_21 if IN_COLAB else None,
    ensure_shared_run=bool(ENABLE_FUSE_MIRROR),
    skip_fuse_mirror=True,
    print_audit=True,
)
OUTPUT_ROOT = str(_picked['output_root'])
RESUME_OUTPUT_ROOT = _picked.get('resume_output_root')
_resume_reason = str(_picked['resume_reason'])
_created_new_run = bool(_picked.get('created_new_run'))
globals()['_created_new_run'] = _created_new_run
globals()['_MYDRIVE_RESUMED'] = bool(_picked.get('mydrive_resumed'))

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
(Path(REPO) / 'outputs').mkdir(parents=True, exist_ok=True)

# Solo en Colab: OUTPUT_ROOT debe estar bajo MyDrive/outputs/madrl_v3_*
if IN_COLAB and str(Path(OUTPUT_ROOT)).startswith('/content/drive/'):
    output_norm = str(Path(OUTPUT_ROOT)).replace('\\', '/')
    _drive_outputs_prefix = '/content/drive/MyDrive/'
    assert 'MADRL_CityLearn_v3' not in output_norm, (
        f'OUTPUT_ROOT en namespace legacy prohibido: {OUTPUT_ROOT}. '
        'Ejecuta celda 1.5 (Drive canonico) y 2.1 de nuevo.'
    )
    assert output_norm.startswith(_drive_outputs_prefix) and '/outputs/madrl_v3_' in output_norm, (
        f'OUTPUT_ROOT fuera del namespace Drive esperado: {OUTPUT_ROOT}'
    )

assert Path(SCHEMA_PATH).exists(), f'Schema Iquitos no encontrado: {SCHEMA_PATH}'

# Punteros + monitor (fuente unica; lo reusan 7.2/7.3 via latest_colab_output_root.txt)
_common21.sync_output_root_pointer_files(
    Path(REPO), Path(OUTPUT_ROOT), gdrive_root=_gdrive_root_path
)
if not (globals().get('_MYDRIVE_RESUMED') and not _created_new_run):
    _mon_rc = _common21.refresh_colab_live_monitor_once(
        Path(REPO), Path(OUTPUT_ROOT), python_executable=PYTHON, log_tail=8
    )
    if _mon_rc not in (0, 1):
        print(f'[2.1] Monitor snapshot salio con codigo {_mon_rc} (continuando)')

RUN_CONTEXT = dict(globals().get('COLAB_PROJECT_CONTEXT', {}))
RUN_CONTEXT.update({
    'timestamp': TIMESTAMP,
    'run_label': RUN_LABEL,
    'output_root': OUTPUT_ROOT,
    'in_colab': IN_COLAB,
    'repo': REPO,
    'schema_path': SCHEMA_PATH,
    'resumed_existing_output_root': bool(RESUME_OUTPUT_ROOT),
    'created_new_run': _created_new_run,
    'base_output_parent': BASE_OUTPUT_PARENT,
    'drive_required': REQUIRE_GOOGLE_DRIVE,
    'drive_project_root': GDRIVE_ROOT,
})
with open(f'{OUTPUT_ROOT}/run_context_manifest.json', 'w') as f:
    json.dump(RUN_CONTEXT, f, indent=2)

print(f"Entorno     : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"CODE_ROOT   : {CODE_ROOT}  (codigo fuente — launcher/monitor)")
print(f"OUTPUT_ROOT : {OUTPUT_ROOT}  (checkpoints/artefactos)")
print(f"MODO RUN    : {_resume_reason}")
print(f"TIMESTAMP   : {TIMESTAMP}")
print(f"SCHEMA_PATH : {SCHEMA_PATH}  {'OK' if Path(SCHEMA_PATH).exists() else 'NO ENCONTRADO'}")
print(f"Contexto    : {OUTPUT_ROOT}/run_context_manifest.json")


### 2.1b Verificación de reanudación (ejecutar después de 2.1)

Lista, para los **12 jobs**, si **7.2** hará **SKIP** (completo), **REANUDA** desde checkpoint o **fresh** — misma lógica que `--skip-completed` del launcher (`preview_job_launcher_decision()`). **Nunca** marca COMPLETO si el launcher imprimiría `RUN ... not skipping`.

**Vuelve a ejecutar esta celda después de 7.1** (cuando `HAPPO_ROLLOUT_THREADS` ya está fijado). La celda 7.1 también imprime el mismo preview antes de 7.2.

**Flujo tras desconexión:** `1.1 → 1.5 → 2.1 → 2.1b → 6.1 → 7.0 → 7.1 → [2.1b otra vez] → 7.2` · **2.3** solo si HAPPO salvage 49/50 sin KPIs.

| Estado en reporte | Acción recomendada |
|-------------------|-------------------|
| HAPPO `REANUDA 49/50` salvage sin KPIs | Opcional: **2.3** (`dry_run` → `execute`); **7.2** también puede cubrir tails |


In [ ]:
# ── 2.1b  Preview SKIP/REANUDA/fresh para 12 jobs (misma logica que --skip-completed) ──
# Fuente unica: notebook_jobs_resume_preview() en citylearn_v3_training_common.
from pathlib import Path

assert 'OUTPUT_ROOT' in globals(), 'Ejecuta la celda 2.1 primero.'
root = Path(str(OUTPUT_ROOT))
assert root.exists(), f'NO EXISTE OUTPUT_ROOT: {OUTPUT_ROOT}'

if globals().get('_created_new_run'):
    print('AVISO: run NUEVO — los 12 jobs empezaran fresh en 7.2 (salvo rescates en 2.2/2.3).\n')
elif not bool(globals().get('RESUME_OUTPUT_ROOT')):
    print('AVISO: sin RESUME_OUTPUT_ROOT — revisa la auditoria de 2.1.\n')

_repo21b = Path(globals().get('CODE_ROOT', globals().get('REPO', '.')))
_common21b = _common21 if '_common21' in globals() else None
if _common21b is None:
    _common21b = __import__('importlib').import_module('citylearn_v3_training_common')

_report = _common21b.notebook_jobs_resume_preview(
    root,
    target_episodes=int(globals().get('N_EPISODES', globals().get('EPISODES', 50))),
    episode_time_steps=int(globals().get('EPISODE_STEPS', 8760)),
    happo_rollout_threads=int(globals().get('HAPPO_ROLLOUT_THREADS', 0)) or None,
    label='2.1b',
    show_footer_hint=True,
    require_canonical_plan=not bool(globals().get('_created_new_run')),
)


### 2.1c Limpieza de runs `madrl_v3_*` duplicados (Drive)

Detecta carpetas `madrl_v3_*` redundantes en `outputs/` (reinicios de Colab). **Conserva el run activo (`OUTPUT_ROOT`) y el más completo**; nunca borra el que tiene más jobs terminados.

- Por defecto **solo reporta** (`DELETE_DUPLICATE_RUNS = False`).
- Para borrar: `DELETE_DUPLICATE_RUNS = True` y re-ejecuta.

Segura tras **2.1**; si no hay duplicados, no hace nada.


In [ ]:
# ── 2.1c  Limpieza de runs madrl_v3_* duplicados ───────────────────────────
# Usa plan_madrl_duplicate_run_cleanup() (misma puntuacion por artefactos que 2.1).
import shutil
from pathlib import Path as _Path

DELETE_DUPLICATE_RUNS = globals().get('DELETE_DUPLICATE_RUNS', False)

_parent = _Path(globals().get('BASE_OUTPUT_PARENT', '') or '.')
_active = _Path(globals().get('OUTPUT_ROOT', '')).resolve() if globals().get('OUTPUT_ROOT') else None
_common21c = _common21 if '_common21' in globals() else None
if _common21c is None:
    import importlib
    _common21c = importlib.import_module('citylearn_v3_training_common')

_plan = _common21c.plan_madrl_duplicate_run_cleanup(
    _parent,
    active_output_root=_active,
    target_episodes=int(globals().get('N_EPISODES', globals().get('EPISODES', 50))),
    episode_time_steps=int(globals().get('EPISODE_STEPS', 8760)),
    happo_rollout_threads=int(globals().get('HAPPO_ROLLOUT_THREADS', 0)) or None,
)

if not _plan.get('summaries'):
    print(f'[2.1c] No hay carpetas madrl_v3_* en {_parent} -> nada que limpiar.')
else:
    print(f'[2.1c] Runs madrl_v3_* en {_parent} (score por artefactos MADRL):')
    for s in _plan['summaries']:
        r = _Path(str(s['output_root']))
        _flag = '  <- ACTIVO' if _active and r.resolve() == _active else ''
        _keep = '  [CONSERVAR]' if str(r) in _plan['keep'] else ''
        print(
            f"   {r.name:30s} completos={s.get('completed_jobs', 0)}/12  "
            f"reanudables={s.get('resumable_jobs', 0)}  "
            f"~{float(s.get('progress_pct') or 0.0):.1f}% ep{_flag}{_keep}"
        )
    print(f"\n[2.1c] CONSERVAR: {', '.join(_Path(p).name for p in _plan['keep'])}")
    if not _plan['delete']:
        print('[2.1c] No hay duplicados para borrar.')
    else:
        for p in _plan['delete']:
            r = _Path(p)
            if DELETE_DUPLICATE_RUNS:
                shutil.rmtree(r, ignore_errors=True)
                print(f'[2.1c] BORRADO   : {r.name}')
            else:
                print(f'[2.1c] borraria   : {r.name}')
        if not DELETE_DUPLICATE_RUNS:
            print('\n[2.1c] Pon DELETE_DUPLICATE_RUNS = True y re-ejecuta para borrar.')


### 2.2 (Opcional) Rescatar HAPPO de un run fallido

> **Fuera del flujo desde cero.** `RESCUE_MODE='skip'` por defecto — **no ejecutar** en lanzamiento normal.

Solo si un run previo murió (p. ej. MASAC OOM) y quieres conservar checkpoints HAPPO parciales antes de borrar o relanzar con `OUTPUT_ROOT` nuevo.

1. Define `FAILED_OUTPUT_ROOT` (y opcionalmente `HAPPO_RESCUE_ARCHIVE`).
2. Ejecuta → copia checkpoints + `live_progress.json` a `outputs/rescued_happo_*`.
3. Para retomar HAPPO en un run nuevo: **2.1** (nuevo timestamp) → esta celda en modo `inject` → **7.2**.

> El rescate no completa el experimento 12/12; solo preserva progreso HAPPO intra-job.


In [ ]:
# ── 2.2  (Opcional, RESCUE_MODE='skip' por defecto — fuera del flujo desde cero)  Rescate HAPPO (opcional; RESCUE_MODE='skip' por defecto) ───────────
# Fuera del flujo from-scratch / two_phase. Solo si un run fallo y quieres conservar HAPPO.
import json
import subprocess
import sys
from pathlib import Path

RESCUE_MODE = 'skip'  # 'skip' | 'rescue' | 'inject'
FAILED_OUTPUT_ROOT = None  # ej. '/content/drive/.../outputs/madrl_v3_20260624_175429'
HAPPO_RESCUE_ARCHIVE = None  # default: outputs/rescued_happo_<run_name>

if 'OUTPUT_ROOT' not in globals():
    raise RuntimeError('Ejecuta celda 2.1 antes de 2.2.')

_repo = Path(globals().get('REPO', Path.cwd()))
_script = _repo / 'CityLearn' / 'scripts' / 'colab_rescue_happo_checkpoints.py'
if not _script.is_file():
    raise FileNotFoundError(f'No se encuentra {_script}. Ejecuta celda 1.2 (git sync).')

_py = globals().get('PYTHON', sys.executable)

if RESCUE_MODE == 'skip':
    print('[2.2] RESCUE_MODE=skip — sin rescate HAPPO.')
elif RESCUE_MODE == 'rescue':
    if not FAILED_OUTPUT_ROOT:
        raise ValueError('Define FAILED_OUTPUT_ROOT con el OUTPUT_ROOT del run fallido.')
    cmd = [_py, '-B', str(_script), 'rescue', '--source-run', str(FAILED_OUTPUT_ROOT)]
    if HAPPO_RESCUE_ARCHIVE:
        cmd.extend(['--dest', str(HAPPO_RESCUE_ARCHIVE)])
    print('[2.2] rescue:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=str(_repo))
    _archive = Path(HAPPO_RESCUE_ARCHIVE) if HAPPO_RESCUE_ARCHIVE else _repo / 'outputs' / f"rescued_happo_{Path(FAILED_OUTPUT_ROOT).name.replace('madrl_v3_', '')}"
    _manifest = _archive / 'rescue_manifest.json'
    if _manifest.is_file():
        print(json.dumps(json.loads(_manifest.read_text(encoding='utf-8')), indent=2)[:2000])
    print(f'[2.2] Archive: {_archive}')
elif RESCUE_MODE == 'inject':
    if not HAPPO_RESCUE_ARCHIVE:
        raise ValueError('Define HAPPO_RESCUE_ARCHIVE (directorio del rescate).')
    cmd = [_py, '-B', str(_script), 'inject', '--archive', str(HAPPO_RESCUE_ARCHIVE), '--target-run', str(OUTPUT_ROOT)]
    print('[2.2] inject:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=str(_repo))
    print(f'[2.2] HAPPO inyectado en {OUTPUT_ROOT}. Continua con 6.1 -> 7.2.')
else:
    raise ValueError(f'RESCUE_MODE invalido: {RESCUE_MODE!r}')


### 2.3 (Opcional) Completar HAPPO salvage (49→50) + KPIs

> **Fuera del flujo desde cero.** `HAPPO_KPI_MODE='skip'` por defecto.
> Solo al **reanudar** un run donde HAPPO tiene checkpoints pero `results.json` salvage **sin KPIs**.

1. Ejecuta **2.1** (mismo `OUTPUT_ROOT`).
2. `HAPPO_KPI_MODE = 'dry_run'` → preflight + comando resume.
3. `HAPPO_KPI_MODE = 'execute'` → 1 episodio restante × escenario + KPIs OE.
4. Continúa con **7.2** (si faltan jobs) o **8.x** (si ya están los 12).

> Entrenamiento desde cero: **no uses esta celda**. **7.2** + `--skip-completed` cubre tails HAPPO.


In [ ]:
# ── 2.3  (Opcional) Completar HAPPO salvage (49→50) + KPIs OE ────────────────
# Default HAPPO_KPI_MODE='skip': flujo desde cero / two_phase usa 7.2.
# Activa dry_run/execute solo si reanudas HAPPO salvage sin KPIs en results.json.
import json
import subprocess
import sys
from pathlib import Path

HAPPO_KPI_MODE = 'skip'  # 'skip' | 'dry_run' | 'execute'  (skip = flujo desde cero)
HAPPO_KPI_SCENARIOS = ('E1', 'E2', 'E3')
# A100 Colab: 2 rollouts (igual que 7.2). None = leer de hyperparameters del job.
HAPPO_KPI_ROLLOUT_THREADS = globals().get('HAPPO_ROLLOUT_THREADS', 2)

if 'OUTPUT_ROOT' not in globals():
    raise RuntimeError('Ejecuta celda 2.1 antes de 2.3.')

_repo = Path(globals().get('REPO', Path.cwd()))
_regen = _repo / 'CityLearn' / 'scripts' / 'regenerate_happo_kpis.py'
_prepare = _repo / 'CityLearn' / 'scripts' / 'prepare_happo_colab_resume.py'
_verify = _repo / 'CityLearn' / 'scripts' / 'colab_verify_critical_patches.py'
_py = globals().get('PYTHON', sys.executable)

for _p in (_regen, _prepare, _verify):
    if not _p.is_file():
        raise FileNotFoundError(f'No se encuentra {_p}. Ejecuta celda 1.2 (git sync).')

subprocess.run([_py, '-B', str(_verify), '--repo', str(_repo)], check=True, cwd=str(_repo))

_prep = [_py, '-B', str(_prepare), '--output-root', str(OUTPUT_ROOT), '--sync-salvage']
if HAPPO_KPI_ROLLOUT_THREADS is not None:
    _prep.extend(['--happo-rollout-threads', str(int(HAPPO_KPI_ROLLOUT_THREADS))])
print('[2.3] preflight:', ' '.join(_prep))
_prep_rc = subprocess.run(_prep, cwd=str(_repo)).returncode
if _prep_rc != 0 and HAPPO_KPI_MODE == 'execute':
    raise RuntimeError(
        '[2.3] Preflight falló: monta OUTPUT_ROOT en Drive con checkpoints HAPPO '
        'o ejecuta primero dry_run.'
    )

if HAPPO_KPI_MODE == 'skip':
    print('[2.3] HAPPO_KPI_MODE=skip — sin resume.')
else:
    for _scen in HAPPO_KPI_SCENARIOS:
        cmd = [
            _py, '-B', str(_regen),
            '--output-root', str(OUTPUT_ROOT),
            '--scenario', _scen,
            '--sync-salvage',
        ]
        if HAPPO_KPI_ROLLOUT_THREADS is not None:
            cmd.extend(['--n-rollout-threads', str(int(HAPPO_KPI_ROLLOUT_THREADS))])
        if HAPPO_KPI_MODE == 'execute':
            cmd.append('--execute')
        else:
            cmd.append('--dry-run')
        print('[2.3]', ' '.join(cmd))
        subprocess.run(cmd, check=True, cwd=str(_repo))
        _job = Path(OUTPUT_ROOT) / 'HAPPO' / _scen
        _results = _job / 'data' / 'results.json'
        if not _results.is_file():
            _results = _job / 'results.json'
        if _results.is_file():
            _payload = json.loads(_results.read_text(encoding='utf-8'))
            _axis = _payload.get('project_axis_metrics') or {}
            print(
                f'[2.3] {_scen}: status={_payload.get("status")!r} '
                f'KPIs={bool(_axis)} recorded={_payload.get("episodes_recorded")} '
                f'checkpoints={_payload.get("checkpoint_count")}'
            )
        else:
            print(f'[2.3] {_scen}: sin results.json en {_job}')
    if HAPPO_KPI_MODE == 'execute':
        print('[2.3] Re-ejecuta agregador: python tools/eval/aggregate_colab_drive_kpis.py')


## Sección 3: Dataset Iquitos 2023-2025

**17 edificios reales** · 26 304 pasos horarios · 222 CSV sin NaN/Inf · schema CityLearn v2 compatible con MADRL v3.

| Recurso | Detalle |
|---|---|
| Período | 2023-2025 · año horario completo (8 760 pasos/episodio de entrenamiento) |
| BESS total | 26 266 kWh / 6 648 kW |
| PV total | 48 790 kWp (PVGIS TMY/pvlib) |
| EV chargers | 185 tomas · 96 equipos · 1 850 EVs en pool |
| V2G | 31 tomas de camiones (B01 Electro Oriente) |
| Intensidad carbono | 0.671-0.790 kgCO₂/kWh (MINAM RAGEI 2019) |
| Tarifa punta (18-22h) | 0.38 USD/kWh · fuera punta: 0.26 USD/kWh |


In [ ]:
# ── 3.1  Verificar estructura del dataset Iquitos 2023-2025 ──────────────────
# Valida dataset LOCAL real del proyecto. NO modifica ningún archivo.
# Columnas verificadas son las del dataset real (snake_case CityLearn v2).
import json
import os
import pandas as pd
from pathlib import Path

DATASET_DIR = Path(REPO) / "CityLearn/data/datasets/citylearn_iquitos_2023_2025"

with open(SCHEMA_PATH) as f:
    schema = json.load(f)

buildings = schema.get("buildings", {})
n_blds = len(buildings)
assert n_blds == 17, f"Se esperaban 17 edificios, encontrados: {n_blds}"
print("Dataset         : citylearn_iquitos_2023_2025")
print(f"Schema          : {SCHEMA_PATH}")
print(f"Edificios       : {n_blds} / 17  OK")
print(f"Pasos simulacion: {schema.get('simulation_end_time_step', 0) + 1}  (26304 = 3 años)")
print(f"Agente central  : {schema.get('central_agent', False)}")
print()

# ── Columnas reales del Building CSV (snake_case CityLearn v2) ─────────────────
# Fuente: Building_1.csv — mismo esquema en los 17 edificios
BUILDING_REQUIRED_COLS = [
    "month",
    "hour",
    "day_type",
    "non_shiftable_load",       # carga electrica no desplazable [kWh]
    "solar_generation",         # generacion FV del edificio [W/kW]
    "cooling_demand",           # demanda de enfriamiento [kWh]
    "dhw_demand",               # agua caliente sanitaria [kWh]
    "heating_demand",           # calefaccion [kWh]
]

# ── Columnas reales del weather CSV (compartido por todos los edificios) ───────
WEATHER_REQUIRED_COLS = [
    "outdoor_dry_bulb_temperature",
    "outdoor_relative_humidity",
    "direct_solar_irradiance",
    "diffuse_solar_irradiance",
]

# ── Validar weather.csv (compartido) ──────────────────────────────────────────
weather_csv = DATASET_DIR / "weather.csv"
assert weather_csv.exists(), f"weather.csv no encontrado: {weather_csv}"
df_weather = pd.read_csv(weather_csv)
assert len(df_weather) == 26304, f"weather.csv: se esperaban 26304 filas, hay {len(df_weather)}"
missing_weather = [c for c in WEATHER_REQUIRED_COLS if c not in df_weather.columns]
assert not missing_weather, f"Columnas faltantes en weather.csv: {missing_weather}"
print(f"weather.csv     : {len(df_weather)} filas x {len(df_weather.columns)} cols  OK")

# ── Validar carbon_intensity.csv (compartido) ─────────────────────────────────
carbon_csv = DATASET_DIR / "carbon_intensity.csv"
assert carbon_csv.exists(), "carbon_intensity.csv no encontrado"
df_carbon = pd.read_csv(carbon_csv)
assert len(df_carbon) == 26304, f"carbon_intensity.csv: {len(df_carbon)} filas (esperado 26304)"
assert "carbon_intensity" in df_carbon.columns, "Columna 'carbon_intensity' no encontrada"
print(f"carbon_intensity: {len(df_carbon)} filas | rango [{df_carbon['carbon_intensity'].min():.3f}, {df_carbon['carbon_intensity'].max():.3f}] kgCO2/kWh  OK")

# ── Validar pricing.csv (tarifas eléctricas Iquitos) ─────────────────────────
pricing_csv = DATASET_DIR / "pricing.csv"
if pricing_csv.exists():
    df_price = pd.read_csv(pricing_csv)
    assert len(df_price) == 26304
    print(f"pricing.csv     : {len(df_price)} filas | rango [{df_price['electricity_pricing'].min():.3f}, {df_price['electricity_pricing'].max():.3f}] USD/kWh  OK")
else:
    print("pricing.csv     : no disponible (tarifas integradas en schema)")
print()

# ── Validar Building CSVs + PV + BESS + EV ───────────────────────────────────
ev_buildings = 0
bess_buildings = 0
pv_buildings = 0
csv_errors = []

print(f"{'Edificio':<22} {'Filas':>6} {'BldCols':>7} {'PV kW':>8} {'BESS kWh':>9} {'EV':>5}")
print("-" * 60)

for i, (name, bld) in enumerate(buildings.items()):
    # CSV de energía del edificio
    csv_rel = bld.get("energy_simulation", f"{name}.csv")
    csv_full = DATASET_DIR / csv_rel
    try:
        df_bld = pd.read_csv(csv_full)
        row_ok = len(df_bld) == 26304
    except Exception as e:
        csv_errors.append((name, str(e)))
        df_bld = pd.DataFrame()
        row_ok = False

    # Columnas obligatorias del Building CSV
    cols_ok = all(col in df_bld.columns for col in BUILDING_REQUIRED_COLS) if not df_bld.empty else False

    # PV — campo 'pv' -> 'attributes' -> 'nominal_power'
    pv_info = bld.get("pv", {})
    pv_kw = pv_info.get("attributes", {}).get("nominal_power", 0) if pv_info else 0
    if pv_kw > 0:
        pv_buildings += 1

    # BESS — campo 'electrical_storage' -> 'attributes' -> 'capacity'
    bess_info = bld.get("electrical_storage", {})
    bess_kwh = bess_info.get("attributes", {}).get("capacity", 0) if bess_info else 0
    if bess_kwh > 0:
        bess_buildings += 1

    # EV chargers — campo 'chargers' es un dict con un entry por punto de carga
    chargers = bld.get("chargers", {})
    n_ev = len(chargers)
    if n_ev > 0:
        ev_buildings += 1

    # Verificar que los CSV de chargers existen
    for ch_name, ch_data in chargers.items():
        ch_csv = ch_data.get("charger_simulation", "")
        ch_full = DATASET_DIR / ch_csv
        if ch_csv and not ch_full.exists():
            csv_errors.append((f"{name}/{ch_name}", f"charger CSV falta: {ch_csv}"))

    if i < 6 or i >= n_blds - 2:
        row_str = str(len(df_bld)) if not df_bld.empty else "ERR"
        print(
            f"{name:<22} {row_str:>6} {'OK' if cols_ok else 'ERR':>7}"
            f" {pv_kw:>8.0f} {bess_kwh:>9.0f} {n_ev:>5}"
        )
    elif i == 6:
        print(f"  ... ({n_blds - 8} edificios más) ...")

print()
print(f"Edificios con PV   : {pv_buildings}/{n_blds}")
print(f"Edificios con BESS : {bess_buildings}/{n_blds}")
print(f"Edificios con EV   : {ev_buildings}/{n_blds}")
total_ev_points = sum(len(bld.get("chargers", {})) for bld in buildings.values())
print(f"Puntos carga EV    : {total_ev_points}  (IEC 61851 Modo 3 CA)")
total_bess_kwh = sum(
    bld.get("electrical_storage", {}).get("attributes", {}).get("capacity", 0)
    for bld in buildings.values()
)
print(f"BESS total         : {total_bess_kwh:.0f} kWh")
total_pv_kw = sum(
    bld.get("pv", {}).get("attributes", {}).get("nominal_power", 0)
    for bld in buildings.values()
)
print(f"PV total           : {total_pv_kw:.0f} kW")

if csv_errors:
    print(f"\nERRORES: {len(csv_errors)}")
    for bld_name, err in csv_errors:
        print(f"  {bld_name}: {err}")
    raise RuntimeError(f"{len(csv_errors)} archivos CSV no encontrados. Revisa el dataset.")

# ── Conteo final de archivos ───────────────────────────────────────────────────
import glob as _glob
total_csvs = len(_glob.glob(str(DATASET_DIR / "*.csv")))
charger_csvs = len(_glob.glob(str(DATASET_DIR / "charger_*.csv")))
print(f"\nTotal CSV dataset : {total_csvs}  (17 building + {charger_csvs} charger + 3 especiales + 17 washing)")
print("Dataset Iquitos 2023-2025: VALIDADO — PV / BESS / EV / Clima / CO₂ / Precios")


## Sección 4: Entorno Dec-POMDP — 17 agentes

**Dec-POMDP:** cada edificio es un agente con observación parcial local.
**CTDE:** el crítico usa el estado global durante entrenamiento; la ejecución es completamente local.

```
Dataset: citylearn_iquitos_2023_2025 (17 edificios reales, Iquitos 2023-2025)

Observación local oᵢ(t): HETEROGÉNEA por edificio (varía según nº de chargers EV)
  ├── Building_1:  64 dimensiones (3 chargers EV Modo 3 AC)
  ├── Building_2:  78 dimensiones (5 chargers EV)
  ├── Building_3:  92 dimensiones (7 chargers EV)
  └── ... (rango: 57–330 dims según chargers EV del edificio)

  Componentes por agente:
  ├── Tiempo (mes, hora, day_type)
  ├── Física edificio (non_shiftable_load, dhw_demand, cooling_demand, solar_generation)
  ├── BESS (SOC, nominal_power, acciones previas)
  ├── EV por charger (SOC_k, departure_time_k, required_soc_k, estimated_arrival_k, state_k)
  └── Señales globales (carbon_intensity, electricity_pricing, outdoor_dry_bulb_temp,
                        diffuse_solar_irradiance, direct_solar_irradiance)

Acción local aᵢ(t): HETEROGÉNEA por edificio
  ├── Building_1: 6 acciones  (BESS_charge + 3 EV_charger_power + 2 adicionales)
  ├── Building_2: 8 acciones
  └── ... (rango: 5–44 acciones según nº de EV chargers)

Nota HAPPO: share_param=False — cada edificio tiene política INDEPENDIENTE
            (patrones locales distintos: industrial vs residencial).
```

> **MAPPO no es baseline ni algoritmo de este flujo.** Solo antecedente metodológico de políticas compartidas; los 4 MADRL oficiales son HAPPO, MASAC, MATD3 y MAAC.


In [ ]:
# ── 4.1  Smoke test del entorno Dec-POMDP con dataset Iquitos 2023-2025 ──────
# IMPORTANTE: se pasa SCHEMA_PATH explícitamente para garantizar que el entorno
# usa citylearn_iquitos_2023_2025 y no el DEFAULT (citylearn_challenge_2022).
import json
import os
import subprocess
import sys

PYTHON = globals().get('PROJECT_PYTHON', globals().get('PYTHON', sys.executable))
REPO   = globals().get('REPO', '/content/MADRLCitytleranflexresdr')

smoke_code = r'''
import json, sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "CityLearn"))
from citylearn.v3.environment import make_citylearn_v3_env, describe_environment
from citylearn.v3.config import CityLearnV3ExperimentConfig
IQUITOS_SCHEMA = os.path.join(
    os.getcwd(),
    "CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json"
)
assert os.path.exists(IQUITOS_SCHEMA), f"Schema Iquitos no encontrado: {IQUITOS_SCHEMA}"
cfg = CityLearnV3ExperimentConfig()
results = {}
for scenario in ("E1", "E2", "E3"):
    env = make_citylearn_v3_env(
        cfg,
        schema_path=IQUITOS_SCHEMA,
        scenario=scenario,
        seed=0,
        episode_time_steps=4,
        reward_aggregation="team_mean",
        normalize_observations=True,
        madrl_algorithm="MATD3",
        use_citylearn_v3_reward=True,
    )
    try:
        desc = describe_environment(env)
        # Verificar que el dataset cargado es Iquitos (no challenge_2022)
        inner = env.env
        while hasattr(inner, "env"):
            inner = inner.env
        schema_root = inner.schema.get("root_directory", "") if isinstance(inner.schema, dict) else ""
        is_iquitos = "iquitos" in schema_root.lower() or "iquitos" in str(IQUITOS_SCHEMA).lower()
        obs, info = env.reset()
        agents = list(obs.keys())
        obs_dim = len(obs[agents[0]])
        acts = {a: env.action_space(a).sample() for a in env.agents}
        obs2, rews, terms, truncs, infos = env.step(acts)
        rew_mean = sum(float(r) for r in rews.values()) / len(rews)
        results[scenario] = {
            "num_agents": desc["num_agents"],
            "obs_dim": obs_dim,
            "action_dim": desc["action_dims"].get(agents[0], "?") if desc["action_dims"] else "?",
            "reward_function": desc.get("reward_function", "N/A"),
            "dataset": "iquitos_2023_2025" if is_iquitos else "WRONG_DATASET",
            "schema_root": schema_root,
            "reward_mean_step1": round(rew_mean, 5),
        }
    finally:
        env.close()
print(json.dumps(results, indent=2, default=str))
'''

result = subprocess.run(
    [PYTHON, '-c', smoke_code],
    cwd=REPO,
    capture_output=True,
    text=True,
    env=os.environ.copy(),
)
if result.stderr:
    # Filtrar mensajes INFO normales de CityLearn
    stderr_lines = [line for line in result.stderr.splitlines() if not line.startswith("INFO:")]
    if stderr_lines:
        print("\n".join(stderr_lines[-20:]))
if result.returncode != 0:
    raise RuntimeError(f'Smoke-test CityLearn v3 falló (exit={result.returncode})\n{result.stderr[-800:]}')

raw = [line for line in result.stdout.strip().splitlines() if line.strip().startswith("{") or line.strip().startswith('"') or line.strip().startswith("}")]
json_text = "\n".join(result.stdout.strip().splitlines())
results = json.loads(json_text)

print(f"{'Escenario':<10} {'Agentes':>8} {'Obs':>5} {'Act':>4} {'Dataset':>22} {'Rew(s1)':>10}")
print("-" * 66)
for sc, r in results.items():
    dataset_ok = r['dataset'] == 'iquitos_2023_2025'
    if not dataset_ok:
        raise RuntimeError(f"CRÍTICO: escenario {sc} usa dataset incorrecto: {r['schema_root']}")
    print(f"{sc:<10} {r['num_agents']:>8} {r['obs_dim']:>5} {str(r['action_dim']):>4} {'iquitos_2023_2025 ✓':>22} {r['reward_mean_step1']:>10.5f}")

print()
print(f"Reward function : {list(results.values())[0]['reward_function']}")
print(f"Python          : {PYTHON}")
print()
print("OK: Entorno Dec-POMDP verificado con dataset Iquitos 2023-2025 en E1/E2/E3.")
print("    reset() → step() funciona. CityLearn v3 conectado al dataset local del proyecto.")


## Sección 5: Función de recompensa multiobjetivo

La recompensa **durante entrenamiento** pondera flex/CO₂/costo por escenario (E1/E2/E3). La **evaluación post-train (8.x/9.x)** usa los **KPIs OE completos** de `objectives.py` (54 métricas), no un subconjunto reducido de 3 KPIs.

### Componentes de recompensa (v4)

| Componente | Descripción |
|---|---|
| **Flexibilidad** | peak_penalty + ramping_penalty + load_factor + ev_service |
| **CO₂** | carbon_emissions × carbon_intensity |
| **Costo** | electricity_cost × price_signal |
| **EV urgency** | SOC_deficit × 1/horas_hasta_salida |
| **BESS degradación** | C-rate penalty Arrhenius LiFePO₄ (v4) |

### Pesos por escenario (entrenamiento)

| Escenario | flex | carbon | cost |
|:---:|:---:|:---:|:---:|
| **E1** | **0.70** | 0.15 | 0.15 |
| **E2** | 0.15 | **0.70** | 0.15 |
| **E3** | 0.25 | 0.15 | **0.60** |

### Recompensa mixta CTDE (team_ratio = 0.70)
```
r_i_mix = 0.30 × r_i_local  +  0.70 × mean(r₁,...,r₁₇)
```


In [ ]:
# ── 5.1  Visualizar pesos de recompensa por escenario ────────────────────
# Nota: pesos E1/E2/E3 son de ENTRENAMIENTO; post-train usa KPIs OE completos (54).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import os

WEIGHTS = {
    "E1": {"Flexibilidad": 0.70, "CO₂": 0.15, "Costo": 0.15},
    "E2": {"Flexibilidad": 0.15, "CO₂": 0.70, "Costo": 0.15},
    "E3": {"Flexibilidad": 0.25, "CO₂": 0.15, "Costo": 0.60},
}
TEAM_RATIO  = 0.70
LOCAL_RATIO = 0.30
N_AGENTS    = 17

COLORS = ["#3b82f6", "#22c55e", "#f59e0b"]
LABELS = list(WEIGHTS["E1"].keys())

fig, axes = plt.subplots(1, 3, figsize=(13, 6.0), sharey=True)
fig.suptitle("Pesos de recompensa por escenario (CityLearnV3MADRLRewardFunction v4)",
             fontsize=13, fontweight="bold")
for ax, (sc, wts), in zip(axes, WEIGHTS.items()):
    vals = list(wts.values())
    bars = ax.bar(LABELS, vals, color=COLORS, edgecolor="white", linewidth=1.5, width=0.55)
    ax.set_title(f"Escenario {sc}", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 0.85)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.25)
    ax.set_facecolor("#f8fafc")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{v:.2f}", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.subplots_adjust(bottom=0.20)
# Formula annotation using Unicode (no LaTeX needed)
_local  = LOCAL_RATIO
_team   = TEAM_RATIO
_n      = N_AGENTS
_formula = (
    f"r_i_mix = {_local:.2f} × r_i_local  +  {_team:.2f} × mean(r₁,...,r₁₇)"
    f"          [team_ratio={_team}, local_ratio={_local}]"
)
fig.text(0.5, 0.05, _formula,
         ha="center", va="center", fontsize=12, style="italic",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="#eff6ff",
                   edgecolor="#3b82f6", alpha=0.9))

os.makedirs(f"{OUTPUT_ROOT}/figures", exist_ok=True)
plt.savefig(f"{OUTPUT_ROOT}/figures/reward_weights.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅  Figura: {OUTPUT_ROOT}/figures/reward_weights.png")

# ── Imprimir recompensa mixta CTDE ────────────────────────────────────────
SEP = "─" * 64
print("")
print(SEP)
print("  RECOMPENSA MIXTA CTDE  (Centralized Training, Decentralized Execution)")
print(SEP)
print(f"  r_i_mix = {LOCAL_RATIO:.2f} × r_i_local  +  {TEAM_RATIO:.2f} × mean(r₁,...,r₁{N_AGENTS})")
print(f"  local_ratio = {LOCAL_RATIO:.2f}   |   team_ratio = {TEAM_RATIO:.2f}   |   N_agentes = {N_AGENTS}")
print(SEP)
for sc, wts in WEIGHTS.items():
    print(f"  Escenario {sc}  r_i_local = " + " + ".join(f"{w:.2f}·{c}" for c, w in wts.items()))
    print(f"             r_i_mix  = {LOCAL_RATIO:.2f}·r_i_local + {TEAM_RATIO:.2f}·mean_equipo")
    print("             Pesos:   " + "   |   ".join(f"{c}={v:.2f}" for c, v in wts.items()))
    print("")
print(SEP)


## Sección 6: Hiperparámetros y estrategia two_phase

### Estrategia de entrenamiento en dos fases (Colab A100 / H100)

| Fase | Algoritmos | Jobs paralelos | Notas |
|:---:|:---|:---:|:---|
| **1** | HAPPO + MASAC | 6 (E1/E2/E3) | Arrancan los 6 a la vez; buffers según `SIX_JOB_*` (celda 6.1) |
| **2** | MATD3 + MAAC | 6 (E1/E2/E3) | Backfill dinámico: cada job de Fase 2 entra al terminar uno de Fase 1 (lightest-first) |

Regla (`run_dynamic_backfill_jobs`): la Fase 2 **nunca** arranca antes de que termine un job de Fase 1. Modo: `--execution-mode two_phase_happo_masac`. Reanudación: `--skip-completed`. HPs canónicos: **`SIX_JOB_*` en celda 6.1** (no tablas antiguas de buffers).


### 6.1 Fuente única de hiperparámetros (A100/H100 · 50 episodios/corrida)

Perfil `two_phase_happo_masac`: 6 jobs/fase (sin stagger). Celda **6.1** = **fuente única** (`N_EPISODES=50`, `SIX_JOB_*` CUDA/buffers, hilos auto por vCPU). El launcher aplica overrides de fase.

| Algoritmo | Parametro clave | Valor two_phase (`SIX_JOB_*`) | OOM retry |
|---|---|---|---|
| **HAPPO** | hidden / n_rollout_threads | 512 / **auto (vCPU)** | hidden 256 |
| **MASAC** | buffer ep / max GiB / critic_batch | **4 / 8.0 / 1** | 2 / 6.0 / 1 (+ CPU replay) |
| **MASAC** | rnn / qmix / hyper hidden | **64 / 32 / 64** | 64 / 32 / 64 |
| **MATD3** | batch / buffer / hidden | **256 / 4096 / 256** | 768 / 1M / 512 |
| **MAAC** | batch / buffer / hidden / num_updates | **768 / 450K / 768 / 12** | 512 / 500K / 512 / 8 |

| Global | Valor |
|---|:---:|
| Episodios × pasos | 50 × 8 760 (por corrida; `LAUNCH_FULL_TRAINING=True` en 7.2) |
| Torch threads | auto por fase (celda 6.1) |
| CUDA fraction HAPPO/MATD3/MAAC | **0.14–0.15** (`SIX_JOB_CUDA_FRAC`) |
| CUDA fraction MASAC | **0.16–0.22** (`SIX_JOB_MASAC_CUDA_FRAC`) |
| MASAC replay device | **cpu** |
| Reanudacion | `--skip-completed` |


In [ ]:
# ── 6.1  HPs canonicos: N_EPISODES=50, EXECUTION_MODE=two_phase_happo_masac, SIX_JOB_* ──
import os
import sys
import subprocess
import json
from pathlib import Path

# ── REPO: reusa el global fijado por 1.2 / 2.1 (fuente unica de deteccion) ──
import importlib.util
_in_colab_61 = importlib.util.find_spec('google.colab') is not None

REPO = globals().get('REPO', None)
if not REPO or not Path(REPO).exists():
    raise RuntimeError(
        'REPO no definido o inexistente. Ejecuta las celdas 1.2 y 2.1 (definen REPO) antes de 6.1.'
    )
if _in_colab_61 and 'MADRL_CityLearn_v3' in str(Path.cwd()):
    raise RuntimeError(
        f'CWD en clone legacy Drive: {Path.cwd()}. '
        'Abre el notebook desde GitHub o /content; ejecuta 1.2.'
    )
PYTHON      = globals().get('PROJECT_PYTHON', globals().get('PYTHON', sys.executable))
SCHEMA_PATH = globals().get(
    'SCHEMA_PATH',
    f'{REPO}/CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json',
)
LAUNCHER    = f'{REPO}/CityLearn/scripts/colab_a100_official_launcher.py'
MONITOR     = f'{REPO}/CityLearn/scripts/colab_a100_live_monitor.py'
PROTOCOL_GUARD = f'{REPO}/CityLearn/scripts/colab_protocol_guard.py'
if _in_colab_61:
    for _p in (LAUNCHER, MONITOR, PROTOCOL_GUARD):
        assert _p.startswith('/content/MADRLCitytleranflexresdr/'), (
            f'Ruta de codigo invalida en Colab: {_p}'
        )
        assert 'MADRL_CityLearn_v3' not in _p, f'Ruta legacy prohibida: {_p}'

# ── QUICK_TEST (off por defecto) ─────────────────────────────────────────
# Entrenamiento oficial: QUICK_TEST=False, N_EPISODES=50, LAUNCH_FULL_TRAINING=True (7.2).
# ── QUICK_TEST ────────────────────────────────────────────────────────────
# False → entrenamiento oficial: N_EPISODES=50 via two_phase + LAUNCH_FULL_TRAINING=True (7.2).
# True  → (a) celda 6.2: smoke 1 ep/algo; (b) si lanzas 7.2, EPISODES=3 (infra).
# No uses QUICK_TEST=True para resultados de tesis.
QUICK_TEST      = False
N_EPISODES      = 50           # Oficial: 50 ep/corrida (two_phase). No usar tablas HP antiguas; ver SIX_JOB_*.
EPISODES        = 3 if QUICK_TEST else N_EPISODES
EPISODE_STEPS   = 8760
NUM_ENV_STEPS   = EPISODES * EPISODE_STEPS
# Campaña canónica del proyecto: N_SEEDS=12 (seeds 0..11). Este notebook lanza
# 12 jobs = 4 algos × 3 escenarios × 1 semilla (SEED=0) como smoke/campaña single-seed.
# No expande a 4×3×12=144 jobs; para multi-seed, repetir el launcher con --seed 1..11.
N_SEEDS         = 12
SEEDS           = list(range(N_SEEDS))  # 0..11 (diseño multi-seed)
SEED            = 0                     # semilla activa de esta corrida (CLI/single-seed)

# ── Parametros de rendimiento (auto-ajuste H100 ~26 vCPU primario / A100 12 vCPU) ──
# Detecta las vCPU reales del runtime y reparte hilos por fase SIN sobre-suscribir.
# Funciona igual en A100 (12), H100 Pro+ (~26) o cualquier maquina, sin editar a mano.
import os as _os_cpu
try:
    _sched_affinity = getattr(_os_cpu, 'sched_getaffinity', None)
    if _sched_affinity is not None:
        USABLE_VCPUS = len(_sched_affinity(0))
    else:
        USABLE_VCPUS = _os_cpu.cpu_count() or 12
except Exception:
    USABLE_VCPUS = _os_cpu.cpu_count() or 12

def _alloc_phase1(vcpus, max_rollout=16):
    # Fase 1 = 3 HAPPO (torch+rollout) + 3 MASAC (torch). Maximiza rollout (paraleliza
    # env.step de HAPPO = el cuello CPU) manteniendo demanda total <= vcpus.
    # max_rollout=16: maquinas H100/Blackwell con muchas vCPU y RAM holgada (177 GiB).
    best = (1, 1)
    for torch_t in (1, 2):
        for rollout in range(1, max_rollout + 1):
            if 3 * (torch_t + rollout) + 3 * torch_t <= vcpus:
                best = (torch_t, rollout)
    return best

TWO_PHASE_P1_TORCH, HAPPO_ROLLOUT_THREADS = _alloc_phase1(USABLE_VCPUS)
# Referencia GPU: num_mini_batch auto = ceil(rollout/ref) mantiene VRAM HAPPO ~constante
# al subir rollouts (mas buffer numpy en RAM de sistema, mismo minibatch GPU).
HAPPO_GPU_ROLLOUT_REF = 8
TWO_PHASE_P2_TORCH = max(1, USABLE_VCPUS // 6)   # Fase 2: 6 jobs single-env (sin rollout)
TORCH_THREADS      = TWO_PHASE_P2_TORCH          # fallback global = fase 2
# Cap de concurrencia del backfill dinamico. 0 = auto (= ancho de fase = 6).
# En maquinas con mas vCPU que Colab (H100 ~26), subirlo solapa fase 1 y fase 2
# para usar nucleos ociosos. LIMITADO POR RAM: cada job de fase 2 (MATD3/MAAC) ocupa
# ~18 GiB; mantener total < RAM del sistema (8 jobs ~144 GiB en 177 GiB). NO sube el
# FPS por job (env.step de CityLearn es single-thread); solo acorta el makespan global.
# VRAM detectada UNA sola vez; la reusan el cap de concurrencia y las fracciones CUDA.
def _detect_vram_gib():
    try:
        import subprocess as _sp
        _mib = int(_sp.check_output(
            ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
            text=True, stderr=_sp.DEVNULL).strip().splitlines()[0])
        return _mib / 1024.0
    except Exception:
        return 80.0

_VRAM_GIB = _detect_vram_gib()
_vcpus_cap = USABLE_VCPUS   # reusa la deteccion de vCPU de arriba (no re-detectar)
# Auto-recomendacion: solapa fase 2 hasta donde la RAM lo permita (~18 GiB/job, margen 90%).
try:
    import psutil as _ps_cap
    _ram_gib_cap = _ps_cap.virtual_memory().total / 1024**3
except Exception:
    _ram_gib_cap = 177.0
_ram_budget_jobs = max(6, int((_ram_gib_cap * 0.90) // 18))
# Acotar TAMBIEN por VRAM (no solo RAM): la fase 1 (HAPPO+MASAC x3) debe caber en la GPU;
# solo se admiten jobs EXTRA de fase 2 en el headroom de VRAM. Sin esto, un cap por-RAM
# (177 GiB -> 8) rellena los slots libres con fase 2 (MAAC, lightest-first) en t=0 y sobre-
# suscribe una GPU de 96 GiB (3xHAPPO+3xMASAC+2xMAAC ~133 GiB). Mismo metodo y fracciones
# que _detect_vram_gib / SIX_JOB_*_CUDA_FRAC de abajo, y que el clamp del launcher.
_vram_gib_cap = _VRAM_GIB   # VRAM detectada una sola vez (ver _detect_vram_gib arriba)
_happo_frac_cap = 0.14 if _vram_gib_cap <= 85 else 0.15
_masac_frac_cap = 0.16 if _vram_gib_cap <= 85 else 0.22
_phase1_vram_cap = 3 * _happo_frac_cap * _vram_gib_cap + 3 * _masac_frac_cap * _vram_gib_cap
_extra_p2_cap = max(0, int((0.92 * _vram_gib_cap - _phase1_vram_cap)
                           // max(_happo_frac_cap * _vram_gib_cap, 1e-6)))
_vram_budget_jobs = 6 + _extra_p2_cap
# Cap final = min(RAM, VRAM, 12). Solo sube de 6 si hay vCPU ociosas (>12) Y cabe en VRAM.
_cap_auto = min(12, _ram_budget_jobs, _vram_budget_jobs)
MAX_CONCURRENT_JOBS = _cap_auto if (_vcpus_cap > 12 and _cap_auto > 6) else 0
print(f'[cap] concurrencia backfill = {MAX_CONCURRENT_JOBS or "auto(6)"} '
      f'(vCPU={_vcpus_cap}, RAM~{_ram_gib_cap:.0f} GiB -> {_ram_budget_jobs} jobs, '
      f'VRAM~{_vram_gib_cap:.0f} GiB -> {_vram_budget_jobs} jobs)')
_d1 = 3 * (TWO_PHASE_P1_TORCH + HAPPO_ROLLOUT_THREADS) + 3 * TWO_PHASE_P1_TORCH
_d2 = 6 * TWO_PHASE_P2_TORCH
print(f'[cpu] vCPUs={USABLE_VCPUS} | Fase1 torch={TWO_PHASE_P1_TORCH} rollout={HAPPO_ROLLOUT_THREADS}'
      f' -> {_d1}/{USABLE_VCPUS} | Fase2 torch={TWO_PHASE_P2_TORCH} -> {_d2}/{USABLE_VCPUS}')
import math as _math_happo
_happo_nmb = max(1, _math_happo.ceil(HAPPO_ROLLOUT_THREADS / HAPPO_GPU_ROLLOUT_REF))
print(f'[happo] num_mini_batch auto={_happo_nmb} (ref={HAPPO_GPU_ROLLOUT_REF} rollouts) '
      f'-> minibatch GPU ~{HAPPO_GPU_ROLLOUT_REF * EPISODE_STEPS} pasos (VRAM estable)')
LIVE_PROGRESS_INT    = 300   # snapshot cada 300 pasos; fps se calcula en live_progress.json
LIVE_HEARTBEAT_SEC   = 120   # heartbeat menos frecuente durante actualizaciones del backend
EST_MIN_PER_EPISODE  = 12    # prior fase (max HAPPO/MASAC); ETA real usa campo fps
EST_MIN_BY_ALGO      = {'happo': 11, 'masac': 15, 'matd3': 12, 'maac': 8}
# GPU 95.6 GiB (RTX PRO 6000 Blackwell / A100-80 / H100): batches GPU mas grandes
# (mejor gradiente, absorbido en tiempo ocioso de GPU -> sin costo de velocidad).
# Buffers en RAM y nº de updates se mantienen conservadores: subirlos no acelera
# (cuello = CPU env.step) y arriesga el OOM/contencion que ya corregimos.
# Cap POR PROCESO (torch.cuda.set_per_process_memory_fraction), NO es el total de GPU.
# MASAC critic_batch_size = episodios por update QMIX (Rashid et al. 2018: 32 ep x 60-120 pasos SMAC).
# CityLearn = 8760 pasos/ep -> critic_batch=1 (ver launch_citylearn_v3_iquitos_training.ps1).
SIX_JOB_CUDA_FRAC       = 0.14 if _VRAM_GIB <= 85 else 0.15   # HAPPO / MATD3 / MAAC
SIX_JOB_MASAC_CUDA_FRAC = 0.16 if _VRAM_GIB <= 85 else 0.22   # MASAC critic bursts
print(f'[gpu] VRAM={_VRAM_GIB:.0f} GiB | cap/job HAPPO={SIX_JOB_CUDA_FRAC*_VRAM_GIB:.1f} '
      f'MASAC={SIX_JOB_MASAC_CUDA_FRAC*_VRAM_GIB:.1f} GiB (per-process, no total)')
SIX_JOB_MASAC_BUF    = 4     # ep replay en RAM (2->4: mas muestras off-policy sin OOM 6-parallel)
SIX_JOB_MASAC_GIB    = 8.0   # cap preflight; replay CPU en 6-parallel estable
SIX_JOB_MASAC_BATCH  = 1     # episodios QMIX/update (NO transiciones GPU); 8760 pasos/ep
# Fase 2 (MATD3+MAAC x3): buffers RAM conservadores; batches GPU aprovechan los 96 GiB
# MATD3 alineado a la corrida v4 (ganadora, 3/3 exit_code=0): config conservadora y estable.
SIX_JOB_MATD3_BUF    = 4096  # v4 estable: replay corto (~0.5 ep); 17 politicas x doble critic
SIX_JOB_MATD3_BATCH  = 256   # v4 estable
SIX_JOB_MATD3_HIDDEN = 256   # v4 estable
SIX_JOB_MAAC_BUF     = 450_000  # 50ep*8760=438k; cap automatico en train_maac (~3 GiB/job)
SIX_JOB_MAAC_BATCH   = 768   # batch GPU mayor; updates intactos (no frenar wall-clock)
SIX_JOB_MAAC_HIDDEN  = 768
SIX_JOB_MAAC_UPDATES = 12
MONITOR_INTERVAL     = 120   # panel visible cada 2 min; evita saturar VS Code/Chrome
AUTO_DISCONNECT_COLAB = False  # True solo si quieres apagar el runtime al terminar; False mantiene el kernel vivo para 7.3+
AUTO_RUN_POST_TRAINING = True  # Tras 7.2 OK: verifica artefactos y ejecuta 7.3→9.x sin desconectar
POST_TRAINING_INCLUDE_SECTION_8 = True   # 8.1 (+export) y 8.2
POST_TRAINING_INCLUDE_SECTION_9 = True   # 9.1 (+resumen_comparativo)
LOG_TAIL             = 4     # pocas lineas por snapshot; mantiene el notebook liviano
ARTIFACT_PROFILE     = 'efficient'  # conserva results.json + timeseries + checkpoints
TRACE_INTERVAL       = 8760  # traza diagnostica 1 vez por episodio; no cada dia
TRACE_DETAIL         = 'compact'

# GPU_PROFILE: 'aws' aplica TF32 + expandable_segments en A100/H100/RTX PRO 6000.
# En two_phase, el cap por job lo fija SIX_JOB_CUDA_FRAC; este 0.92 es para runs
# de 1 sola GPU (no 6-parallel). Sirve para A100-80, H100-80 y Blackwell-96.
GPU_PROFILE          = 'aws'
CUDA_MEMORY_FRACTION = 0.92  # 92% VRAM (~88 GiB en Blackwell-96 / ~73.6 GiB en A100-80)

SCENARIOS  = ['E1', 'E2', 'E3']
ALGORITHMS = ['happo', 'masac', 'matd3', 'maac']

# ── Validar que OUTPUT_ROOT ya esta configurado (celda 2.1) ─────────────
if 'OUTPUT_ROOT' not in globals():
    raise RuntimeError(
        "OUTPUT_ROOT no definido. Ejecuta las celdas 1.x y 2.1 en orden antes de 6.1."
    )

mode = 'QUICK_TEST (3 ep)' if QUICK_TEST else (
    f'PRUEBA ({N_EPISODES} ep)' if N_EPISODES < 10 else f'FULL TRAINING ({N_EPISODES} ep)'
)
print(f'Modo          : {mode}')
print(f'Episodios     : {EPISODES} x {EPISODE_STEPS} pasos = {NUM_ENV_STEPS:,} pasos/corrida')
print(f'Corridas total: {len(SCENARIOS) * len(ALGORITHMS)} ({len(ALGORITHMS)} algos x {len(SCENARIOS)} escenarios)')
print(f'GPU profile   : {GPU_PROFILE} (A100 TF32 + expandable_segments)')
print(f'CUDA fraccion : {CUDA_MEMORY_FRACTION} ({CUDA_MEMORY_FRACTION*80:.0f} GiB reservados en A100-80GB)')
print(f'Output root   : {OUTPUT_ROOT}')
print(f'Launcher      : {LAUNCHER}')
EXECUTION_MODE = 'two_phase_happo_masac'
print(f'Ejecucion     : {EXECUTION_MODE} (2 fases: HAPPO+MASAC×3 → MATD3+MAAC×3; 6 paralelos/fase)')
# Budget A100 80GB / 167 GiB RAM (2 fases × 6 jobs — modo agresivo VRAM/RAM)
_cuda_frac = SIX_JOB_CUDA_FRAC
_masac_cuda_frac = SIX_JOB_MASAC_CUDA_FRAC
_gpu_phase = 3 * _masac_cuda_frac * _VRAM_GIB + 3 * _cuda_frac * _VRAM_GIB  # 6 jobs fase 1
_gpu_masac = 3 * 11.0   # replay CPU ~33 GiB RAM (buf=8); GPU solo batches
_ram_matd3 = 3 * 14.0   # 2M transitions/job
_ram_maac = 3 * 7.0     # 1M steps/job
_total_wall_h = 2 * EST_MIN_PER_EPISODE * EPISODES / 60
print(f'  VRAM fase ~{_gpu_phase:.0f}/{_VRAM_GIB:.0f} GiB (per-process caps) | MASAC replay ~{_gpu_masac:.0f} GiB RAM (3 jobs)')
print(f'  RAM MATD3 ~{_ram_matd3:.0f} GiB | MAAC ~{_ram_maac:.0f} GiB (3 jobs c/u)')
print(f'  Tiempo prior  : ~{EST_MIN_PER_EPISODE} min/ep/fase -> ~{_total_wall_h:.0f} h total (2 fases)')

# ─────────────────────────────────────────────────────────────────────────────
# HIPERPARAMETROS CENTRALIZADOS — A100 80GB · 50 episodios/corrida · gamma=0.9999
# Referencia: training_summary.json de corrida v4 + launcher defaults A100
# ─────────────────────────────────────────────────────────────────────────────
HYPERPARAMS = {
    "HAPPO": {
        # On-policy heterogeneous PPO (HARL) — 17 politicas independientes
        "actor_lr"          : 1e-4,
        "critic_lr"         : 5e-4,
        "gamma"             : 0.9999,
        "gae_lambda"        : 0.95,
        "clip_ratio"        : 0.2,       # ppo_clip_param
        "entropy_coef"      : 0.01,
        "value_loss_coef"   : 1.0,
        "batch_size"        : None,      # on-policy: usa rollout completo
        "update_epochs"     : 5,         # ppo_epoch
        "hidden_sizes"      : [512, 512],
        "max_grad_norm"     : 1.0,
        "action_aggregation": "mean",
        "share_param"       : False,     # HAPPO heterogeneo
        "use_recurrent"     : False,
        "n_rollout_threads" : HAPPO_ROLLOUT_THREADS,  # auto por vCPU (celda 6.1)
    },
    "MASAC": {
        # Off-policy multi-agent SAC + QMIX — acciones discretizadas (axis 89)
        # Buffers alineados a SIX_JOB_MASAC_* (fuente de verdad del launcher 6-parallel).
        "actor_lr"          : 3e-4,
        "critic_lr"         : 5e-4,
        "alpha_lr"          : 3e-4,
        "gamma"             : 0.9999,
        "tau"               : 0.005,     # fijo en backend QMIX (qmix_msac.soft_update)
        "batch_size"        : 512,
        "replay_buffer_size": SIX_JOB_MASAC_BUF,
        "max_replay_buffer_gib": SIX_JOB_MASAC_GIB,
        "update_frequency"  : 2,
        "rnn_hidden_dim"    : 64,
        "qmix_hidden_dim"   : 32,
        "hyper_hidden_dim"  : 64,
        "action_bins"       : 3,
        "n_discrete_actions": 89,
        "grad_norm_clip"    : 1.0,
        "actor_sample_times": 10,        # A100-80GB: 2x actualizaciones actor por paso critico
    },
    "MATD3": {
        # Off-policy Multi-Agent Twin Delayed DDPG — acciones continuas
        # Alineado a SIX_JOB_MATD3_* (config estable 6-parallel fase 2).
        "actor_lr"          : 3e-4,
        "critic_lr"         : 3e-4,
        "gamma"             : 0.9999,
        "tau"               : 0.005,
        "policy_noise"      : 0.2,       # ruido en actualizacion del target
        "noise_clip"        : 0.5,
        "policy_delay"      : 2,         # twin delayed: 1 actor cada 2 critic updates
        "batch_size"        : SIX_JOB_MATD3_BATCH,
        "replay_buffer_size": SIX_JOB_MATD3_BUF,
        "hidden_size"       : SIX_JOB_MATD3_HIDDEN,
        "max_grad_norm"     : 1.0,
        "train_interval"    : 50,
        "share_policy"      : False,
    },
    "MAAC": {
        # Off-policy SAC con critic de atencion multiagente — acciones discretas
        # Alineado a SIX_JOB_MAAC_* (6-parallel fase 2).
        "actor_lr"          : 3e-4,
        "critic_lr"         : 1e-3,
        "gamma"             : 0.9999,
        "tau"               : 5e-3,
        "batch_size"        : SIX_JOB_MAAC_BATCH,
        "attention_heads"   : 4,         # launcher build_jobs usa attend_heads=4 (estable)
        "hidden_dim"        : SIX_JOB_MAAC_HIDDEN,
        "replay_buffer_size": SIX_JOB_MAAC_BUF,
        "steps_per_update"  : 50,
        "num_updates"       : SIX_JOB_MAAC_UPDATES,
        "reward_scale"      : 10.0,
        "action_bins"       : 3,
        "n_discrete_actions": 89,
    },
}

print("Hiperparametros centralizados:")
for algo, hp in HYPERPARAMS.items():
    lr_a = hp.get("actor_lr", hp.get("lr", "N/A"))
    lr_c = hp.get("critic_lr", "N/A")
    gamma = hp.get("gamma", "N/A")
    bs = hp.get("batch_size", "rollout")
    print(f"  {algo:<6} actor_lr={lr_a}  critic_lr={lr_c}  gamma={gamma}  batch={bs}")


### 6.2 Prueba rápida de validación — 1 episodio por algoritmo

> **SOLO PARA VERIFICAR EL PIPELINE.** No usar como resultado de tesis.
> Entrenamiento oficial: **N_EPISODES = 50** + **7.2** (`LAUNCH_FULL_TRAINING=True`), reanudable con `--skip-completed`.

Activa `QUICK_TEST = True` en **6.1**, luego ejecuta esta celda. Comprueba que launcher/scripts cargan y cada algoritmo arranca 1 episodio corto (168 pasos).

Si `QUICK_TEST = False` (default), solo imprime instrucciones y no entrena.


In [ ]:
# ── 6.2  Prueba rapida de validacion — 1 episodio (NO es entrenamiento oficial) ──
# Solo verifica que el pipeline funciona. El entrenamiento oficial usa N_EPISODES=50 por corrida.
# Controla con QUICK_TEST: si True, ejecuta; si False, imprime instrucciones y sale.

_N_EPISODES_TEST = 1   # Prueba rapida: 1 episodio por corrida
_EPISODE_STEPS   = 168 # 1 semana en pasos horarios (rapido para validar)

print("=" * 70)
print("  PRUEBA RAPIDA DE VALIDACION — 1 episodio x algoritmo x escenario")
print("  Este bloque NO genera resultados de tesis.")
print("  Para entrenamiento oficial: ejecuta la Seccion 7 (N_EPISODES=50 por corrida).")
print("=" * 70)

if not globals().get('QUICK_TEST', False):
    print()
    print("  QUICK_TEST = False → prueba desactivada.")
    print("  Para activar: cambia QUICK_TEST = True en la celda 6.1.")
    print("  Para entrenamiento oficial: ejecuta directamente la celda 7.2.")
else:
    import subprocess
    import sys
    import os
    import json
    from pathlib import Path

    _REPO    = globals().get('REPO', '/content/MADRLCitytleranflexresdr')
    _PYTHON  = globals().get('PROJECT_PYTHON', globals().get('PYTHON', sys.executable))
    _SCHEMA  = globals().get('SCHEMA_PATH', f'{_REPO}/CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json')
    _LAUNCHER = f'{_REPO}/CityLearn/scripts/colab_a100_official_launcher.py'
    _OUT_ROOT = str(Path(globals().get('OUTPUT_ROOT', f'{_REPO}/outputs')) / 'quick_test')
    Path(_OUT_ROOT).mkdir(parents=True, exist_ok=True)

    _test_algos = ['happo', 'masac', 'matd3', 'maac']
    _test_scenarios = ['E1', 'E2', 'E3']
    _results_quick = {}

    for algo in _test_algos:
        for scenario in _test_scenarios:
            script = f'{_REPO}/CityLearn/scripts/train_citylearn_v3_{algo}.py'
            if not Path(script).exists():
                print(f"  [SKIP] {algo.upper()} {scenario}: script no encontrado")
                continue
            cmd = [
                _PYTHON, '-B', script,
                '--schema-path', _SCHEMA,
                '--scenario', scenario,
                '--episodes', str(_N_EPISODES_TEST),
                '--episode-time-steps', str(_EPISODE_STEPS),
                '--seed', '0',
                '--output-dir', f'{_OUT_ROOT}/{algo}/{scenario}_seed_0',
                '--gpu-profile', 'aws',
            ]
            print(f"  Probando {algo.upper()} {scenario} ...", end=' ', flush=True)
            try:
                r = subprocess.run(cmd, capture_output=True, text=True, timeout=300, cwd=_REPO)
                ok = r.returncode == 0
                _results_quick[f'{algo}_{scenario}'] = 'OK' if ok else f'ERROR(exit={r.returncode})'
                print('OK' if ok else f'FALLO (exit={r.returncode})')
                if not ok:
                    print('    stderr:', r.stderr[-300:])
            except subprocess.TimeoutExpired:
                _results_quick[f'{algo}_{scenario}'] = 'TIMEOUT'
                print('TIMEOUT (>300s)')
            except Exception as e:
                _results_quick[f'{algo}_{scenario}'] = f'EXCEPTION({e})'
                print(f'EXCEPCION: {e}')

    ok_count = sum(1 for v in _results_quick.values() if v == 'OK')
    total    = len(_results_quick)
    print()
    print(f"  Resultado prueba rapida: {ok_count}/{total} corridas OK")
    if ok_count == total:
        print("  ✅ Pipeline validado. Procede a la Seccion 7 para el entrenamiento oficial (50 ep).")
    else:
        failed = [k for k, v in _results_quick.items() if v != 'OK']
        print(f"  ⚠️  Fallos: {failed}")
        print("     Revisa logs antes de ejecutar el entrenamiento oficial.")


## Sección 7: Lanzamiento oficial recuperable

Orquestador único (`colab_a100_official_launcher.py`, protocolo `two_phase_happo_masac_v3`):
**Fase1** HAPPO+MASAC → **Fase2** MATD3+MAAC, 12 jobs, `official_full_status.json` / manifest, checkpoint/resume y monitor visible.

Defaults: `N_EPISODES=50`, `LAUNCH_FULL_TRAINING=True` en **7.2**. Celdas **2.2/2.3** no forman parte de este lanzamiento.


In [ ]:
# ── 7.0  Helpers de ejecucion y monitor ─────────────────────────────────────────
# IMPORTANTE: tras git pull, re-ejecuta celdas 1.2 → 6.1 → 7.0 → 7.1 antes de 7.2.
import subprocess
import sys
import os
import json
import re
from pathlib import Path


def run_cmd(cmd, *, cwd=REPO, check=True):
    print('\n' + '=' * 80)
    print(' '.join(str(c) for c in cmd))
    print('=' * 80)
    sys.stdout.flush()
    proc = subprocess.run(cmd, cwd=cwd, text=True, stderr=subprocess.PIPE)
    if proc.stderr:
        print(proc.stderr, end='', file=sys.stderr, flush=True)
    if check and proc.returncode != 0:
        stderr_snippet = (proc.stderr or '').strip()[-1500:]
        msg = f'Comando fallo con exit={proc.returncode}'
        if stderr_snippet:
            msg += f'\n--- stderr (ultimas 1500 chars) ---\n{stderr_snippet}'
        raise RuntimeError(msg)
    return proc.returncode


def _launcher_flags():
    """Devuelve el conjunto de flags --foo registrados en el launcher."""
    try:
        src = Path(LAUNCHER).read_text(encoding='utf-8')
        return set(re.findall(r'add_argument\(["\'](-{1,2}[\w-]+)["\']', src))
    except Exception:
        return set()


def _launcher_has_two_phase():
    """True si el launcher implementa two_phase_happo_masac (no el layout antiguo 9+3)."""
    try:
        src = Path(LAUNCHER).read_text(encoding='utf-8')
        return (
            'run_two_phase_happo_masac_jobs' in src
            and 'TWO_PHASE_P1_HM' in src
            and ('two_phase_happo_masac' in src)
        )
    except Exception:
        return False


def _monitor_has_two_phase():
    try:
        src = Path(MONITOR).read_text(encoding='utf-8')
        return 'two_phase_happo_masac' in src and 'TWO_PHASE_P1' in src and 'TWO_PHASE_P2' in src
    except Exception:
        return False


def verify_two_phase_protocol():
    """Verifica launcher/monitor two_phase_happo_masac_v3 (Fase1 HAPPO+MASAC, Fase2 MATD3+MAAC; 6+6)."""
    launcher_path = Path(LAUNCHER)
    monitor_path = Path(MONITOR)
    if not launcher_path.exists() or not monitor_path.exists():
        raise RuntimeError(
            f'Scripts no encontrados: launcher={launcher_path} monitor={monitor_path}. '
            'Ejecuta celda 1.2 (hard reset) y vuelve a 7.0.'
        )
    launcher_src = launcher_path.read_text(encoding='utf-8')
    monitor_src = monitor_path.read_text(encoding='utf-8')
    required = [
        'run_two_phase_happo_masac_jobs',
        'TWO_PHASE_P1_HM',
        'LAUNCHER_PROTOCOL_ID',
        'two_phase_happo_masac_v3',
    ]
    forbidden = [
        'TWO_PHASE_LIGHT',
        'run_two_phase_jobs',
        'algo_sequential',
        'FASE 1: HAPPO + MATD3',
    ]
    missing = [s for s in required if s not in launcher_src]
    legacy = [s for s in forbidden if s in launcher_src]
    if missing or legacy:
        msg = ['Protocolo two_phase_happo_masac_v3 NO verificado en launcher.']
        if missing:
            msg.append(f'  Faltan: {missing}')
        if legacy:
            msg.append(f'  Layout antiguo detectado: {legacy}')
        msg.append('  Ejecuta celda 1.2 (checkout -B CityLearn) y re-ejecuta 6.1 -> 7.0 -> 7.1.')
        raise RuntimeError('\n'.join(msg))
    if 'MONITOR_PROTOCOL_ID' not in monitor_src or 'two_phase_happo_masac_v3' not in monitor_src:
        raise RuntimeError(
            'Monitor sin MONITOR_PROTOCOL_ID two_phase_happo_masac_v3. '
            'Ejecuta celda 1.2 (checkout -B CityLearn).'
        )
    print(f'[protocol] launcher={launcher_src.splitlines()[0][:40]}... OK')
    print('[protocol] verify_two_phase_protocol PASSED')
    if Path(PROTOCOL_GUARD).is_file():
        subprocess.check_call(
            [str(PYTHON), str(PROTOCOL_GUARD), 'verify-repo', '--repo', str(REPO)],
            cwd=str(REPO),
        )
    return True


_LAUNCHER_SCRIPTS = (
    'scripts/colab_a100_official_launcher.py',
    'scripts/colab_a100_live_monitor.py',
)


def _ensure_launcher_parallel():
    """Garantiza launcher + monitor two_phase (Fase1 HAPPO+MASAC → Fase2 MATD3+MAAC; 6+6).

    No degrada a argparse parcial ni al layout antiguo (Fase1 HAPPO+MATD3+MAAC x9).
    Orden: mac-tapia fetch → submodule update --remote → submodule init.
    """
    if _launcher_has_two_phase() and _monitor_has_two_phase():
        return True

    print('\n[launcher] Scripts desactualizados — se requiere two_phase_happo_masac.')
    if Path(LAUNCHER).exists():
        src = Path(LAUNCHER).read_text(encoding='utf-8')
        if 'TWO_PHASE_LIGHT' in src or 'run_two_phase_jobs' in src:
            print('[launcher] Detectado layout ANTIGUO (9+3 con stagger). Actualizando...')

    cl_dir = str(Path(LAUNCHER).parent.parent)
    _CL_BRANCH = globals().get('CITYLEARN_BRANCH', 'codex/iquitos-distillation-madrl-docs')
    _remotes = ('mac-tapia', 'origin')

    for remote in _remotes:
        r_fetch = subprocess.run(
            ['git', 'fetch', remote, _CL_BRANCH],
            cwd=cl_dir, capture_output=True, text=True, timeout=90
        )
        if r_fetch.returncode != 0:
            continue
        r_co = subprocess.run(
            ['git', 'checkout', f'{remote}/{_CL_BRANCH}', '--', *_LAUNCHER_SCRIPTS],
            cwd=cl_dir, capture_output=True, text=True, timeout=30
        )
        if r_co.returncode == 0 and _launcher_has_two_phase() and _monitor_has_two_phase():
            print(f'[launcher] Actualizado via {remote}/{_CL_BRANCH} — two_phase OK.')
            return True

    for cmd in (
        ['git', 'submodule', 'update', '--init', '--remote', 'CityLearn'],
        ['git', 'submodule', 'update', '--init', 'CityLearn'],
    ):
        r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True, timeout=120)
        if r.returncode == 0 and _launcher_has_two_phase() and _monitor_has_two_phase():
            print('[launcher] CityLearn sincronizado — two_phase OK.')
            return True

    print('[launcher] *** FALLO: no se pudo obtener two_phase_happo_masac. ***')
    print('[launcher] Reinicia runtime, ejecuta celda 1.2 y vuelve a 7.0/7.1.')
    return False


def launcher_base_args():
    # Todos los hiperparametros A100-SXM4-80GB explicitamente para maxima visibilidad.
    _flags = _launcher_flags()

    def _opt(flag, *values):
        return [flag, *values] if flag in _flags else []

    base = [
        PYTHON, '-B', LAUNCHER,
        '--scenario', 'ALL',
        '--seed', str(SEED),
        '--episode-time-steps', str(EPISODE_STEPS),
        '--episodes', str(EPISODES),
        '--schema-path', SCHEMA_PATH,
        '--output-root', OUTPUT_ROOT,
        '--torch-threads', str(TORCH_THREADS),
        '--live-progress-interval', str(LIVE_PROGRESS_INT),
        '--live-heartbeat-seconds', str(LIVE_HEARTBEAT_SEC),
        '--artifact-profile', ARTIFACT_PROFILE,
        '--trace-record-interval', str(TRACE_INTERVAL),
        '--trace-detail', TRACE_DETAIL,
        '--gpu-profile', GPU_PROFILE,
        '--cuda-memory-fraction', str(CUDA_MEMORY_FRACTION),
        '--require-a100',
        '--smoke-imports',
        '--oom-retry',
        # El panel del notebook (celda 7.2) ya renderiza el dashboard global con
        # componentes/kpis; --no-live-monitor evita duplicar la salida del launcher.
        '--no-live-monitor',
        # ── HAPPO ─────────────────────────────────────────────────────────────
        '--happo-hidden-size', '512',
        '--happo-n-rollout-threads', str(HAPPO_ROLLOUT_THREADS),
        # num_mini_batch=0 -> auto: mas rollouts usan RAM; minibatch GPU ~constante (VRAM).
        '--happo-num-mini-batch', '0',
        '--happo-gpu-rollout-ref', str(HAPPO_GPU_ROLLOUT_REF),
        # ppo_epoch/critic_epoch 5->10: mas pasadas GPU/update (aprovecha GPU ociosa on-policy)
        '--happo-ppo-epoch', '10',
        '--happo-critic-epoch', '10',
        # ── MASAC (6-parallel: replay CPU, critic_batch=1 ep QMIX) ────────────
        '--masac-critic-batch-size', str(SIX_JOB_MASAC_BATCH),
        '--masac-buffer-size', str(SIX_JOB_MASAC_BUF),
        '--masac-max-replay-buffer-gib', str(SIX_JOB_MASAC_GIB),
        '--masac-rnn-hidden-dim', '64',
        '--masac-qmix-hidden-dim', '32',
        '--masac-hyper-hidden-dim', '64',
        '--masac-preload-batch-device', 'auto',  # replay RAM (CPU) + batch en GPU (fallback CPU si OOM)
        '--masac-actor-sample-times', '1',
        '--masac-critic-train-steps', '1',
        # ── MATD3 (6-parallel fase 2: buffers RAM conservadores) ────────────────
        '--matd3-batch-size', str(SIX_JOB_MATD3_BATCH),
        '--matd3-buffer-size', str(SIX_JOB_MATD3_BUF),
        '--matd3-hidden-size', str(SIX_JOB_MATD3_HIDDEN),
        '--matd3-train-interval', '100',  # v4 estable (ganadora 3/3)
        # ── MAAC (6-parallel fase 2: buffers RAM conservadores) ────────────────
        '--maac-batch-size', str(SIX_JOB_MAAC_BATCH),
        '--maac-buffer-length', str(SIX_JOB_MAAC_BUF),
        '--maac-hidden-size', str(SIX_JOB_MAAC_HIDDEN),
        '--maac-steps-per-update', '50',
        '--maac-num-updates', str(SIX_JOB_MAAC_UPDATES),
    ]
    base += [
        '--execution-mode', 'two_phase_happo_masac',
        '--two-phase-torch-threads', str(TORCH_THREADS),
        # Auto-escala a las vCPU del runtime (sin sobre-suscribir): H100 ~26 vCPU
        # (primario) -> Fase1 torch=2/rollout=4; A100 12 vCPU -> torch=1/rollout=2.
        '--two-phase-p1-torch-threads', str(TWO_PHASE_P1_TORCH),
        '--two-phase-p2-torch-threads', str(TWO_PHASE_P2_TORCH),
        '--six-job-cuda-fraction', str(SIX_JOB_CUDA_FRAC),
        '--six-job-masac-cuda-fraction', str(SIX_JOB_MASAC_CUDA_FRAC),
        '--six-job-masac-buffer-size', str(SIX_JOB_MASAC_BUF),
        '--six-job-masac-max-replay-gib', str(SIX_JOB_MASAC_GIB),
        '--six-job-masac-critic-batch-size', str(SIX_JOB_MASAC_BATCH),
    ]
    if int(globals().get('MAX_CONCURRENT_JOBS', 0) or 0) > 0:
        base += ['--max-concurrent-jobs', str(int(MAX_CONCURRENT_JOBS))]
    return base


def monitor_once():
    """Snapshot unico del monitor (fuente: citylearn_v3_training_common)."""
    try:
        import importlib
        _cm = importlib.import_module('citylearn_v3_training_common')
        if hasattr(_cm, 'refresh_colab_live_monitor_once'):
            rc = _cm.refresh_colab_live_monitor_once(
                Path(REPO), Path(OUTPUT_ROOT),
                python_executable=PYTHON, log_tail=18,
            )
            return rc
    except Exception:
        pass
    return run_cmd([PYTHON, '-B', MONITOR, '--output-root', OUTPUT_ROOT, '--once', '--log-tail', '18'], check=False)


def resolve_output_root_or_latest():
    """OUTPUT_ROOT desde el scope, o puntero resuelto a ruta absoluta.

    Fuente unica del fallback que antes estaba copiado en 7.2/7.3/7.5.
    """
    _repo = Path(globals().get('REPO', '.'))
    _gdrive = globals().get('GDRIVE_ROOT')
    _gdrive_path = Path(_gdrive) if _gdrive else None
    _scripts = _repo / 'CityLearn' / 'scripts'
    if str(_scripts) not in sys.path:
        sys.path.insert(0, str(_scripts))
    import importlib
    _common = importlib.import_module('citylearn_v3_training_common')
    return _common.resolve_notebook_output_root(
        _repo,
        output_root=globals().get('OUTPUT_ROOT', ''),
        gdrive_root=_gdrive_path,
    )


def _launch_signature():
    """Firma de la config de lanzamiento; invalida el dry-run si algo cambia."""
    return (
        str(globals().get('OUTPUT_ROOT', '')),
        int(globals().get('EPISODES', 0) or 0),
        int(globals().get('HAPPO_ROLLOUT_THREADS', 0) or 0),
        int(globals().get('MAX_CONCURRENT_JOBS', 0) or 0),
    )


def mark_dry_run_validated():
    """Lo llama 7.1 tras un dry-run exitoso para que 7.2 no lo repita."""
    globals()['_DRY_RUN_VALIDATED'] = _launch_signature()


def dry_run_already_validated():
    """True si 7.1 ya valido el dry-run para esta misma config (OUTPUT_ROOT/episodios/hilos)."""
    return globals().get('_DRY_RUN_VALIDATED') == _launch_signature()


def _tutorial_notebook_path():
    repo = Path(globals().get('REPO', '/content/MADRLCitytleranflexresdr'))
    return repo / 'examples_madrl_v3/madrl_citylearn_v3_tutorial.ipynb'


POST_TRAINING_MARKERS = (
    '# ── 7.3',
    '# ── 7.4  Auditoría',
    '# ── 7.4b',
    '# ── 7.5',
    '#  7.6',
    '# ── 7.7',
)
SECTION_8_MARKERS = (
    '# ── 8.1 ',
    '# ── 8.1b',
    '# ── 8.2 ',
)
SECTION_9_MARKERS = (
    '# ── 9.1',
    '# ── 9.2',
)


def verify_training_artifacts_complete(output_root=None):
    """True cuando 12/12 jobs tienen KPIs auditados (episodes>=target, citylearn_v3_report)."""
    import importlib.util
    output_root = Path(output_root or resolve_output_root_or_latest())
    repo = Path(globals().get('REPO', '.'))
    common_mod = repo / 'CityLearn/scripts/citylearn_v3_training_common.py'
    spec = importlib.util.spec_from_file_location('_cl_v3_common_verify', common_mod)
    if spec is None or spec.loader is None:
        raise ImportError(f'No se pudo cargar citylearn_v3_training_common desde {common_mod}')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    n_ep = int(globals().get('N_EPISODES', globals().get('EPISODES', 50)))
    seed = int(globals().get('SEED', 0))
    report = mod.build_jobs_resume_report(
        output_root,
        target_episodes=n_ep,
        seed=seed,
        happo_rollout_threads=globals().get('HAPPO_ROLLOUT_THREADS'),
    )
    missing = []
    kpi_ok = 0
    for row in report.get('jobs', []):
        algo = str(row['algorithm']).lower()
        scen = str(row['scenario'])
        run_dir = mod.resolve_existing_job_run_dir(output_root, algo, scen, seed)
        if run_dir is None:
            missing.append(f'{algo.upper()}/{scen}: sin carpeta de run')
            continue
        if mod.job_meets_launcher_complete_requirements(run_dir, target_episodes=n_ep, output_root=output_root):
            kpi_ok += 1
            continue
        payload = mod.read_job_results_json(run_dir) or {}
        kpi_ep = mod._kpi_evaluated_episodes_from_results(payload)
        blockers = mod.job_launcher_completion_blockers(run_dir, target_episodes=n_ep, output_root=output_root)
        detail = f'kpi_ep={kpi_ep}/{n_ep}'
        if blockers:
            detail += ' | ' + '; '.join(blockers[:2])
        missing.append(f"{algo.upper()}/{scen}: {detail}")
    return {
        'ok': kpi_ok == 12 and not missing,
        'output_root': str(output_root),
        'jobs_complete': kpi_ok,
        'audit_ok': kpi_ok,
        'missing': missing,
        'report': report,
    }


def print_training_artifacts_verdict(verdict):
    print('\n[verify] Verificacion de artefactos post-entrenamiento')
    print(f"  OUTPUT_ROOT: {verdict['output_root']}")
    print(f"  Jobs con KPIs auditados (50/50): {verdict['jobs_complete']}/12")
    if verdict.get('missing'):
        print(f"  PROBLEMAS ({len(verdict['missing'])}):")
        for item in verdict['missing']:
            print(f'    x {item}')
    elif verdict.get('ok'):
        print('  OK 12/12 jobs completos con artefactos verificados.')
    else:
        print('  Verificacion incompleta — revisa celda 7.4.')


def run_post_training_notebook_cells(
    *,
    include_section_8: bool = True,
    include_section_9: bool = True,
):
    """Ejecuta celdas 7.3→7.7 y secciones 8/9 del tutorial (mismo kernel)."""
    import nbformat
    nb_path = _tutorial_notebook_path()
    if not nb_path.is_file():
        raise FileNotFoundError(f'Notebook no encontrado: {nb_path}')
    nb = nbformat.read(str(nb_path), as_version=4)
    markers: list[str] = list(POST_TRAINING_MARKERS)
    if include_section_8:
        markers.extend(SECTION_8_MARKERS)
    if include_section_9:
        markers.extend(SECTION_9_MARKERS)
    g = globals()
    ran = 0
    for cell in nb.cells:
        if cell.cell_type != 'code':
            continue
        src = cell.source or ''
        first = src.split('\n', 1)[0].strip()
        if not any(first.startswith(m) for m in markers):
            continue
        print(f'\n{"=" * 78}\n[post-train] >>> {first}\n{"=" * 78}')
        exec(compile(src, f'<{first}>', 'exec'), g)
        ran += 1
    tail = '9.x' if include_section_9 else ('8.x' if include_section_8 else '7.7')
    print(f'\n[post-train] {ran} celdas ejecutadas (7.3→{tail}).')
    return ran


### 7.1 Preflight y dry-run obligatorio

Valida GPU, CUDA, imports, rutas, manifest y los **12 comandos** planificados **sin entrenar**. Si falla aquí, no ejecutes **7.2**.

**Tras `git pull`:** re-ejecuta **1.2 → 6.1 → 7.0 → 7.1** antes de **7.2**.


In [ ]:
# ── 7.1  Preflight A100 + dry-run oficial ─────────────────────────────────────
# 0. Verificar existencia de launcher y schema
_launcher_path = Path(LAUNCHER)
_schema_path   = Path(SCHEMA_PATH)
if not _launcher_path.exists():
    raise FileNotFoundError(
        f'Launcher no encontrado: {LAUNCHER}\n'
        f'  → Vuelve a ejecutar la celda de clonado (1.2) para restaurar el submodulo CityLearn.'
    )
if not _schema_path.exists():
    raise FileNotFoundError(
        f'Schema no encontrado: {SCHEMA_PATH}\n'
        f'  → Genera el dataset Iquitos primero (celdas 3.x).'
    )

# 0b. Verificar protocolo two_phase_happo_masac_v3 (bloquea layout antiguo 9+3)
verify_two_phase_protocol()
_parallel_ok = _ensure_launcher_parallel()
if _parallel_ok:
    print(f'Launcher : {LAUNCHER}  [two_phase_happo_masac ✓]')
    print(f'Monitor  : {MONITOR}  [two_phase ✓]')
else:
    raise RuntimeError(
        'Launcher/monitor sin two_phase_happo_masac. Ejecuta celda 1.2 y vuelve a 7.0/7.1.'
    )
print(f'Schema   : {SCHEMA_PATH}')

# 1. Dry-run oficial: valida CUDA/A100, imports, rutas y 12 comandos planificados
dry_run_cmd = launcher_base_args() + ['--dry-run', '--skip-completed']
run_cmd(dry_run_cmd)
monitor_once()

# 2. Leer y validar status.json (desde colab_protocol_guard.py sincronizado en 1.2)
import importlib.util

_guard_path = Path(REPO) / 'CityLearn/scripts/colab_protocol_guard.py'
_spec = importlib.util.spec_from_file_location('_colab_protocol_guard', _guard_path)
if _spec is None or _spec.loader is None:
    raise ImportError(f'No se pudo cargar colab_protocol_guard desde {_guard_path}')
_pg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pg)

status_path = Path(OUTPUT_ROOT) / 'official_full_status.json'
with open(status_path) as f:
    status = json.load(f)
_pg.validate_dry_run_status(status)
_strategy = (status.get('parallelization') or {}).get('strategy', '')

# 3. Verificar que cada output_dir es unico y esta dentro de OUTPUT_ROOT
expected_root = Path(str(OUTPUT_ROOT)).resolve()
seen_outputs = set()
for job in status['jobs']:
    job_output = Path(job['output_dir'])
    if not job_output.is_absolute():
        job_output = Path(REPO) / job_output
    job_output = job_output.resolve()
    rel = job_output.relative_to(expected_root)
    parts = rel.parts
    assert len(parts) == 2, f'Layout inesperado: {job_output}'
    _algos_lower = {a.lower() for a in ALGORITHMS}
    assert parts[0].lower() in _algos_lower, f'Algoritmo inesperado en output_dir: {parts[0]}'
    assert parts[1].upper() in set(SCENARIOS), f'Scenario inesperado: {parts[1]}'
    seen_outputs.add(str(job_output))
assert len(seen_outputs) == 12, f'Output dirs duplicados o incompletos: {len(seen_outputs)}'

print('Dry-run validado: 12 jobs, 2 fases (6 paralelos/fase), sin stagger, outputs aislados en OUTPUT_ROOT.')
print(f'  strategy: {_strategy}')

# Marca el dry-run como validado: 7.2 omitira su dry-run interno si la config no cambia.
if 'mark_dry_run_validated' in globals():
    mark_dry_run_validated()

# 4. Preview skip/resume con HAPPO_ROLLOUT_THREADS ya fijado (misma fuente que 2.1b).
import importlib
_c71 = importlib.import_module('citylearn_v3_training_common')
_c71.notebook_jobs_resume_preview(
    Path(str(OUTPUT_ROOT)),
    target_episodes=int(globals().get('N_EPISODES', globals().get('EPISODES', 50))),
    episode_time_steps=int(globals().get('EPISODE_STEPS', 8760)),
    happo_rollout_threads=int(globals().get('HAPPO_ROLLOUT_THREADS', 0)) or None,
    label='7.1',
    show_footer_hint=False,
    require_canonical_plan=not bool(globals().get('_created_new_run')),
)


### 7.2 Entrenamiento de 50 episodios por corrida (reanudable con --skip-completed)

Ejecuta **12 corridas desde cero o reanudadas** en **dos fases** (`two_phase_happo_masac`):
- **Fase 1** HAPPO×3 + MASAC×3 (6 paralelos) → **Fase 2** MATD3×3 + MAAC×3 (6 paralelos)

Prior de tiempo: ~12 min/ep por fase → **~20 h** wall con 50 ep (ETA dinámico vía `live_progress.json`).

**Desde cero:** deja `LAUNCH_FULL_TRAINING = True` y ejecuta tras **6.1 → 7.0 → 7.1**.

**Reanudación:** re-ejecuta **solo 7.2**. Bootstrap: git hard sync, Drive, mismo `OUTPUT_ROOT`, dry-run interno y `--skip-completed`.

> **Opcional:** salvage HAPPO 49/50 — **7.2** cubre tails; **2.3** es atajo solo-KPI (`skip` por defecto).

Al terminar (exit=0): verifica 12/12 y, si `AUTO_RUN_POST_TRAINING=True`, encadena **7.3→9.x**.

**KPIs en entrenamiento:** al completar cada job, `write_training_artifacts` calcula y guarda los **54 OE** (OE1/OE2/OE3) a nivel **distrito** (`objective_kpis.csv`, `district_kpis.csv`) y **edificio** (`building_objective_kpis.csv`, `building_kpis.csv`) — no un subconjunto ad-hoc de 3 métricas.


In [ ]:
# ── 7.2  Lanzar entrenamiento + monitor en paralelo ─────────────────────────
# SOLO en VM Colab (A100). No ejecutar con kernel Python local en VS Code.
# True = entrenamiento oficial desde cero / resume (50 ep, two_phase). False = solo imprime ayuda.
LAUNCH_FULL_TRAINING = True

import importlib.util
_IN_COLAB_72 = importlib.util.find_spec('google.colab') is not None

# Bootstrap integral en Colab: git 1.2 + Drive 1.5 + OUTPUT_ROOT 2.1 + config 6.1/7.0.
if _IN_COLAB_72:
    import sys as _sys72
    import subprocess as _sp72
    from pathlib import Path as _Path72
    _repo72 = _Path72(globals().get('REPO', '/content/MADRLCitytleranflexresdr'))
    _helpers72 = _repo72 / 'CityLearn/scripts/colab_notebook_launch_helpers.py'
    if not _helpers72.is_file() and not (_repo72 / '.git').is_dir():
        print('[7.2] Runtime nuevo: clonando repo para bootstrap...')
        _sp72.check_call([
            'git', 'clone', '--branch', 'codex/fix-madrl-traceability-docs',
            '--depth', '1', '--recurse-submodules', '--shallow-submodules',
            'https://github.com/Mac-Tapia/MADRLCitytleranflexresdr.git', str(_repo72),
        ])
    if not _helpers72.is_file():
        raise RuntimeError(
            f'Falta {_helpers72}. Ejecuta celda 1.2 una vez o reinicia con REPO vacio.'
        )
    _spec72 = importlib.util.spec_from_file_location('_nb72_helpers', _helpers72)
    _mod72 = importlib.util.module_from_spec(_spec72)
    _spec72.loader.exec_module(_mod72)
    _mod72.prepare_colab_cell_72_standalone(
        globals(),
        repo=_repo72,
        resume_output_root=globals().get('RESUME_OUTPUT_ROOT'),
        skip_git_sync=bool(globals().get('CELL_72_SKIP_GIT_SYNC', False)),
    )

if LAUNCH_FULL_TRAINING and not _IN_COLAB_72:
    raise RuntimeError(
        'Entrenamiento oficial bloqueado en maquina local.\n'
        '  En VS Code: Select Kernel -> Google Colab -> A100 High-RAM (no .venv local).\n'
        '  O abre el notebook en colab.research.google.com y ejecuta 1.2 -> 7.2.\n'
        '  Local solo sirve para editar codigo / dry-run ligero; el compute es la VM Google.'
    )

if not LAUNCH_FULL_TRAINING:
    print('LAUNCH_FULL_TRAINING=False — cambia a True para entrenar.')
else:
    import signal as _signal
    import subprocess
    import sys
    import time
    import json as _json
    import threading as _th
    from pathlib import Path as _P
    from datetime import datetime as _DT, timezone as _TZ

    _repo    = globals().get('REPO', '/content/MADRLCitytleranflexresdr')
    if not Path(_repo).exists():
        _repo = next((p for p in ('d:/MADRLCitytleranflexresdr', str(Path.cwd()))
                      if (Path(p) / 'CityLearn').exists()), _repo)
    _python  = globals().get('PROJECT_PYTHON', globals().get('PYTHON', sys.executable))
    _MON_INTERVAL = int(globals().get('MONITOR_INTERVAL', 120))
    _POLL_SLEEP   = 10   # segundos entre polls del proceso; reduce CPU local del notebook
    _AUTO_DISCONNECT = bool(globals().get('AUTO_DISCONNECT_COLAB', False))

    def _disconnect_colab(reason, delay_s=20):
        if not _AUTO_DISCONNECT:
            print(f'\n[7.2] Kernel activo (AUTO_DISCONNECT_COLAB=False). {reason}.')
            return
        try:
            from google.colab import runtime as _colab_rt  # type: ignore[import-not-found]
            print(f'\n[7.2] Desconectando runtime Colab en {delay_s}s ({reason})...')
            sys.stdout.flush()
            time.sleep(delay_s)
            _colab_rt.unassign()
        except Exception as _dc_exc:
            print(f'[7.2] No se pudo desconectar Colab automaticamente: {_dc_exc}')

    _TWO_PHASE = (
        ('happo', 'masac'),
        ('matd3', 'maac'),
    )
    _EST_BY_ALGO = globals().get('EST_MIN_BY_ALGO', {'happo': 11, 'masac': 15, 'matd3': 12, 'maac': 8})
    _EST_PHASE = int(globals().get('EST_MIN_PER_EPISODE', 12))

    def _detect_hw_label():
        """Banner dinamico: GPU real (nvidia-smi) + RAM real, no valores fijos A100."""
        gpu = 'GPU?'
        try:
            _out = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
                text=True, stderr=subprocess.DEVNULL,
            ).strip().splitlines()[0]
            _name, _vram = _out.split(',')
            gpu = f'{_name.strip()} {int(_vram)/1024:.0f}GiB'
        except Exception:
            pass
        ram = ''
        try:
            with open('/proc/meminfo') as _f:
                for _ln in _f:
                    if _ln.startswith('MemTotal'):
                        ram = f" + {int(_ln.split()[1]) / (1024 * 1024):.0f}GiB RAM"
                        break
        except Exception:
            pass
        return f'{gpu}{ram}'

    _HW_LABEL = _detect_hw_label()

    def _job_done(j):
        if j.get('planned_only'):
            return False
        if j.get('skipped'):
            return True
        return j.get('exit_code') == 0

    def _job_running(j):
        return (j.get('completed_at') is None
                and not j.get('planned_only')
                and not j.get('skipped'))

    def _infer_phase(all_jobs):
        for phase_idx, algos in enumerate(_TWO_PHASE, 1):
            phase_jobs = [j for j in all_jobs if j.get('name') in algos]
            if not phase_jobs:
                continue
            if any(not _job_done(j) for j in phase_jobs):
                return phase_idx, algos
        return 0, ()

    _LP_STALE_SEC = int(globals().get('LIVE_PROGRESS_STALE_SEC', 600))

    def _ep_progress(lp, n_ep, ep_steps):
        """Grounded episode index + step for ETA and display (handles boundary writes)."""
        if not lp:
            return 0, 0, 0, 0
        completed = int(lp.get('completed_episode_count') or 0)
        ep = int(lp.get('episode', 0))
        ep_step = int(lp.get('episode_step', 0))
        if ep_step >= ep_steps:
            ep = min(ep + 1, n_ep)
            ep_step = 0
        ep_show = min(max(completed, ep + 1), n_ep)
        ep_step_show = ep_steps if ep_step >= ep_steps else ep_step
        return ep, ep_step, ep_show, ep_step_show

    def _eta_minutes(lp, algo, n_ep, ep_steps):
        if not lp:
            return n_ep * _EST_BY_ALGO.get(algo, 13)
        ep, ep_step, _, _ = _ep_progress(lp, n_ep, ep_steps)
        rem_steps = max(0, (n_ep - ep - 1) * ep_steps + (ep_steps - ep_step))
        fps = lp.get('fps')
        try:
            fps = float(fps)
        except (TypeError, ValueError):
            fps = 0.0
        if fps > 0.1:
            return rem_steps / fps / 60.0
        mins_per_ep = _EST_BY_ALGO.get(algo, 13)
        return (rem_steps / max(ep_steps, 1)) * mins_per_ep

    _SCENARIO_WEIGHTS = {
        'E1': {'OE1 flex': 0.70, 'OE2 CO2': 0.15, 'OE3 cost': 0.15},
        'E2': {'OE1 flex': 0.15, 'OE2 CO2': 0.70, 'OE3 cost': 0.15},
        'E3': {'OE1 flex': 0.25, 'OE2 CO2': 0.15, 'OE3 cost': 0.60},
    }
    _ALGOS    = ['HAPPO', 'MASAC', 'MATD3', 'MAAC']
    _SCENS    = ['E1', 'E2', 'E3']
    _SEP      = '=' * 78
    _SEP_THIN = '-' * 78

    # ── helpers ───────────────────────────────────────────────────────────────
    def _progress_bar(n, total, width=18):
        filled = int(width * n / max(total, 1))
        return '█' * filled + '░' * (width - filled)

    def _fmt_pct(v):
        if v is None:
            return '   N/A '
        sign = '+' if v >= 0 else ''
        return f'{sign}{v * 100:.1f}%'

    def _utc_now():
        return _DT.now(_TZ.utc)

    def _lag_seconds(ts_str):
        if not ts_str:
            return None
        try:
            ts = _DT.fromisoformat(ts_str.replace('Z', '+00:00'))
            return (_utc_now() - ts).total_seconds()
        except Exception:
            return None

    def _resolve_job_lp(out, job, lp_cache):
        algo = str(job.get('name', '')).lower()
        esc = str(job.get('scenario', '')).upper()
        key = (algo, esc)
        if key in lp_cache:
            return lp_cache[key]
        odir = str(job.get('output_dir') or '').strip()
        if not odir:
            return None
        lpf = _P(odir) if _P(odir).is_absolute() else out / odir
        lpf = lpf / 'live_progress.json'
        if not lpf.is_file():
            return None
        try:
            lp = _json.loads(lpf.read_text(encoding='utf-8'))
        except Exception:
            return None
        lag = _lag_seconds(lp.get('live_status_updated_at', ''))
        lp['_lag'] = lag
        lp['_stale'] = lag is None or lag > _LP_STALE_SEC
        lp_cache[key] = lp
        return lp

    def _collect_running_progress(out, all_jobs):
        lp_cache = {}
        for lpf in out.rglob('live_progress.json'):
            try:
                lp = _json.loads(lpf.read_text(encoding='utf-8'))
            except Exception:
                continue
            key = (str(lp.get('algorithm', '')).lower(),
                   str(lp.get('scenario', '')).upper())
            lag = _lag_seconds(lp.get('live_status_updated_at', ''))
            lp['_lag'] = lag
            lp['_stale'] = lag is None or lag > _LP_STALE_SEC
            lp_cache[key] = lp
        active_lp = []
        for job in all_jobs:
            if not _job_running(job):
                continue
            lp = _resolve_job_lp(out, job, lp_cache)
            if lp is None:
                lp = {
                    'algorithm': job.get('name', '?'),
                    'scenario': job.get('scenario', '?'),
                    '_lag': None,
                    '_stale': True,
                    'live_status': 'waiting_for_progress',
                }
            active_lp.append(lp)
        order = {a: i for i, a in enumerate(_ALGOS)}
        scen_order = {s: i for i, s in enumerate(_SCENS)}
        active_lp.sort(key=lambda lp: (
            order.get(str(lp.get('algorithm', '')).upper(), 99),
            scen_order.get(str(lp.get('scenario', '')).upper(), 99),
        ))
        return active_lp, lp_cache

    def _load_resume_report(out):
        """Misma fuente que 2.1b / 7.1 / --skip-completed (artefactos en disco)."""
        import importlib.util
        common_mod = _P(_repo) / 'CityLearn/scripts/citylearn_v3_training_common.py'
        spec = importlib.util.spec_from_file_location('_cl_v3_panel', common_mod)
        if spec is None or spec.loader is None:
            return {'jobs': [], 'completed': 0, 'resumable': 0, 'pending': 0, 'progress_pct': 0.0}
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        n_ep = int(globals().get('N_EPISODES', globals().get('EPISODES', 50)))
        seed = int(globals().get('SEED', 0))
        return mod.build_jobs_resume_report(
            _P(out),
            target_episodes=n_ep,
            seed=seed,
            episode_time_steps=int(globals().get('EPISODE_STEPS', 8760)),
            happo_rollout_threads=int(globals().get('HAPPO_ROLLOUT_THREADS', 0)) or None,
        )

    def _resumable_action(row):
        act = str((row or {}).get('action') or '').lower()
        return act in ('resume', 'happo_salvage_kpi')

    def _resume_row_map(report):
        rows = report.get('jobs') or []
        return {
            (str(r.get('algorithm', '')).lower(), str(r.get('scenario', '')).upper()): r
            for r in rows
        }

    def _artifact_complete(resume_map, algo, esc):
        row = resume_map.get((str(algo).lower(), str(esc).upper()))
        return bool(row and row.get('action') == 'skip')

    # ── panel principal de estado ─────────────────────────────────────────────
    def _print_panel(output_root):
        out = _P(output_root)
        now_str = _utc_now().strftime('%Y-%m-%d %H:%M:%S UTC')
        print('\n' + _SEP)
        print(f'  MADRL CityLearn v3  |  {_HW_LABEL}  |  {now_str}')
        print(f'  Run: {out.name}')
        _est_min_ep = int(globals().get('EST_MIN_PER_EPISODE', 13))
        _n_ep = int(globals().get('N_EPISODES', globals().get('EPISODES', 50)))
        _ep_steps = int(globals().get('EPISODE_STEPS', 8760))
        _total_prior_h = 2 * _EST_PHASE * _n_ep / 60
        _dyn_bf = bool(globals().get('DYNAMIC_BACKFILL', True))
        _mode_txt = ('backfill dinámico (6 en paralelo; fase 2 entra solo al terminar un job de fase 1)'
                     if _dyn_bf else '2 fases (6 jobs/fase)')
        print(f'  Modo: {globals().get("EXECUTION_MODE", "two_phase_happo_masac")} | '
              f'{_mode_txt} | prior ~{_total_prior_h:.0f} h (límite superior) | ETA dinámico con FPS')
        print(_SEP)

        # ── 1. Estado global (artefactos en disco = misma verdad que launcher) ──
        _resume = _load_resume_report(out)
        _resume_map = _resume_row_map(_resume)
        status_file = out / 'official_full_status.json'
        all_jobs = []
        st = {}
        if status_file.exists():
            try:
                st = _json.loads(status_file.read_text())
                all_jobs = st.get('jobs', [])
            except Exception as e:
                print(f'  [status] error leyendo official_full_status.json: {e}')

        done_art = int(_resume.get('completed') or 0)
        resume_n = int(_resume.get('resumable') or 0)
        pending_n = int(_resume.get('pending') or 0) + int(_resume.get('restart_fresh') or 0)
        fail = sum(1 for j in all_jobs if j.get('exit_code') not in (None, 0) and not j.get('skipped'))
        run = resume_n
        total = 12
        _gp = float(_resume.get('progress_pct') or 0) / 100.0 * total
        bar12 = _progress_bar(_gp, total, 24)
        print(f'\n  PROGRESO GLOBAL  [{bar12}]  '
              f'{done_art}/{total} OK (artefactos)  {run} activas  {fail} fallo  {pending_n} pendientes')
        if st:
            print(f'  status = {st.get("status", "?")}')
            _par = st.get('parallelization') or {}
            if _par:
                print(f'  paralelismo: {_par.get("strategy", "?")}')
        _dyn_bf = bool((st.get('parallelization') or {}).get('dynamic_backfill', globals().get('DYNAMIC_BACKFILL', True)))
        _p1_done = sum(1 for a in ('happo', 'masac') for s in _SCENS if _artifact_complete(_resume_map, a, s))
        _p2_done = sum(1 for a in ('matd3', 'maac') for s in _SCENS if _artifact_complete(_resume_map, a, s))
        _p2_run = sum(1 for a in ('matd3', 'maac') for s in _SCENS
                      if not _artifact_complete(_resume_map, a, s)
                      and _resumable_action(_resume_map.get((a, s))))
        if done_art >= total:
            print('  progreso     : completado (12/12 artefactos verificados)')
        elif _dyn_bf:
            print(f'  backfill dinámico | fase1 HAPPO+MASAC {_p1_done}/6 ok | '
                  f'fase2 MATD3+MAAC {_p2_done}/6 ok ({_p2_run} activos) | {run} activos total (cap 6)')
        else:
            print(f'  jobs por artefactos: {done_art} ok | {run} reanudables | {pending_n} pendientes')

        # ── 2. Live progress (solo jobs NO completos por artefacto) ─────────────
        _running_jobs = [
            j for j in all_jobs
            if _job_running(j)
            and not _artifact_complete(_resume_map, j.get('name'), j.get('scenario'))
        ]
        active_lp, lp_cache = _collect_running_progress(out, _running_jobs) if _running_jobs else ([], {})
        _lp_key = dict(lp_cache)

        try:
            _dyn_bf = bool(globals().get('DYNAMIC_BACKFILL', True))
            _cap = 6
            _rem = []
            _run_rem = []
            for row in _resume.get('jobs') or []:
                if row.get('action') == 'skip':
                    continue
                _a = str(row.get('algorithm', '')).lower()
                _s = str(row.get('scenario', '')).upper()
                _lp = _lp_key.get((_a, _s))
                if _resumable_action(row) and _lp:
                    _m = _eta_minutes(_lp, _a, _n_ep, _ep_steps)
                    _run_rem.append(_m)
                else:
                    _done_ep = int(row.get('completed_episodes') or 0)
                    _m = max(0, _n_ep - _done_ep) * _EST_BY_ALGO.get(_a, 13)
                _rem.append(_m)
            if _rem:
                _eta_run = max(_run_rem) if _run_rem else 0.0
                _eta_total = max(max(_rem), sum(_rem) / max(_cap, 1))
                print(f'  ETA activos: ~{_eta_run/60:.1f} h | '
                      f'ETA restante ({len(_rem)} jobs, cap {_cap}): ~{_eta_total/60:.1f} h')
        except Exception:
            pass

        if active_lp:
            print(f'\n  CORRIDAS ACTIVAS  ({len(active_lp)} en paralelo)')
            hdr = f'  {"ALGO/ESC":<10} {"Episodio":>10}  {"ep_step":>12}  {"FPS":>6}  {"r_mix_mean":>11}  {"Lag":>6}  Estado'
            print(hdr)
            print('  ' + _SEP_THIN[:76])
            def _cn(v, nd=3):
                try:
                    return f'{float(v):+.{nd}f}'
                except (TypeError, ValueError):
                    return '   —  '
            for lp in active_lp:
                algo  = lp.get('algorithm', '?').upper()
                esc   = lp.get('scenario', '?').upper()
                ep_steps = int(lp.get('episode_time_steps', _ep_steps))
                _, _, ep, ep_step = _ep_progress(lp, _n_ep, ep_steps)
                try:
                    fps = float(lp.get('fps') or 0.0)
                except (TypeError, ValueError):
                    fps = 0.0
                ret = lp.get('mean_return', lp.get('episode_reward_mean_cumulative'))
                lag = lp.get('_lag')
                lag_s = f'{lag:.0f}s' if lag is not None else '  n/d'
                if lp.get('_stale'):
                    lag_s += '!'
                bar_s = _progress_bar((ep - 1) + ep_step / max(ep_steps, 1), _n_ep, 16)
                ret_s = f'{float(ret):+.4f}' if ret is not None else '    —   '
                fps_s = f'{fps:.1f}' if fps > 0 else '  —  '
                stage = str(lp.get('live_status') or '?')
                if lp.get('backend_training_active'):
                    stage += '+GPU'
                print(f'  {algo}/{esc:<5}  [{bar_s}] {ep:>3}/{_n_ep}  {ep_step:>5}/{ep_steps:<5}  '
                      f'{fps_s:>6}  {ret_s:>11}  {lag_s:>6}  {stage}')
                print(f'  {"":<12}comp  flex={_cn(lp.get("reward_component_flex_mean"))} '
                      f'co2={_cn(lp.get("reward_component_carbon_mean"))} '
                      f'cost={_cn(lp.get("reward_component_cost_mean"))} '
                      f'ev={_cn(lp.get("reward_component_ev_mean"))} '
                      f'team={_cn(lp.get("reward_team_reward"))}')
                print(f'  {"":<12}kpi   cost={_cn(lp.get("district_net_electricity_consumption_cost"), 2)} '
                      f'co2={_cn(lp.get("district_net_electricity_consumption_emission"), 2)} '
                      f'load={_cn(lp.get("district_net_electricity_consumption"), 2)} '
                      f'price={_cn(lp.get("electricity_price_mean"))}')
        else:
            print('\n  (sin corridas activas aun — el launcher puede estar iniciando)')

        # ── 3. Pesos multiobjetivo ────────────────────────────────────
        print('\n  PESOS MULTIOBJETIVO  '
              'r_mix_i = 0.30 × r_local_i  +  0.70 × mean(r₁…r₁₇)')
        print(f'  {"Escenario":<12}  {"OE1 flex":>10}  {"OE2 CO2":>9}  {"OE3 cost":>10}')
        print('  ' + '-' * 46)
        for esc, w in _SCENARIO_WEIGHTS.items():
            print(f'  {esc:<12}  {w["OE1 flex"]:>10.2f}  {w["OE2 CO2"]:>9.2f}  {w["OE3 cost"]:>10.2f}')

        # ── 4. Ganancias de jobs completados ──────────────────────────
        gains_rows = {}
        for algo in _ALGOS:
            for esc in _SCENS:
                # Carpeta canonica del launcher = MAYUSCULA (HAPPO/E1); respaldo minuscula/legacy.
                # Identico para los 4 MADRL (sin preferencia) y robusto en Colab (case-sensitive).
                jdir = next(
                    (c for c in (out / algo.upper() / f'{esc}' / 'data',
                                 out / algo.lower() / f'{esc}' / 'data',
                                 out / algo.upper() / f'{esc}_seed_0' / 'data',
                                 out / algo.lower() / f'{esc}_seed_0' / 'data')
                     if c.exists()),
                    out / algo.upper() / f'{esc}' / 'data',
                )
                for fname in ('training_summary.json', 'results.json'):
                    jf = jdir / fname
                    if not jf.exists():
                        continue
                    try:
                        td = _json.loads(jf.read_text())
                        # Buscar claves de ganancia/improvement
                        gain_keys = [k for k in td
                                     if any(x in k.lower()
                                            for x in ('gain', 'improvement', 'delta',
                                                       'reduction', 'saving'))]
                        if gain_keys:
                            gains_rows[f'{algo}/{esc}'] = {
                                k: td[k] for k in gain_keys[:5]
                            }
                            break
                    except Exception:
                        pass

        if gains_rows:
            print('\n  GANANCIAS vs BASELINE (corridas completadas):')
            for key, gd in list(gains_rows.items())[:9]:
                parts = []
                for k, v in gd.items():
                    short = (k.replace('_gain', '').replace('_improvement', '')
                              .replace('_reduction', '').replace('_saving', ''))
                    try:
                        parts.append(f'{short}={_fmt_pct(float(v))}')
                    except Exception:
                        parts.append(f'{short}={v}')
                print(f'  {key:<14}  ' + '  '.join(parts))

        # ── 5. Tabla 4x3 (artefactos en disco, no solo status.json) ───
        if _resume_map:
            print('\n  TABLA DE CORRIDAS (4 algoritmos x 3 escenarios):')
            print(f'  {"ALGO":<8}  {"E1":^14}  {"E2":^14}  {"E3":^14}')
            print('  ' + '-' * 56)
            for algo in _ALGOS:
                cells = []
                for esc in _SCENS:
                    row = _resume_map.get((algo.lower(), esc))
                    match = [j for j in all_jobs
                             if j.get('name', '').upper() == algo
                             and j.get('scenario', '').upper() == esc]
                    if row and row.get('action') == 'skip':
                        cells.append(f'OK {_n_ep}/{_n_ep}ep')
                    elif row and _resumable_action(row):
                        _lk = (algo.lower(), esc)
                        _jlp = lp_cache.get(_lk) or _lp_key.get(_lk)
                        _ce = int(row.get('completed_episodes') or 0)
                        if match and _job_running(match[0]):
                            ep_str = f'ep{_ce}' if not _jlp else ''
                            if _jlp:
                                _, _, _ep_show, _ = _ep_progress(_jlp, _n_ep, _ep_steps)
                                ep_str = f'ep{_ep_show}'
                            cells.append(f'activo {ep_str:>4}')
                        elif str(row.get('action') or '').lower() == 'happo_salvage_kpi':
                            cells.append(f'cola ep{_n_ep:>2}')
                        else:
                            ep_str = f'ep{_ce + 1}' if _ce < _n_ep else f'ep{_n_ep}'
                            cells.append(f'reanuda {ep_str:>4}')
                    elif match and match[0].get('skipped'):
                        cells.append('  [SKIP]   ')
                    elif match and match[0].get('exit_code') not in (None, 0) and match[0].get('completed_at'):
                        cells.append('  [FALLO]  ')
                    elif row:
                        cells.append(' [pendiente]')
                    else:
                        cells.append('     —     ')
                print(f'  {algo:<8}  {cells[0]:^14}  {cells[1]:^14}  {cells[2]:^14}')

        print(_SEP + '\n')
        sys.stdout.flush()

    # ── arrancar proceso ──────────────────────────────────────────────────────
    verify_two_phase_protocol()
    if not globals().get('_created_new_run'):
        _scripts72 = _P(_repo) / 'CityLearn' / 'scripts'
        if str(_scripts72) not in sys.path:
            sys.path.insert(0, str(_scripts72))
        import importlib as _il72
        _cm72 = _il72.import_module('citylearn_v3_training_common')
        _out72 = _P(str(globals().get('OUTPUT_ROOT', '') or resolve_output_root_or_latest()))
        if not _out72.is_dir():
            raise RuntimeError(f'OUTPUT_ROOT no existe: {_out72}')
        _pre72 = _cm72.build_jobs_resume_report(
            _out72,
            target_episodes=int(globals().get('N_EPISODES', globals().get('EPISODES', 50))),
            episode_time_steps=int(globals().get('EPISODE_STEPS', 8760)),
            happo_rollout_threads=int(globals().get('HAPPO_ROLLOUT_THREADS', 0)) or None,
            seed=int(globals().get('SEED', 0)),
        )
        _cm72.assert_canonical_colab_skip_plan(_pre72, output_root=_out72)
    # Omite el dry-run interno si 7.1 ya lo valido para esta misma config (no duplicar).
    if 'dry_run_already_validated' in globals() and dry_run_already_validated():
        print('[preflight] dry-run ya validado en 7.1 para esta config — se omite el dry-run interno.')
    else:
        _preflight = launcher_base_args() + ['--dry-run', '--skip-completed']
        _pf = subprocess.run(_preflight, cwd=_repo, capture_output=True, text=True)
        if _pf.returncode != 0:
            print(_pf.stdout)
            print(_pf.stderr)
            raise RuntimeError(f'Preflight dry-run fallo exit={_pf.returncode}')
        if 'protocol=two_phase_happo_masac_v3' not in _pf.stdout:
            raise RuntimeError(
                'Launcher sin protocol=two_phase_happo_masac_v3 — scripts legacy en Colab. '
                'Runtime restart + celdas 1.2 -> 1.5 -> 2.1 -> 6.1 -> 7.1.'
            )
        _st_path = _P(globals().get('OUTPUT_ROOT', '')) / 'official_full_status.json'
        if _st_path.exists():
            _st = _json.loads(_st_path.read_text(encoding='utf-8'))
            import importlib.util as _ilu
            _gp = Path(_repo) / 'CityLearn/scripts/colab_protocol_guard.py'
            _gs = _ilu.spec_from_file_location('_pg_launch', _gp)
            if _gs is None or _gs.loader is None:
                raise ImportError(f'No se pudo cargar colab_protocol_guard desde {_gp}')
            _pgl = _ilu.module_from_spec(_gs)
            _gs.loader.exec_module(_pgl)
            _pgl.validate_dry_run_status(_st)
        print('[preflight] dry-run OK — two_phase_happo_masac_v3')
    train_cmd = launcher_base_args() + ['--skip-completed']
    print('\n' + _SEP)
    print('  Lanzando entrenamiento...')
    print('  protocol: two_phase_happo_masac_v3 | execution: two_phase_happo_masac')
    print('  ' + ' '.join(str(c) for c in train_cmd))
    print(_SEP + '\n')
    sys.stdout.flush()

    proc = subprocess.Popen(
        train_cmd,
        cwd=_repo,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    # Hilo que imprime cada linea del launcher en tiempo real
    def _stream(p):
        try:
            for line in p.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()
        except Exception:
            pass

    _th.Thread(target=_stream, args=(proc,), daemon=True).start()

    # ── stop gracioso ─────────────────────────────────────────────────────────
    def _graceful_stop(p):
        if p.poll() is not None:
            return
        print('\n[7.2] SIGINT al launcher (checkpoints se guardan)...')
        try:
            p.send_signal(_signal.SIGINT)
        except Exception:
            pass
        try:
            p.wait(timeout=40)
        except subprocess.TimeoutExpired:
            p.kill()  # SIGKILL fallback after graceful SIGINT timeout
            p.wait(timeout=10)

    # ── bucle de monitoreo ────────────────────────────────────────────────────
    _last_panel = time.monotonic()
    try:
        while proc.poll() is None:
            _now = time.monotonic()
            if _now - _last_panel >= _MON_INTERVAL:
                _root = resolve_output_root_or_latest()
                if _root and _P(_root).exists():
                    _print_panel(_root)
                _last_panel = _now
            time.sleep(_POLL_SLEEP)
    except KeyboardInterrupt:
        print('\n[7.2] Interrumpido.')
        _graceful_stop(proc)
        print('[7.2] Para reanudar: Interrupt → git pull (celda 1.2) → run ONLY celda 7.2.')
        raise

    time.sleep(1)   # dar tiempo al hilo de streaming para vaciar el buffer
    _exit = int(proc.returncode or 0)

    # Panel final
    _root = resolve_output_root_or_latest()
    if _root and _P(_root).exists():
        _print_panel(_root)

    if _exit == 0:
        print(_SEP)
        print('  ENTRENAMIENTO COMPLETADO')
        print(_SEP + '\n')
        _root_done = resolve_output_root_or_latest()
        _auto_post = bool(globals().get('AUTO_RUN_POST_TRAINING', True))
        _artifacts_ok = False
        if _root_done and 'verify_training_artifacts_complete' in globals():
            _verdict = verify_training_artifacts_complete(_root_done)
            print_training_artifacts_verdict(_verdict)
            _artifacts_ok = bool(_verdict.get('ok'))
        elif _root_done:
            print('[7.2] verify_training_artifacts_complete no disponible — re-ejecuta celda 7.0.')
        if _auto_post and _artifacts_ok and 'run_post_training_notebook_cells' in globals():
            print('\n[7.2] Artefactos OK — ejecutando pipeline post-entrenamiento (7.3→...)')
            try:
                run_post_training_notebook_cells(
                    include_section_8=bool(globals().get('POST_TRAINING_INCLUDE_SECTION_8', True)),
                    include_section_9=bool(globals().get('POST_TRAINING_INCLUDE_SECTION_9', True)),
                )
                print('\n[7.2] Pipeline post-entrenamiento finalizado. Kernel activo.')
            except Exception as _post_exc:
                print(f'\n[7.2] Post-proceso automatico fallo: {_post_exc}')
                print('  Ejecuta manualmente celdas 7.3 en adelante.')
        elif _auto_post and not _artifacts_ok:
            print('\n[7.2] Artefactos incompletos — no se ejecuta pipeline automatico.')
            print('  Corrige y ejecuta 7.4+ manualmente, o re-lanza 7.2 con --skip-completed.')
        elif not _auto_post:
            print('\n[7.2] AUTO_RUN_POST_TRAINING=False — ejecuta 7.3+ manualmente.')
        else:
            print('  Procede con seccion 7.3 — Monitor y auditoria.')
        _disconnect_colab('entrenamiento completado', delay_s=30)
    else:
        # Diagnostico de fallo
        print(f'\n[7.2] FALLO (exit={_exit})\n')
        if _root:
            _sp = _P(_root) / 'official_full_status.json'
            if _sp.exists():
                try:
                    _s = _json.loads(_sp.read_text())
                    for _j in _s.get('jobs', []):
                        if _j.get('exit_code') not in (None, 0) and not _j.get('skipped'):
                            print(f'  FAIL: {_j.get("name","?").upper()}/'
                                  f'{_j.get("scenario","?")}  '
                                  f'attempt={_j.get("attempt",0)}')
                except Exception:
                    pass
            for _ep in sorted(_P(_root).glob('logs/*.stderr.log')):
                if _ep.stat().st_size == 0:
                    continue
                _et = _ep.read_text(errors='replace')
                print(f'\n  === {_ep.name} (ultimas 25 lineas) ===')
                print('  ' + '\n  '.join(_et.strip().splitlines()[-25:]))
        print('\n  RELAUNCH: Interrupt → git pull (celda 1.2) → run ONLY celda 7.2.')
        _disconnect_colab(f'fallo exit={_exit} — reanuda con RESUME_OUTPUT_ROOT', delay_s=45)
        raise RuntimeError(f'Entrenamiento fallo exit={_exit}')




### 7.3 Monitor visible manual

Ejecuta tras el entrenamiento o al reabrir el notebook para ver el último estado. Muestra fase activa, jobs en paralelo y ETA. Durante **7.2**, el panel imprime snapshots cada `MONITOR_INTERVAL` s. Si `AUTO_RUN_POST_TRAINING=True` (celda 6.1), **7.2** lanza esta celda al completar 12/12.


In [ ]:
# ── 7.3  Monitor visible en notebook ────────────────────────────────────────  # protocolo two_phase_happo_masac_v3
# Autosuficiente: funciona aunque el kernel haya sido reiniciado.
import subprocess
import sys
import os
from pathlib import Path

_repo   = globals().get('REPO', '/content/MADRLCitytleranflexresdr')
_mon    = f'{_repo}/CityLearn/scripts/colab_a100_live_monitor.py'
_python = globals().get('PROJECT_PYTHON', globals().get('PYTHON', sys.executable))

# OUTPUT_ROOT del scope o del archivo de referencia del launcher.
# Usa el helper de 7.0 si existe; si no (kernel reiniciado), fallback autosuficiente.
if 'resolve_output_root_or_latest' in globals():
    _output_root = resolve_output_root_or_latest()
else:
    _output_root = globals().get('OUTPUT_ROOT', '')
    _ref = Path(_repo) / 'outputs' / 'latest_colab_output_root.txt'
    if not _output_root and _ref.exists():
        _output_root = _ref.read_text(encoding='utf-8').strip()

if not _output_root:
    print('[7.3] OUTPUT_ROOT no disponible. Ejecuta la celda 6.1 o espera a que el launcher escriba outputs/latest_colab_output_root.txt.')
else:
    if 'MADRL_CityLearn_v3' in _output_root:
        raise RuntimeError(
            f'OUTPUT_ROOT legacy prohibido: {_output_root}. Re-ejecuta 1.5 y 2.1.'
        )
    _guard = f'{_repo}/CityLearn/scripts/colab_protocol_guard.py'
    if Path(_guard).is_file():
        subprocess.check_call([_python, _guard, 'verify-repo', '--repo', _repo])
    result = subprocess.run(
        [_python, '-B', _mon, '--output-root', _output_root, '--once', '--log-tail', str(globals().get('LOG_TAIL', 4))],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if 'protocol=two_phase_happo_masac_v3' not in (result.stdout or ''):
        raise RuntimeError(
            'Monitor legacy (sin protocol=two_phase_happo_masac_v3). Ejecuta celda 1.2.'
        )
    for _bad in ('FASE 1: HAPPO + MATD3', 'En espera de inicio: delay=600'):
        if _bad in (result.stdout or ''):
            raise RuntimeError(f'Monitor layout 9+3 detectado: {_bad!r}')
    if result.returncode not in (0, 1):
        print(f'[7.3] Monitor salio con codigo {result.returncode}')


### 7.4 Auditoría y reorganización de artefactos

**7.4** verifica que cada uno de los 12 jobs tenga `results.json` (con KPIs OE), `timeseries.csv` y checkpoints. **7.4b** (opcional) exporta al layout canónico de tesis bajo `outputs/{MADRL}/{escenario}/`.


In [ ]:
# ── 7.4  Auditoría de artefactos — estructura y archivos por job ────────────  # verifica results.json con KPIs OE por job
# Verifica que cada uno de los 12 jobs (4 algos x 3 escenarios) tiene:
#   data/results.json · data/timeseries.csv · data/trace.csv
#   checkpoints/*.pt  · data/artifact_audit.json
# Muestra tamaño, episodios registrados y si el job esta completo.
import json
import os
from pathlib import Path

ALGOS = ['happo', 'masac', 'matd3', 'maac']
SCENS = ['E1', 'E2', 'E3']
REQUIRED = {
    'data/results.json'       : 'results',
    'data/timeseries.csv'     : 'timeseries',
    'data/trace.csv'          : 'trace',
    'data/checkpoint_manifest.json': 'ckpt_manifest',
    'data/artifact_audit.json': 'audit',
}
OPTIONAL = {
    'data/training_summary.json'       : 'train_summary',
    'data/building_behavior_summary.csv': 'bldg_summary',
    'data/building_kpis.csv'           : 'bldg_kpis',
}

_repo = globals().get('REPO', '/content/MADRLCitytleranflexresdr')

# Auto-descubrir OUTPUT_ROOT aunque se haya reiniciado el kernel.
if 'resolve_output_root_or_latest' in globals():
    _output_root = resolve_output_root_or_latest()
else:
    _output_root = globals().get('OUTPUT_ROOT', '')
    _ref = Path(_repo) / 'outputs' / 'latest_colab_output_root.txt'
    if not _output_root and _ref.exists():
        _output_root = _ref.read_text(encoding='utf-8').strip()

if not _output_root:
    raise RuntimeError(
        '[7.4] OUTPUT_ROOT no disponible. Ejecuta celda 2.1 o espera a que '
        'el launcher escriba outputs/latest_colab_output_root.txt.'
    )

out = Path(_output_root)
SEP  = '=' * 76
THIN = '-' * 76

def _sz(p):
    try:
        b = p.stat().st_size
        if b < 1024:
            return f'{b}B'
        if b < 1048576:
            return f'{b/1024:.1f}KB'
        return f'{b/1048576:.1f}MB'
    except Exception:
        return '?'

def _n_rows(p):
    try:
        with open(p, encoding='utf-8', errors='replace') as f:
            return sum(1 for _ in f) - 1  # minus header
    except Exception:
        return -1

def _ep_from_ts(data_dir):
    ts = data_dir / 'timeseries.csv'
    if not ts.exists():
        return None
    try:
        with open(ts, encoding='utf-8') as f:
            rows = sum(1 for _ in f) - 1
        return rows  # 1 row per episode
    except Exception:
        return None

# ── Estado global del manifest ───────────────────────────────────────────────
status_path = out / 'official_full_status.json'
all_jobs = []
global_status = '?'
if status_path.exists():
    st = json.loads(status_path.read_text())
    all_jobs = st.get('jobs', [])
    global_status = st.get('status', '?')

def _job_status(algo, scen):
    for j in all_jobs:
        if j.get('name','').lower() == algo and j.get('scenario','').upper() == scen:
            if j.get('skipped'):
                return 'SKIP'
            if j.get('exit_code') == 0:
                return 'OK  '
            if j.get('completed_at') is None:
                return 'RUN '
            return 'FAIL'
    return '----'

# ── Checkpoints ──────────────────────────────────────────────────────────────
def _ckpt_info(job_dir):
    ckpt_dir = job_dir / 'checkpoints'
    exts = {'.pt', '.pth', '.pkl', '.ckpt', '.zip'}
    files = [p for p in ckpt_dir.rglob('*') if p.is_file() and p.suffix.lower() in exts] if ckpt_dir.exists() else []
    if not files:
        # fallback: buscar en el dir raiz del job
        files = [p for p in job_dir.glob('*') if p.is_file() and p.suffix.lower() in exts]
    if not files:
        return 'sin checkpoints'
    total_mb = sum(p.stat().st_size for p in files) / 1048576
    return f'{len(files)} archivo(s)  {total_mb:.1f} MB'

# ── Imprimir cabecera ─────────────────────────────────────────────────────────
try:
    _hw = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip().splitlines()[0].strip()
except Exception:
    _hw = ''
print(SEP)
print(f"  AUDITORÍA DE ARTEFACTOS — MADRL CityLearn v3{' · ' + _hw if _hw else ''}")
print(f'  Run: {out.name}')
print(f'  Ruta: {_output_root}')
print(f'  Estado global: {global_status}')
print(SEP)

# ── Espacio en disco ─────────────────────────────────────────────────────────
try:
    import shutil
    usage = shutil.disk_usage(str(out) if out.exists() else '/content/drive/MyDrive')
    free_gib = usage.free / (1024**3)
    used_run = sum(f.stat().st_size for f in out.rglob('*') if f.is_file()) / (1024**3) if out.exists() else 0
    print(f'  Drive libre: {free_gib:.1f} GiB  |  Uso esta corrida: {used_run:.2f} GiB')
except Exception:
    pass
print()

# ── Por algoritmo y escenario ─────────────────────────────────────────────────
missing_critical = []
total_jobs = 0
ok_jobs = 0

for algo in ALGOS:
    print(f'  ┌── {algo.upper()} ─────────────────────────────────────────────────────')
    for scen in SCENS:
        total_jobs += 1
        job_dir  = out / algo.upper() / f'{scen}'
        data_dir = job_dir / 'data'
        jst = _job_status(algo, scen)

        print(f'  │  {algo.upper()}/{scen}  [{jst}]  {job_dir.relative_to(out) if job_dir.exists() else "(sin carpeta)"}')

        if not job_dir.exists():
            print('  │    ⚠ Carpeta no existe aún (job pendiente o no iniciado)')
            missing_critical.append(f'{algo.upper()}/{scen}: carpeta {job_dir.name} inexistente')
            print('  │')
            continue

        # Archivos requeridos
        all_ok = True
        for rel, label in REQUIRED.items():
            p = job_dir / rel
            if p.exists():
                sz = _sz(p)
                extra = ''
                if rel == 'data/timeseries.csv':
                    ep = _n_rows(p)
                    extra = f'  ({ep} episodios)' if ep >= 0 else ''
                elif rel == 'data/results.json':
                    try:
                        rd = json.loads(p.read_text())
                        algo_key = rd.get('algorithm', '')
                        kw_p = rd.get('kruskal_wallis', {}).get('p_value', None)
                        extra = f'  [algo={algo_key}]' + (f'  KW_p={kw_p:.4f}' if kw_p else '')
                    except Exception:
                        pass
                print(f'  │    ✓ {label:<16} {sz:>8}{extra}')
            else:
                print(f'  │    ✗ {label:<16} FALTA')
                all_ok = False
                if jst == 'OK  ':
                    missing_critical.append(f'{algo.upper()}/{scen}: {rel} falta (job=OK)')

        # Archivos opcionales
        for rel, label in OPTIONAL.items():
            p = job_dir / rel
            if p.exists():
                print(f'  │    · {label:<16} {_sz(p):>8}')

        # Checkpoints
        ckpt_info = _ckpt_info(job_dir)
        ckpt_icon = '✓' if 'archivo' in ckpt_info else '⚠'
        print(f'  │    {ckpt_icon} checkpoints     {ckpt_info}')

        # Figuras
        figs = list((job_dir / 'figures').glob('*.png')) if (job_dir / 'figures').exists() else []
        if figs:
            fig_mb = sum(f.stat().st_size for f in figs) / 1048576
            print(f'  │    · figuras          {len(figs)} PNG  {fig_mb:.1f} MB')

        # live_progress (si activo)
        lp = job_dir / 'live_progress.json'
        if lp.exists():
            try:
                lpd = json.loads(lp.read_text())
                ep  = lpd.get('episode', '?')
                st  = lpd.get('global_step', '?')
                fps = lpd.get('fps', '?')
                print(f'  │    ► live_progress    ep={ep}  step={st}  fps={fps}')
            except Exception:
                print('  │    ► live_progress    (ilegible)')

        if all_ok and jst == 'OK  ':
            ok_jobs += 1

        print('  │')
    print('  └' + '─' * 70)
    print()

# ── Resumen final ─────────────────────────────────────────────────────────────
print(SEP)
print(f'  RESUMEN: {ok_jobs}/{total_jobs} jobs con artefactos completos')
if missing_critical:
    print(f'  PROBLEMAS ({len(missing_critical)}):')
    for m in missing_critical:
        print(f'    ✗ {m}')
else:
    if ok_jobs == total_jobs:
        print('  ✓ Todos los jobs completados y artefactos verificados.')
    else:
        print(f'  ℹ {total_jobs - ok_jobs} jobs pendientes/en progreso.')
print(SEP)

# Mostrar estructura esperada
print()
print('  ESTRUCTURA ESPERADA POR JOB:')
print('  {OUTPUT_ROOT}/{MADRL}/{scenario}/')
print('    data/results.json         ← KPIs finales, ganancia vs baseline')
print('    data/timeseries.csv       ← retorno y métricas por episodio (N_EPISODES filas)')
print('    data/trace.csv            ← observ./acciones muestreadas (cada 24 pasos)')
print('    data/checkpoint_manifest.json')
print('    data/artifact_audit.json')
print('    checkpoints/              ← modelos .pt (actor/critic por agente)')
print('    figures/*.png             ← 13 gráficas de convergencia y KPIs')
print(SEP)


In [ ]:
# ── 7.4b  Reorganizar outputs al formato canónico: outputs/{MADRL}/{escenario}/ ──  # verifica results.json con KPIs OE por job
# El launcher escribe: {OUTPUT_ROOT}/HAPPO/E1/data/results.json
# Export opcional:     {OUTPUT_ROOT}/HAPPO/E1/metrics.csv  etc.
# Este paso genera la estructura canónica junto a los artefactos del launcher.
import csv
import json
import shutil
import os
import pandas as pd
from pathlib import Path
from datetime import datetime

_out  = Path(globals().get('OUTPUT_ROOT', '/tmp/madrl_output'))
_repo = Path(globals().get('REPO', '/content/MADRLCitytleranflexresdr'))
_hp   = globals().get('HYPERPARAMS', {})
_seed = globals().get('SEED', 0)

SCENARIO_MAP = {'E1': 'E1', 'E2': 'E2', 'E3': 'E3'}
ALGO_UPPER   = {'happo': 'HAPPO', 'masac': 'MASAC', 'matd3': 'MATD3', 'maac': 'MAAC'}

print('Reorganizando artefactos al formato outputs/{MADRL}/{escenario}/ ...')
_reorganized = []
_missing = []

for algo_lower, algo_upper in ALGO_UPPER.items():
    for sc_short, sc_long in SCENARIO_MAP.items():
        src_data = _out / algo_lower / f'{sc_short}' / 'data'
        dst_dir  = _out / algo_upper / sc_long
        dst_dir.mkdir(parents=True, exist_ok=True)
        (dst_dir / 'figures').mkdir(exist_ok=True)
        ok_files = []

        # 1. metrics.csv — desde results.json[citylearn_v3_report.all_values]
        results_json = src_data / 'results.json'
        if results_json.exists():
            with open(results_json, encoding='utf-8') as _f:
                _r = json.load(_f)
            _all_v = _r.get('citylearn_v3_report', {}).get('all_values', {})
            with open(dst_dir / 'metrics.csv', 'w', newline='', encoding='utf-8') as _cf:
                _w = csv.writer(_cf)
                _w.writerow(['metric', 'value'])
                for _k, _v in _all_v.items():
                    _w.writerow([_k, _v])
            ok_files.append('metrics.csv')
        else:
            _missing.append(f'{algo_upper}/{sc_long}: results.json no encontrado')

        # 2. rewards.csv — desde timeseries.csv, agregado por episodio
        ts_csv = src_data / 'timeseries.csv'
        if ts_csv.exists():
            try:
                _df_ts = pd.read_csv(ts_csv)
                if 'episode' in _df_ts.columns and 'reward_mean' in _df_ts.columns:
                    _ep_df = _df_ts.groupby('episode')['reward_mean'].agg(
                        ['mean', 'sum', 'min', 'max']).reset_index()
                    _ep_df.columns = ['episode', 'reward_mean', 'reward_sum',
                                      'reward_min', 'reward_max']
                    _ep_df.to_csv(dst_dir / 'rewards.csv', index=False)
                else:
                    _df_ts.to_csv(dst_dir / 'rewards.csv', index=False)
                ok_files.append('rewards.csv')
            except Exception as _e:
                print(f'  [WARN] rewards.csv {algo_upper}/{sc_long}: {_e}')
        else:
            _missing.append(f'{algo_upper}/{sc_long}: timeseries.csv no encontrado')

        # 3. training_monitor.csv — desde training_summary.json
        summary_json = src_data / 'training_summary.json'
        if summary_json.exists():
            with open(summary_json, encoding='utf-8') as _f:
                _s = json.load(_f)
            _ep_sum = _s.get('episode_summaries', [])
            if _ep_sum and isinstance(_ep_sum, list) and isinstance(_ep_sum[0], dict):
                pd.DataFrame(_ep_sum).to_csv(dst_dir / 'training_monitor.csv',
                                             index=False)
            else:
                with open(dst_dir / 'training_monitor.csv', 'w', newline='',
                          encoding='utf-8') as _cf:
                    _w = csv.writer(_cf)
                    _w.writerow(['metric', 'value'])
                    for _k, _v in _s.items():
                        if not isinstance(_v, (dict, list)):
                            _w.writerow([_k, _v])
            ok_files.append('training_monitor.csv')
        else:
            with open(dst_dir / 'training_monitor.csv', 'w', newline='',
                      encoding='utf-8') as _cf:
                _w = csv.writer(_cf)
                _w.writerow(['metric', 'value', 'note'])
                _w.writerow(['status', 'pendiente',
                             'training_summary.json no encontrado'])

        # 4. resource_usage.csv
        _ru_snap = _out / 'resource_usage_snapshot.json'
        _ru_csv  = dst_dir / 'resource_usage.csv'
        if _ru_snap.exists():
            with open(_ru_snap) as _f:
                _ru = json.load(_f)
            with open(_ru_csv, 'w', newline='', encoding='utf-8') as _cf:
                _w = csv.writer(_cf)
                _w.writerow(['metric', 'value'])
                for _k, _v in _ru.items():
                    _w.writerow([_k, _v])
        else:
            with open(_ru_csv, 'w', newline='', encoding='utf-8') as _cf:
                _w = csv.writer(_cf)
                _w.writerow(['metric', 'value'])
                _w.writerow(['generated_at', datetime.now().isoformat()])
                _w.writerow(['note',
                             'Snapshot no disponible; ejecuta celda 7.7 durante entrenamiento'])
        ok_files.append('resource_usage.csv')

        # 5. config.json
        _cfg = {
            'algorithm': algo_upper,
            'scenario': sc_long,
            'scenario_short': sc_short,
            'seed': _seed,
            'n_episodes': globals().get('N_EPISODES', 50),
            'episode_steps': globals().get('EPISODE_STEPS', 8760),
            'dataset': 'citylearn_iquitos_2023_2025',
            'hyperparams': _hp.get(algo_upper, {}),
            'generated_at': datetime.now().isoformat(),
        }
        with open(dst_dir / 'config.json', 'w', encoding='utf-8') as _f:
            json.dump(_cfg, _f, indent=2, ensure_ascii=False)
        ok_files.append('config.json')

        # 6. checkpoint.pt — tomar el checkpoint real mas reciente del arbol del launcher
        _src_algo_dir = _out / algo_lower / f'{sc_short}'
        _ckpt_cands = (list(_src_algo_dir.rglob('*.pt')) +
                       list(_src_algo_dir.rglob('*.pth')) +
                       list(_src_algo_dir.rglob('*.pkl')))
        if _ckpt_cands:
            _latest_ckpt = max(_ckpt_cands, key=lambda p: p.stat().st_mtime)
            shutil.copy2(_latest_ckpt, dst_dir / 'checkpoint.pt')
            ok_files.append('checkpoint.pt')
        else:
            _missing.append(f'{algo_upper}/{sc_long}: sin checkpoint .pt')

        # 7. Copiar figuras relevantes
        _src_figs = _out / 'figures'
        if _src_figs.exists():
            for _fig in list(_src_figs.glob(f'*{sc_short}*')) + list(_src_figs.glob(f'*{algo_lower}*')):
                shutil.copy2(_fig, dst_dir / 'figures' / _fig.name)

        _reorganized.append((algo_upper, sc_long, ok_files))

# 8. resumen_comparativo/ — estructura para comparacion global final
_resumen_dir = _out / 'resumen_comparativo'
_resumen_dir.mkdir(parents=True, exist_ok=True)

_cmp_path = _resumen_dir / 'comparison_metrics.csv'
if not _cmp_path.exists():
    with open(_cmp_path, 'w', newline='', encoding='utf-8') as _cf:
        _w = csv.writer(_cf)
        _w.writerow(['algorithm', 'scenario', 'metric', 'value'])
        _w.writerow(['PENDIENTE', '-', '-',
                     'Ejecutar celda 9.1 tras el entrenamiento para completar'])

_sel_path = _resumen_dir / 'best_madrl_selection.csv'
if not _sel_path.exists():
    with open(_sel_path, 'w', newline='', encoding='utf-8') as _cf:
        _w = csv.writer(_cf)
        _w.writerow(['rank', 'algorithm', 'mean_score', 'selected'])
        _w.writerow(['1', 'PENDIENTE', '-', 'Ejecutar celda 9.1 para ranking oficial'])

_rep_path = _resumen_dir / 'best_madrl_report.json'
if not _rep_path.exists():
    with open(_rep_path, 'w', encoding='utf-8') as _f:
        json.dump({
            'status': 'pendiente',
            'nota': 'Ejecutar celda 9.1 para seleccion estadistica oficial.',
            'referencia_v4': {
                'mejor_madrl': 'MATD3',
                'kw_p': 0.0459,
                'score': 0.7445,
            },
        }, _f, indent=2, ensure_ascii=False)

# Reporte final
print()
print(f'  {len(_reorganized)} carpetas reorganizadas:')
for _algo, _sc, _files in _reorganized:
    _n = len(_files)
    _mark = 'OK' if _n >= 4 else 'PARCIAL'
    print(f'    [{_mark}] {_algo}/{_sc}/  ({_n} archivos: {_files})')
if _missing:
    print()
    print('  Artefactos pendientes (se generan tras entrenamiento 50 ep):')
    for _m in _missing:
        print(f'    - {_m}')
print()
print(f'  resumen_comparativo/ preparado: {_resumen_dir}')
print()
print('  Estructura canonica validada:')
print(f'  {_out}/{{MADRL}}/{{escenario}}/')
print('  Completa con celda 9.1 para comparison_metrics.csv y best_madrl_report.json.')


### 7.5 Diagnostico de Drive — ejecutar en cualquier momento

Celda autocontenida (sirve tras reinicio de kernel). Verifica señales de problema y muestra instrucciones de relaunch. También se ejecuta dentro de **7.2** si el entrenamiento falla.


In [ ]:
# ── 7.5  Diagnostico de Drive + senales de problema + relaunch ──────────────
import json
import sys
import os
from pathlib import Path
from datetime import datetime, timezone

_repo = globals().get('REPO', '/content/MADRLCitytleranflexresdr')

# Auto-descubrir OUTPUT_ROOT aunque se haya reiniciado el kernel.
# Usa el helper de 7.0 si existe; si no, fallback autocontenido.
if 'resolve_output_root_or_latest' in globals():
    _out = resolve_output_root_or_latest()
else:
    _ref = Path(_repo) / 'outputs' / 'latest_colab_output_root.txt'
    _out = globals().get('OUTPUT_ROOT', '') or (
        _ref.read_text(encoding='utf-8').strip() if _ref.exists() else ''
    )
if not _out:
    print('[DIAG] ERROR: OUTPUT_ROOT desconocido.')
    print('  Opcion A: ejecuta celda 2.1 primero.')
    print('  Opcion B: define manualmente:')
    print('    _out = "/content/drive/MyDrive/MADRLCitytleranflexresdr/outputs/madrl_v3_YYYYMMDD_HHMMSS"')
    _out = None

if _out:
    print(f'OUTPUT_ROOT = {_out}')
    print()

    # ── SENAL 1: official_full_status.json existe? ───────────────────────────
    status_path = Path(_out) / 'official_full_status.json'
    if not status_path.exists():
        print('[SENAL 1] FAIL  official_full_status.json NO EXISTE')
        print('         El launcher no arranco o Drive no esta montado.')
        print(f'         Ruta esperada: {status_path}')
    else:
        with open(status_path) as _f:
            _s = json.load(_f)
        _jobs   = _s.get('jobs', [])
        _failed = [j for j in _jobs
                   if j.get('exit_code') not in (None, 0) and not j.get('skipped')]
        _active = [j for j in _jobs
                   if j.get('completed_at') is None
                   and not j.get('planned_only') and not j.get('skipped')]
        _done   = [j for j in _jobs
                   if j.get('exit_code') == 0 and not j.get('skipped')]

        _status_str = _s.get('status', '?')
        print(f'[SENAL 1] OK    status="{_status_str}"  '
              f'completados={len(_done)}/12  fallidos={len(_failed)}  '
              f'activos={len(_active)}')
        for _j in _jobs:
            if _j.get('planned_only'):
                continue
            if _j.get('skipped'):
                _st = 'SKIP'
            elif _j.get('exit_code') == 0:
                _st = 'OK  '
            elif _j.get('completed_at') is None:
                _st = 'RUN '
            else:
                _st = 'FAIL'
            _name = _j.get('name', '?').upper()
            _scen = _j.get('scenario', '?')
            _att  = _j.get('attempt', 0)
            print(f'  {_name:<6} {_scen:<3} -> {_st}  attempt={_att}')
        print()

        # ── SENAL 2: live_progress.json reciente? ────────────────────────────
        _pfiles = sorted(Path(_out).rglob('live_progress.json'))
        if _pfiles:
            _pf = _pfiles[-1]
            try:
                _prog = json.loads(_pf.read_text())
                _ts   = _prog.get('live_status_updated_at', '')
                _step = _prog.get('global_step', '?')
                _algo = _prog.get('algorithm', '?')
                _scen = _prog.get('scenario', '?')
                _ep   = _prog.get('episode', '?')
                if _ts:
                    _dt  = datetime.fromisoformat(_ts.replace('Z', '+00:00'))
                    _lag = (datetime.now(timezone.utc) - _dt).total_seconds()
                    if _lag < 120:
                        _sig = 'OK  ACTIVO'
                    else:
                        _sig = f'WARN COLGADO ({_lag:.0f}s sin actualizar)'
                else:
                    _lag, _sig = 0, '? sin timestamp'
                print(f'[SENAL 2] {_sig} -- {_algo}/{_scen} ep={_ep} step={_step}')
                print(f'          {_pf.relative_to(Path(_out))}')
            except Exception as _e:
                print(f'[SENAL 2] WARN live_progress.json ilegible: {_e}')
        else:
            print('[SENAL 2] INFO  Sin live_progress.json aun '
                  '(primer job en inicializacion o training no ha arrancado)')
        print()

        # ── SENAL 3: stderr con errores? ─────────────────────────────────────
        _errs = sorted(Path(_out).glob('logs/*.stderr.log'))
        _bad  = [(p, p.read_text(errors='replace'))
                 for p in _errs if p.stat().st_size > 0]
        if _bad:
            print(f'[SENAL 3] FAIL  {len(_bad)} archivo(s) stderr con contenido:')
            for _p, _txt in _bad:
                print(f'  === {_p.name} ===')
                _lines = _txt.strip().splitlines()
                print('  ' + '\n  '.join(_lines[-25:]))
        else:
            print(f'[SENAL 3] OK    Sin errores stderr '
                  f'({len(_errs)} logs revisados)')
        print()

        # ── SENAL 4: artefactos generados ────────────────────────────────────
        _nres  = len(list(Path(_out).rglob('results.json')))
        _nckpt = len(list(Path(_out).rglob('*.pt')))
        _ncsv  = len(list(Path(_out).rglob('*.csv')))
        _nlogs = len(list(Path(_out).glob('logs/*.log')))
        print(f'[SENAL 4] Artefactos en Drive: '
              f'results.json={_nres}/12  checkpoints={_nckpt}  '
              f'CSVs={_ncsv}  logs={_nlogs}')
        print()

        # ── Instrucciones de relaunch si hay problemas ────────────────────────
        _needs_relaunch = _failed or _s.get('status') in ('failed', 'running')
        if _needs_relaunch:
            print('=' * 72)
            print('RELAUNCH RECOMENDADO')
            print('  --skip-completed reanuda automaticamente los jobs ya completados.')
            print('  Pasos:')
            print('    1. En celda 2.1 establece RESUME_OUTPUT_ROOT:')
            print(f'       RESUME_OUTPUT_ROOT = "{_out}"')
            print('    2. Ejecuta las celdas en orden: 1.x setup -> 2.1 -> 6.1 -> 7.2')
        elif _s.get('status') == 'completed':
            print('El entrenamiento esta COMPLETO. Procede con la Seccion 8 (analisis).')
        else:
            print('Entrenamiento en curso. Vuelve a ejecutar esta celda para actualizar.')


### 7.6 Benchmarks comparativos CityLearn v2 (opcionales)

#### Baseline principal: `evaluate_v2` embebido en corridas MADRL

Cada job MADRL calcula la línea base RBC/local vía **`evaluate_v2`** y guarda en `results.json` los campos `objective_kpis`, `axis_baseline_comparison` y `baseline_gain_by_kpi`. **Esa es la baseline oficial para ranking 8.x/9.x.**

#### PPO/SAC/A2C (SB3) — off por defecto

Benchmarks **opcionales** de agente central CityLearn v2 (`central_agent=True`) con Stable-Baselines3. Default: `RUN_CITYLEARN_V2_SB3_BENCHMARKS = False`. No sustituyen a `evaluate_v2` ni a los 12 jobs MADRL.

Schema: `CityLearn/data/datasets/citylearn_iquitos_2023_2025/schema.json`

| Agente | Script | Salida |
|---|---|---|
| PPO | `benchmark_citylearn_v2_ppo.py` | `outputs/citylearn_v2_original_benchmark/ppo/...` |
| SAC | `benchmark_citylearn_v2_sac.py` | `outputs/citylearn_v2_original_benchmark/sac/...` |
| A2C | `benchmark_citylearn_v2_a2c.py` | `outputs/citylearn_v2_original_benchmark/a2c/...` |

> HAPPO/MASAC/MATD3/MAAC = 4 MADRL principales. PPO/SAC/A2C = benchmarks v2 opcionales; **MAPPO/MADDPG no son baseline**.


In [ ]:
#  7.6  Benchmarks CityLearn v2 PPO/SAC/A2C (SB3) — OPCIONAL / OFF por defecto
#
# NO es el baseline del ranking OG/OE: ese viene de evaluate_v2 embebido en cada job MADRL.
# Scripts: CityLearn v2 central_agent=True + StableBaselines3Wrapper (mismo schema Iquitos).


CITYLEARN_V2_BENCHMARKS = ["PPO", "SAC", "A2C"]
RUN_CITYLEARN_V2_SB3_BENCHMARKS = False  # dejar False salvo comparacion v2 central-agent
SB3_BASELINE_SCENARIO = 'ALL'
SB3_BASELINE_TRAIN_EPISODES = 50
SB3_BASELINE_OUTPUT = str(Path(REPO) / 'outputs/citylearn_v2_original_benchmark')

if RUN_CITYLEARN_V2_SB3_BENCHMARKS:
    import subprocess
    sb3_scripts = {
        'ppo': 'CityLearn/scripts/benchmark_citylearn_v2_ppo.py',
        'sac': 'CityLearn/scripts/benchmark_citylearn_v2_sac.py',
        'a2c': 'CityLearn/scripts/benchmark_citylearn_v2_a2c.py',
    }
    for agent_name, script_rel in sb3_scripts.items():
        cmd = [
            PROJECT_PYTHON, '-B', str(Path(REPO) / script_rel),
            '--schema-path', SCHEMA_PATH,
            '--scenario', SB3_BASELINE_SCENARIO,
            '--seed', str(SEED),
            '--episode-time-steps', str(EPISODE_STEPS),
            '--train-episodes', str(SB3_BASELINE_TRAIN_EPISODES),
            '--output-dir', SB3_BASELINE_OUTPUT,
        ]
        print(f'[7.6] CityLearn v2 SB3 benchmark {agent_name.upper()}:')
        print(' '.join(map(str, cmd)))
        subprocess.check_call(cmd, cwd=REPO)
else:
    print('[7.6] PPO/SAC/A2C CityLearn v2 SB3 benchmarks desactivados por defecto.')
    print('      Activar RUN_CITYLEARN_V2_SB3_BENCHMARKS=True para generar artefactos comparables.')



### 7.7 Monitor de recursos del sistema — RAM / VRAM / GPU / CPU

Instantánea de uso de recursos durante o después del entrenamiento. Escribe `resource_usage_snapshot.csv` en `OUTPUT_ROOT`.


In [ ]:
# ── 7.7  Monitor de recursos: RAM / VRAM / CPU / GPU ────────────────────────
import subprocess
import json
import os
import csv
import time
from pathlib import Path
from datetime import datetime

# psutil para RAM y CPU
try:
    import psutil
    ram = psutil.virtual_memory()
    cpu_pct = psutil.cpu_percent(interval=1)
    ram_used_gib  = ram.used / 1024**3
    ram_total_gib = ram.total / 1024**3
    ram_pct = ram.percent
except ImportError:
    ram_used_gib = ram_total_gib = ram_pct = cpu_pct = None
    print("[INFO] psutil no disponible. Instala con: pip install psutil")

# GPU via nvidia-smi
gpu_info = {}
try:
    res = subprocess.run(
        ["nvidia-smi",
         "--query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, timeout=10,
    )
    if res.returncode == 0:
        parts = [p.strip() for p in res.stdout.strip().split(",")]
        if len(parts) >= 6:
            gpu_info = {
                "gpu_index"      : parts[0],
                "gpu_name"       : parts[1],
                "gpu_util_pct"   : float(parts[2]),
                "vram_used_mib"  : float(parts[3]),
                "vram_total_mib" : float(parts[4]),
                "gpu_temp_c"     : float(parts[5]),
                "vram_used_gib"  : float(parts[3]) / 1024,
                "vram_total_gib" : float(parts[4]) / 1024,
                "vram_used_pct"  : 100.0 * float(parts[3]) / max(float(parts[4]), 1),
            }
except Exception as e:
    print(f"[WARN] nvidia-smi error: {e}")

snap = {
    "timestamp"       : datetime.now().isoformat(),
    "ram_used_gib"    : round(ram_used_gib or 0, 2),
    "ram_total_gib"   : round(ram_total_gib or 0, 2),
    "ram_used_pct"    : round(ram_pct or 0, 1),
    "cpu_used_pct"    : round(cpu_pct or 0, 1),
    **{k: round(v, 2) if isinstance(v, float) else v for k, v in gpu_info.items()},
}

# Guardar snapshot CSV
out_dir = Path(globals().get("OUTPUT_ROOT", "/tmp"))
snap_path = out_dir / "resource_usage_snapshot.csv"
file_exists = snap_path.exists()
with open(snap_path, "a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(snap.keys()))
    if not file_exists:
        w.writeheader()
    w.writerow(snap)

print(f"{'=' * 55}")
print(f"  SNAPSHOT DE RECURSOS — {snap['timestamp'][:19]}")
print(f"{'=' * 55}")
print(f"  RAM         : {snap['ram_used_gib']:.1f} / {snap['ram_total_gib']:.1f} GiB  ({snap['ram_used_pct']:.0f}%)")
print(f"  CPU         : {snap['cpu_used_pct']:.0f}% utilizado")
if gpu_info:
    print(f"  GPU         : {gpu_info.get('gpu_name', '?')}")
    print(f"  GPU util    : {gpu_info.get('gpu_util_pct', 0):.0f}%")
    print(f"  VRAM usada  : {gpu_info.get('vram_used_gib', 0):.1f} / {gpu_info.get('vram_total_gib', 0):.1f} GiB  ({gpu_info.get('vram_used_pct', 0):.0f}%)")
    print(f"  Temp GPU    : {gpu_info.get('gpu_temp_c', 0):.0f} C")
print(f"  Guardado en : {snap_path}")


## Sección 8: Análisis de resultados y KPIs CityLearn v3

Fuente de verdad: `citylearn.v3.objectives` — **conjuntos OE completos**, no un subconjunto reducido de 3 KPIs.

| Escenario | Objetivo | KPIs (`objectives.py`) | Conteo |
|-----------|----------|------------------------|:------:|
| **E1** | **OE.1** Flexibilidad | `OE1_FLEXIBILITY_KPIS` | **36** |
| **E2** | **OE.2** Emisiones CO2 | `OE2_EMISSIONS_KPIS` | **7** |
| **E3** | **OE.3** Costos | `OE3_COST_KPIS` | **11** |
| | | **Total** | **54** |

**Niveles:** durante el entrenamiento (`write_training_artifacts`), cada job **MADRL × escenario** persiste KPIs OE a nivel **distrito** y **edificio** (17 buildings).

### Artefactos (formato canónico; escritos al completar cada corrida)
```
{OUTPUT_ROOT}/
  HAPPO|MASAC|MATD3|MAAC/
    E1|E2|E3/
      data/results.json                    # objective_axis_kpis (distrito) + building_objective_kpis
      figures/tables/objective_kpis.csv    # OE distrito (54 nombres lógicos)
      data/district_kpis.csv               # evaluate_v2 nivel district
      data/building_kpis.csv               # evaluate_v2 crudo por edificio
      data/building_objective_kpis.csv     # OE lógicos por edificio (OE1/OE2/OE3)
      data/citylearn_kpi_frame.csv         # frame evaluate_v2 completo
  evaluation/                              # 8.x
  resumen_comparativo/                     # 9.x
```

> Carga canónica distrito: `objective_kpis.csv` → fallback `results.json`.
> Carga canónica edificio: `building_objective_kpis.csv` → fallback `building_kpis.csv`.
> Baseline: `evaluate_v2` embebido (no hace falta 7.6 SB3).


In [ ]:
# ── 8.1  Cargar KPIs CityLearn v3 (OE1/OE2/OE3 completos = 54) ───────────────
import json
import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd

# OE1=36→E1 · OE2=7→E2 · OE3=11→E3 (objectives.py). Baseline=evaluate_v2. Fallback si falla import.
try:
    from citylearn.v3.objectives import (
        OBJECTIVE_AXES,
        OE1_FLEXIBILITY_KPIS,
        OE2_EMISSIONS_KPIS,
        OE3_COST_KPIS,
        HIGHER_IS_BETTER,
        RATIO_TO_BASELINE_KPIS,
    )
except ImportError:
    OE1_FLEXIBILITY_KPIS = (
        "grid_import", "grid_import_control", "grid_import_baseline", "grid_import_delta",
        "zero_net_energy", "net_exchange_control", "net_exchange_baseline", "net_exchange_delta",
        "grid_export_ratio", "grid_export_control", "grid_export_baseline", "grid_export_delta",
        "peak_average", "ramping_average", "one_minus_load_factor_average",
        "pv_generation_total", "pv_generation_daily_average", "pv_export_total", "pv_export_daily_average",
        "pv_self_consumption_ratio",
        "community_local_traded_total", "community_local_traded_daily_average", "community_import_share",
        "battery_charge_total", "battery_discharge_total", "battery_throughput_total",
        "battery_equivalent_full_cycles", "battery_capacity_fade_ratio",
        "ev_departure_count", "ev_departure_met_count", "ev_departure_within_tolerance_count",
        "ev_departure_success_rate", "ev_departure_within_tolerance_rate", "ev_departure_soc_deficit_mean",
        "ev_charge_total", "ev_v2g_export_total",
    )
    OE2_EMISSIONS_KPIS = (
        "carbon_emissions", "carbon_emissions_control", "carbon_emissions_baseline", "carbon_emissions_delta",
        "carbon_emissions_daily_average_control", "carbon_emissions_daily_average_baseline",
        "carbon_emissions_daily_average_delta",
    )
    OE3_COST_KPIS = (
        "electricity_cost", "electricity_cost_control", "electricity_cost_baseline", "electricity_cost_delta",
        "electricity_cost_daily_average_control", "electricity_cost_daily_average_baseline",
        "electricity_cost_daily_average_delta",
        "cost_peak_average", "cost_ramping_average", "cost_one_minus_load_factor_average",
        "price_signal_deviation",
    )
    HIGHER_IS_BETTER = {
        "pv_generation_total", "pv_generation_daily_average", "pv_self_consumption_ratio",
        "community_local_traded_total", "community_local_traded_daily_average", "community_import_share",
        "battery_charge_total", "battery_discharge_total", "battery_throughput_total",
        "battery_equivalent_full_cycles",
        "ev_departure_met_count", "ev_departure_within_tolerance_count",
        "ev_departure_success_rate", "ev_departure_within_tolerance_rate",
        "ev_charge_total", "ev_v2g_export_total",
    }
    RATIO_TO_BASELINE_KPIS = {
        "grid_import", "zero_net_energy", "grid_export_ratio",
        "peak_average", "ramping_average", "one_minus_load_factor_average",
        "carbon_emissions", "electricity_cost",
        "cost_peak_average", "cost_ramping_average", "cost_one_minus_load_factor_average",
    }
    OBJECTIVE_AXES = {
        "OE1": type("A", (), {"code": "OE1", "scenario": "E1", "name": "Flexibilidad energetica", "kpis": OE1_FLEXIBILITY_KPIS})(),
        "OE2": type("A", (), {"code": "OE2", "scenario": "E2", "name": "Emisiones de CO2", "kpis": OE2_EMISSIONS_KPIS})(),
        "OE3": type("A", (), {"code": "OE3", "scenario": "E3", "name": "Costos energeticos", "kpis": OE3_COST_KPIS})(),
    }

SCENARIO_TO_OE = {"E1": "OE1", "E2": "OE2", "E3": "OE3"}
OE_TO_SCENARIO = {"OE1": "E1", "OE2": "E2", "OE3": "E3"}
OE_KPI_SETS = {
    "OE1": tuple(OE1_FLEXIBILITY_KPIS),
    "OE2": tuple(OE2_EMISSIONS_KPIS),
    "OE3": tuple(OE3_COST_KPIS),
}
ALL_OE_KPIS = tuple(dict.fromkeys(
    list(OE1_FLEXIBILITY_KPIS) + list(OE2_EMISSIONS_KPIS) + list(OE3_COST_KPIS)
))


def _safe_float(v):
    try:
        x = float(v)
    except (TypeError, ValueError):
        return np.nan
    return x if np.isfinite(x) else np.nan


def _parse_algo_scenario(fp: Path, output_root: Path):
    parts = fp.parts
    root_name = output_root.name
    try:
        root_idx = next(i for i, p in enumerate(parts) if p == root_name)
    except StopIteration:
        root_idx = max(0, len(parts) - 4)
    algo = parts[root_idx + 1].upper() if root_idx + 1 < len(parts) else "?"
    sc_seed = parts[root_idx + 2] if root_idx + 2 < len(parts) else "?"
    if "_seed_" in sc_seed:
        scenario, seed_s = sc_seed.split("_seed_", 1)
        try:
            seed = int(seed_s)
        except ValueError:
            seed = int(globals().get("SEED", 0))
    else:
        scenario = sc_seed
        seed = int(globals().get("SEED", 0))
    return algo, scenario, seed


def _rows_from_objective_csv(csv_path: Path, algo: str, scenario: str, seed: int = 0):
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        return []
    if df.empty or "kpi" not in df.columns:
        return []
    rows = []
    for _, r in df.iterrows():
        kpi = str(r.get("kpi", "")).strip()
        if not kpi:
            continue
        axis = str(r.get("axis", "") or "").strip().upper()
        if not axis:
            for oe, kpis in OE_KPI_SETS.items():
                if kpi in kpis:
                    axis = oe
                    break
        rows.append({
            "algorithm": algo,
            "scenario": scenario,
            "seed": int(seed),
            "axis": axis or SCENARIO_TO_OE.get(scenario, ""),
            "kpi": kpi,
            "value": _safe_float(r.get("value")),
            "baseline": _safe_float(r.get("baseline")),
            "delta_vs_baseline": _safe_float(r.get("delta_vs_baseline")),
            "improved_vs_baseline": r.get("improved_vs_baseline"),
            "lower_is_better": (
                bool(r.get("lower_is_better"))
                if pd.notna(r.get("lower_is_better"))
                else (kpi not in HIGHER_IS_BETTER)
            ),
            "available": r.get("available"),
            "source": "objective_kpis.csv",
            "run_path": str(csv_path.parent.parent.parent),
        })
    return rows


def _rows_from_results_json(rj_path: Path, algo: str, scenario: str, seed: int = 0):
    try:
        with open(rj_path, encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        return []
    report = data.get("citylearn_v3_report") or {}
    axes = report.get("objective_axis_kpis") or report.get("axes") or {}
    rows = []
    if isinstance(axes, dict) and axes:
        for axis_code, axis in axes.items():
            if not isinstance(axis, dict):
                continue
            kpis = axis.get("kpis") or {}
            if not isinstance(kpis, dict):
                continue
            for kpi, payload in kpis.items():
                payload = payload if isinstance(payload, dict) else {}
                trace = payload.get("trace") if isinstance(payload.get("trace"), dict) else {}
                comparison = payload.get("comparison") if isinstance(payload.get("comparison"), dict) else {}
                rows.append({
                    "algorithm": algo,
                    "scenario": scenario,
                    "seed": int(seed),
                    "axis": str(axis_code).upper(),
                    "kpi": kpi,
                    "value": _safe_float(payload.get("value")),
                    "baseline": _safe_float(comparison.get("baseline")),
                    "delta_vs_baseline": _safe_float(comparison.get("delta_vs_baseline")),
                    "improved_vs_baseline": comparison.get("improved_vs_baseline"),
                    "lower_is_better": bool(trace.get("lower_is_better", kpi not in HIGHER_IS_BETTER)),
                    "available": comparison.get("available"),
                    "source": "results.json:objective_axis_kpis",
                    "run_path": str(rj_path.parent.parent),
                })
        return rows

    # Fallback mínimo: all_values con lista OE completa
    all_v = report.get("all_values") or {}
    if not isinstance(all_v, dict):
        return []
    for kpi in ALL_OE_KPIS:
        if kpi not in all_v:
            continue
        axis = next((oe for oe, ks in OE_KPI_SETS.items() if kpi in ks), "")
        val = _safe_float(all_v.get(kpi))
        baseline = 1.0 if kpi in RATIO_TO_BASELINE_KPIS else np.nan
        rows.append({
            "algorithm": algo,
            "scenario": scenario,
            "seed": int(seed),
            "axis": axis,
            "kpi": kpi,
            "value": val,
            "baseline": baseline,
            "delta_vs_baseline": (val - baseline) if np.isfinite(val) and np.isfinite(baseline) else np.nan,
            "improved_vs_baseline": None,
            "lower_is_better": kpi not in HIGHER_IS_BETTER,
            "available": bool(np.isfinite(val)),
            "source": "results.json:all_values",
            "run_path": str(rj_path.parent.parent),
        })
    return rows


def load_objective_kpis(output_root: str) -> pd.DataFrame:
    """Carga KPIs OE1/OE2/OE3 (forma larga) por algoritmo, escenario y semilla (distrito)."""
    root = Path(output_root)
    records = []
    seen_runs = set()

    for csv_path in sorted(root.glob("*/*/figures/tables/objective_kpis.csv")):
        algo, scenario, seed = _parse_algo_scenario(csv_path, root)
        if scenario not in SCENARIO_TO_OE:
            continue
        rows = _rows_from_objective_csv(csv_path, algo, scenario, seed=seed)
        if rows:
            records.extend(rows)
            seen_runs.add((algo, scenario, seed))

    for rj_path in sorted(root.glob("*/*/data/results.json")):
        algo, scenario, seed = _parse_algo_scenario(rj_path, root)
        if scenario not in SCENARIO_TO_OE:
            continue
        if (algo, scenario, seed) in seen_runs:
            continue
        rows = _rows_from_results_json(rj_path, algo, scenario, seed=seed)
        if rows:
            records.extend(rows)
            seen_runs.add((algo, scenario, seed))

    return pd.DataFrame(records)


def load_building_objective_kpis(output_root: str) -> pd.DataFrame:
    """Carga KPIs OE por edificio (escritos durante el entrenamiento)."""
    root = Path(output_root)
    records = []
    seen = set()
    patterns = (
        "*/*/data/building_objective_kpis.csv",
        "*/*/figures/tables/building_objective_kpis.csv",
        "*/*/building_objective_kpis.csv",
    )
    for pattern in patterns:
        for csv_path in sorted(root.glob(pattern)):
            algo, scenario, seed = _parse_algo_scenario(csv_path, root)
            if scenario not in SCENARIO_TO_OE:
                continue
            key = (algo, scenario, seed, str(csv_path.resolve()))
            if key in seen:
                continue
            seen.add(key)
            try:
                df = pd.read_csv(csv_path)
            except Exception:
                continue
            if df.empty:
                continue
            df = df.copy()
            df["algorithm"] = algo
            df["scenario"] = scenario
            df["seed"] = int(seed)
            df["level"] = df.get("level", "building")
            df["source"] = str(csv_path.relative_to(root)) if csv_path.is_relative_to(root) else str(csv_path)
            records.append(df)
    if not records:
        # Fallback: raw evaluate_v2 building_kpis (cost_function nativo)
        for csv_path in sorted(root.glob("*/*/data/building_kpis.csv")):
            algo, scenario, seed = _parse_algo_scenario(csv_path, root)
            if scenario not in SCENARIO_TO_OE:
                continue
            try:
                df = pd.read_csv(csv_path)
            except Exception:
                continue
            if df.empty:
                continue
            df = df.copy()
            df["algorithm"] = algo
            df["scenario"] = scenario
            df["seed"] = int(seed)
            df["building"] = df.get("name", df.get("building"))
            df["kpi"] = df.get("cost_function", df.get("kpi"))
            df["source"] = f"{csv_path.relative_to(root)} (raw evaluate_v2)"
            records.append(df)
    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()


def objective_long_to_wide(df_long: pd.DataFrame) -> pd.DataFrame:
    """Wide: una fila por (algorithm, scenario[, seed]) con columnas = KPI values (+ baseline/delta)."""
    if df_long.empty:
        return pd.DataFrame()
    keys = ["algorithm", "scenario"] + (["seed"] if "seed" in df_long.columns else [])
    base = (
        df_long.groupby(keys, as_index=False)
        .agg(n_kpis=("kpi", "nunique"), source=("source", "first"))
    )
    piv = df_long.pivot_table(
        index=keys, columns="kpi", values="value", aggfunc="first"
    )
    piv_b = df_long.pivot_table(
        index=keys, columns="kpi", values="baseline", aggfunc="first"
    )
    piv_d = df_long.pivot_table(
        index=keys, columns="kpi", values="delta_vs_baseline", aggfunc="first"
    )
    piv_b = piv_b.add_suffix("__baseline")
    piv_d = piv_d.add_suffix("__delta")
    wide = base.set_index(keys).join(piv).join(piv_b).join(piv_d).reset_index()
    return wide


def scenario_axis_table(df_long: pd.DataFrame, scenario: str) -> pd.DataFrame:
    """Tabla de cumplimiento: todos los KPIs del OE emparejado al escenario."""
    oe = SCENARIO_TO_OE[scenario]
    kpis = list(OE_KPI_SETS[oe])
    sub = df_long[(df_long["scenario"] == scenario) & (df_long["kpi"].isin(kpis))].copy()
    if sub.empty:
        return sub
    sub["oe"] = oe
    sub["objetivo"] = {
        "OE1": "OE.1 Flexibilidad",
        "OE2": "OE.2 Emisiones CO2",
        "OE3": "OE.3 Costos energeticos",
    }[oe]
    cols = [
        "objetivo", "oe", "scenario", "algorithm", "kpi", "value", "baseline",
        "delta_vs_baseline", "improved_vs_baseline", "lower_is_better", "available", "source",
    ]
    return sub[cols].sort_values(["algorithm", "kpi"])


df_objective_long = load_objective_kpis(str(OUTPUT_ROOT))
df_results = objective_long_to_wide(df_objective_long)

os.makedirs(f"{OUTPUT_ROOT}/evaluation", exist_ok=True)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)

print("KPIs CityLearn v3 por eje:")
for oe, kpis in OE_KPI_SETS.items():
    print(f"  {oe} ({OE_TO_SCENARIO[oe]}): {len(kpis)} KPIs")

if df_objective_long.empty:
    print("⚠️  Sin objective_kpis / results.json — ejecuta el entrenamiento (7.2) primero.")
    print("   (Referencia v4: MATD3 KW p=0.0459, Score global 0.7445)")
else:
    n_keys = ["algorithm", "scenario"] + (["seed"] if "seed" in df_objective_long.columns else [])
    n_runs = df_objective_long.groupby(n_keys).ngroups
    n_seeds = (
        int(df_objective_long["seed"].nunique())
        if "seed" in df_objective_long.columns
        else 1
    )
    expected_seeds = int(globals().get("N_SEEDS", 12))
    print(
        f"\n✅  {n_runs} corridas | {n_seeds}/{expected_seeds} semillas | "
        f"{len(df_objective_long)} filas KPI (forma larga)\n"
    )
    df_objective_long.to_csv(f"{OUTPUT_ROOT}/evaluation/all_objective_kpis_long.csv", index=False)
    df_results.to_csv(f"{OUTPUT_ROOT}/evaluation/all_kpis.csv", index=False)

    # Tablas por escenario = OE emparejado (cumplimiento OE.1 / OE.2 / OE.3)
    scenario_tables = {}
    for sc in ("E1", "E2", "E3"):
        tbl = scenario_axis_table(df_objective_long, sc)
        scenario_tables[sc] = tbl
        oe = SCENARIO_TO_OE[sc]
        out_csv = f"{OUTPUT_ROOT}/evaluation/{sc}_{oe}_kpis.csv"
        if not tbl.empty:
            tbl.to_csv(out_csv, index=False)
            print(f"── {sc} → {oe} ({len(OE_KPI_SETS[oe])} KPIs definidos) ──")
            # Vista pivote valor por algoritmo
            piv = tbl.pivot_table(index="kpi", columns="algorithm", values="value", aggfunc="first")
            # Asegurar filas en orden oficial aunque falten valores
            piv = piv.reindex(list(OE_KPI_SETS[oe]))
            print(piv.to_string())
            print(f"  → {out_csv}\n")
        else:
            print(f"── {sc} → {oe}: sin datos\n")

    # Resumen cobertura: cuántos KPIs oficiales aparecen por corrida/eje
    cov_rows = []
    for (algo, sc), g in df_objective_long.groupby(["algorithm", "scenario"]):
        oe = SCENARIO_TO_OE.get(sc)
        if not oe:
            continue
        expected = set(OE_KPI_SETS[oe])
        present = set(g.loc[g["axis"] == oe, "kpi"]) | set(g.loc[g["kpi"].isin(expected), "kpi"])
        avail = g[g["kpi"].isin(expected) & np.isfinite(g["value"].astype(float))]
        cov_rows.append({
            "algorithm": algo,
            "scenario": sc,
            "oe": oe,
            "kpis_definidos": len(expected),
            "kpis_presentes": len(present & expected),
            "kpis_con_valor": int(avail["kpi"].nunique()),
        })
    df_coverage = pd.DataFrame(cov_rows)
    if not df_coverage.empty:
        print("Cobertura de KPIs oficiales (escenario → OE emparejado):")
        print(df_coverage.to_string(index=False))
        df_coverage.to_csv(f"{OUTPUT_ROOT}/evaluation/oe_kpi_coverage.csv", index=False)

# ── 8.1a  KPIs por edificio (artefactos del entrenamiento) ───────────────────
df_building_long = load_building_objective_kpis(str(OUTPUT_ROOT))
if df_building_long.empty:
    print("⚠️  Sin building_objective_kpis / building_kpis — re-entrena o verifica write_training_artifacts.")
else:
    n_bldg = df_building_long["building"].nunique() if "building" in df_building_long.columns else (
        df_building_long["name"].nunique() if "name" in df_building_long.columns else "?"
    )
    n_runs_b = df_building_long.groupby(["algorithm", "scenario"]).ngroups
    print(f"\n✅  Edificio: {n_runs_b} corridas | {n_bldg} edificios | {len(df_building_long)} filas KPI")
    out_b = f"{OUTPUT_ROOT}/evaluation/all_building_objective_kpis_long.csv"
    df_building_long.to_csv(out_b, index=False)
    print(f"  → {out_b}")

# ── 8.1b  Exportar artefactos auxiliares de tesis ────────────────────────────
_exported = 0
for ts_path in sorted(glob.glob(f"{OUTPUT_ROOT}/*/*/data/timeseries.csv")):
    run_dir = Path(ts_path).parent.parent
    summary_path = run_dir / "data" / "training_summary.json"
    try:
        ts_df = pd.read_csv(ts_path)
    except Exception:
        continue

    reward_cols = {
        c: c for c in ts_df.columns
        if any(k in c.lower() for k in ["reward", "episode", "peak", "carbon", "cost", "step"])
    }
    if reward_cols:
        ts_df[list(reward_cols.values())].to_csv(run_dir / "data" / "rewards.csv", index=False)

    ts_df = ts_df.copy()
    ts_df["monitor_ts"] = pd.date_range(start="2026-01-01", periods=len(ts_df), freq="min")
    ts_df.to_csv(run_dir / "data" / "training_monitor.csv", index=False)

    if summary_path.exists():
        with open(summary_path, encoding="utf-8") as f:
            summary = json.load(f)
        config_out = {
            "algorithm": summary.get("algorithm"),
            "scenario": summary.get("scenario"),
            "seed": summary.get("seed"),
            "episodes": summary.get("episodes"),
            "episode_steps": summary.get("episode_time_steps"),
            "num_env_steps": summary.get("num_env_steps"),
            "hyperparameters": summary.get("hyperparameters", {}),
            "backend": summary.get("backend"),
            "output_dir": summary.get("output_dir"),
            "gpu_runtime": summary.get("gpu_runtime", {}),
        }
        with open(run_dir / "data" / "config.json", "w", encoding="utf-8") as f:
            json.dump(config_out, f, indent=2, default=str)

    live_path = run_dir / "live_progress.json"
    if live_path.exists():
        try:
            with open(live_path, encoding="utf-8") as f:
                lp = json.load(f)
            pd.DataFrame([{
                "episode": lp.get("episode"),
                "global_step": lp.get("global_step"),
                "ram_used_gib": lp.get("ram_used_gib"),
                "vram_used_gib": lp.get("vram_used_gib"),
                "gpu_util_pct": lp.get("gpu_util_pct"),
                "live_status": lp.get("live_status"),
            }]).to_csv(run_dir / "data" / "resource_usage.csv", index=False)
        except Exception:
            pass
    _exported += 1

print(f"\nArtefactos auxiliares exportados: {_exported} corridas")
print("  evaluation/all_objective_kpis_long.csv — KPIs OE completos (largo)")
print("  evaluation/E{1,2,3}_OE{1,2,3}_kpis.csv — KPIs del OE emparejado")
print("  evaluation/all_kpis.csv — wide (valores + baseline/delta)")


In [ ]:
# ── 8.2  Curvas de convergencia (timeseries.csv, por episodio) ───────────────
import matplotlib.pyplot as plt
import glob
import pandas as pd
from pathlib import Path

ts_data = {}
for fp in sorted(glob.glob(f"{OUTPUT_ROOT}/*/*/data/timeseries.csv")):
    parts = Path(fp).parts
    root_idx = next(i for i,p in enumerate(parts) if p == Path(OUTPUT_ROOT).name)
    algo     = parts[root_idx + 1].upper()
    sc_seed  = parts[root_idx + 2]
    sc       = sc_seed.split("_seed_")[0] if "_seed_" in sc_seed else sc_seed
    try:
        ts_data[f"{algo}_{sc}"] = pd.read_csv(fp)
    except Exception:
        pass

if ts_data:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    CLR = {"HAPPO":"#3b82f6","MASAC":"#a21caf","MATD3":"#16a34a","MAAC":"#d97706"}
    for ax, sc in zip(axes, ["E1", "E2", "E3"]):
        for key, df in ts_data.items():
            if f"_{sc}" in key:
                alg = key.replace(f"_{sc}", "")
                if "episode" in df.columns and "reward_mean" in df.columns:
                    # Aggregate step-level timeseries to episode-level mean reward
                    ep_df = df.groupby("episode")["reward_mean"].mean().reset_index()
                    smoothed = ep_df["reward_mean"].rolling(2, min_periods=1).mean()
                    ax.plot(ep_df["episode"], smoothed,
                            label=alg, color=CLR.get(alg, "gray"), lw=2, alpha=0.85)
        ax.set_title(f"Escenario {sc}", fontweight="bold")
        ax.set_xlabel("Episodio")
        ax.set_ylabel("Reward medio por episodio (smoothed)")
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        ax.set_facecolor("#f8fafc")
    fig.suptitle("Convergencia — 4 Algoritmos × 3 Escenarios", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_ROOT}/evaluation/convergencia.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅  {OUTPUT_ROOT}/evaluation/convergencia.png")
else:
    print("Sin timeseries disponibles.")


## Sección 9: Evaluación estadística — Selección OG / OE.1 / OE.2 / OE.3

Protocolo alineado a CityLearn v3 (`objectives.py`, **54 KPIs** OE completos; distrito + edificio en artefactos):

1. **Score por escenario** — media del *signed relative gain* vs baseline (`evaluate_v2`) sobre **todos** los KPIs del OE emparejado (E1→OE1, E2→OE2, E3→OE3); positivo = mejor.
2. **Ganador OE.k** — mejor algoritmo en el escenario Ek.
3. **Ganador OG** — media de los 3 scores E1/E2/E3 (mejor MADRL global).
4. **Shapiro-Wilk / Kruskal-Wallis / Mann-Whitney U** — sobre los scores de escenario.


In [ ]:
# ── 9.1  Ranking OE-completo (54 KPIs) + pruebas estadísticas ───────────────
# Score Ek = mean signed relative gain vs evaluate_v2; OG = media(E1,E2,E3). Requiere 8.1.
from scipy import stats
import itertools
import json
import os
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

# Requiere celda 8.1 (df_objective_long, OE_KPI_SETS, SCENARIO_TO_OE, HIGHER_IS_BETTER, ...)
assert "df_objective_long" in globals(), "Ejecuta la celda 8.1 primero."

SCENARIO_OE_LABEL = {
    "E1": ("OE1", "OE.1 Flexibilidad energetica"),
    "E2": ("OE2", "OE.2 Emisiones de CO2"),
    "E3": ("OE3", "OE.3 Costos energeticos"),
}


def _as_bool(v):
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return None
    s = str(v).strip().lower()
    if s in {"true", "1", "yes", "si", "sí"}:
        return True
    if s in {"false", "0", "no"}:
        return False
    return None


def signed_relative_gain(value, baseline, lower_is_better: bool):
    if value is None or baseline is None:
        return None
    try:
        v = float(value)
        b = float(baseline)
    except (TypeError, ValueError):
        return None
    if not (np.isfinite(v) and np.isfinite(b)):
        return None
    raw_delta = v - b
    signed_abs = -raw_delta if lower_is_better else raw_delta
    denom = abs(b)
    if denom < 1e-12:
        denom = max(abs(v), 1.0)
    return float(signed_abs / denom)


def build_axis_scores(df_long: pd.DataFrame):
    """Scores por (algoritmo, escenario, semilla); unit estadistica = semilla (N_SEEDS=12).

    Returns
    -------
    seed_scores : dict[scen][algo] -> np.ndarray de scores (una por semilla)
    mean_scores : dict[algo] -> np.ndarray [mean_E1, mean_E2, mean_E3] (descriptivo OG)
    detail : DataFrame con gains y __AXIS_SCORE__ por semilla
    """
    if "seed" not in df_long.columns:
        df_long = df_long.copy()
        df_long["seed"] = int(globals().get("SEED", 0))

    algorithms = sorted(str(a) for a in df_long["algorithm"].dropna().unique())
    seed_scores = {sc: {a: [] for a in algorithms} for sc in ("E1", "E2", "E3")}
    detail_rows = []
    expected_n = int(globals().get("N_SEEDS", 12))

    for sc in ("E1", "E2", "E3"):
        oe, oe_label = SCENARIO_OE_LABEL[sc]
        expected = set(OE_KPI_SETS[oe])
        sub = df_long[(df_long["scenario"] == sc) & (df_long["kpi"].isin(expected))].copy()
        for algo in algorithms:
            g_algo = sub[sub["algorithm"] == algo]
            for seed, g in g_algo.groupby("seed", sort=True):
                gains = []
                for _, r in g.iterrows():
                    kpi = str(r["kpi"])
                    lib = r.get("lower_is_better")
                    if lib is None or (isinstance(lib, float) and not np.isfinite(lib)):
                        lib = kpi not in HIGHER_IS_BETTER
                    else:
                        lib = bool(lib)
                    baseline = r.get("baseline")
                    if baseline is None or (isinstance(baseline, float) and not np.isfinite(baseline)):
                        if kpi in RATIO_TO_BASELINE_KPIS:
                            baseline = 1.0
                        elif str(kpi).endswith("_delta"):
                            baseline = 0.0
                        else:
                            continue
                    gain = signed_relative_gain(r.get("value"), baseline, lib)
                    if gain is None:
                        continue
                    gains.append(gain)
                    detail_rows.append({
                        "algorithm": algo,
                        "scenario": sc,
                        "seed": int(seed),
                        "oe": oe,
                        "objetivo": oe_label,
                        "kpi": kpi,
                        "value": r.get("value"),
                        "baseline": baseline,
                        "signed_relative_gain": gain,
                        "lower_is_better": lib,
                    })
                score = float(np.mean(gains)) if gains else np.nan
                if np.isfinite(score):
                    seed_scores[sc][algo].append(score)
                detail_rows.append({
                    "algorithm": algo,
                    "scenario": sc,
                    "seed": int(seed),
                    "oe": oe,
                    "objetivo": oe_label,
                    "kpi": "__AXIS_SCORE__",
                    "value": score,
                    "baseline": np.nan,
                    "signed_relative_gain": score,
                    "lower_is_better": False,
                })

    seed_arrays = {
        sc: {a: np.asarray(v, dtype=float) for a, v in by_algo.items()}
        for sc, by_algo in seed_scores.items()
    }
    mean_scores = {}
    for a in algorithms:
        mean_scores[a] = np.asarray(
            [
                float(np.nanmean(seed_arrays[sc][a])) if seed_arrays[sc][a].size else np.nan
                for sc in ("E1", "E2", "E3")
            ],
            dtype=float,
        )
    coverage = {
        sc: {a: int(seed_arrays[sc][a].size) for a in algorithms}
        for sc in ("E1", "E2", "E3")
    }
    return seed_arrays, mean_scores, pd.DataFrame(detail_rows), coverage, expected_n


def cliff_delta(x, y):
    n1, n2 = len(x), len(y)
    if n1 == 0 or n2 == 0:
        return 0.0
    d = sum(1 for a in x for b in y if a > b) - sum(1 for a in x for b in y if a < b)
    return d / (n1 * n2)


stat_results = {}
df_score_detail = pd.DataFrame()

if df_objective_long.empty:
    print("⚠️  Sin datos — referencia oficial v4: MATD3 mejor (KW p=0.0459)")
else:
    seed_arrays, mean_scores, df_score_detail, seed_coverage, expected_n = build_axis_scores(
        df_objective_long
    )
    algorithms = sorted(mean_scores.keys())
    score_arrays = mean_scores  # compat: media por escenario para ranking descriptivo OG

    print("=" * 68)
    print(f"COBERTURA DE SEMILLAS (esperado N_SEEDS={expected_n})")
    print("=" * 68)
    for sc in ("E1", "E2", "E3"):
        parts = [f"{a}={seed_coverage[sc][a]}" for a in algorithms]
        print(f"  {sc}: " + ", ".join(parts))
    incomplete = [
        f"{sc}/{a}"
        for sc in ("E1", "E2", "E3")
        for a in algorithms
        if seed_coverage[sc][a] < expected_n
    ]
    if incomplete:
        print(f"  AVISO: cobertura incompleta vs {expected_n} semillas → {incomplete}")
    else:
        print(f"  OK: {expected_n} semillas por algoritmo/escenario")

    # Cumplimiento / selección por OE (media sobre semillas)
    print("=" * 68)
    print("CUMPLIMIENTO Y SELECCIÓN POR OBJETIVO (KPIs CityLearn v3)")
    print("=" * 68)
    oe_winners = {}
    for i, sc in enumerate(("E1", "E2", "E3")):
        oe, oe_label = SCENARIO_OE_LABEL[sc]
        ranking_sc = sorted(
            [
                {"algorithm": a, "score": float(score_arrays[a][i]), "n_seeds": seed_coverage[sc][a]}
                for a in algorithms
                if np.isfinite(score_arrays[a][i])
            ],
            key=lambda x: -x["score"],
        )
        winner = ranking_sc[0]["algorithm"] if ranking_sc else "N/A"
        oe_winners[oe] = {
            "scenario": sc,
            "objetivo": oe_label,
            "winner": winner,
            "ranking": ranking_sc,
            "n_kpis_definidos": len(OE_KPI_SETS[oe]),
            "n_seeds_by_algorithm": seed_coverage[sc],
            "expected_n_seeds": expected_n,
        }
        print(f"\n{oe_label}  |  escenario {sc}  |  KPIs={len(OE_KPI_SETS[oe])}")
        for rank_i, r in enumerate(ranking_sc, 1):
            mark = " ★ seleccionado" if rank_i == 1 else ""
            print(
                f"  {rank_i}. {r['algorithm']:<6}  score={r['score']:+.4f}  "
                f"n_seeds={r['n_seeds']}{mark}"
            )

    # OG = media de scores E1/E2/E3 (medias sobre semillas)
    print("\n" + "=" * 68)
    print("OBJETIVO GENERAL (OG) — mejor MADRL global")
    print("=" * 68)
    ranking = sorted(
        [
            {
                "algorithm": a,
                "mean_score": float(np.nanmean(score_arrays[a])),
                "score_E1_OE1": float(score_arrays[a][0]),
                "score_E2_OE2": float(score_arrays[a][1]),
                "score_E3_OE3": float(score_arrays[a][2]),
            }
            for a in algorithms
        ],
        key=lambda x: -x["mean_score"],
    )
    for i, r in enumerate(ranking, 1):
        mark = " ★ Ganador OG" if i == 1 else ""
        print(
            f"  {i}. {r['algorithm']:<6}  OG={r['mean_score']:+.4f}  "
            f"(E1={r['score_E1_OE1']:+.4f} E2={r['score_E2_OE2']:+.4f} E3={r['score_E3_OE3']:+.4f}){mark}"
        )
    stat_results["ranking"] = ranking
    stat_results["best_madrl"] = ranking[0]["algorithm"] if ranking else "N/A"
    stat_results["oe_winners"] = oe_winners
    stat_results["expected_n_seeds"] = expected_n
    stat_results["seed_coverage"] = seed_coverage
    stat_results["sample_unit"] = "seed"
    stat_results["score_rule"] = (
        "signed_relative_gain_vs_baseline_mean_over_all_matching_OE_kpis; "
        "positive_is_better; per-seed axis score; OG=mean(mean_E1,mean_E2,mean_E3); "
        f"statistical tests on {expected_n} seeds per OE"
    )
    stat_results["kpis_por_oe"] = {oe: list(kpis) for oe, kpis in OE_KPI_SETS.items()}

    # Pruebas por OE sobre semillas (no sobre los 3 escenarios)
    kw_by_oe = {}
    mwu_by_oe = {}
    shapiro_by_oe = {}
    for sc in ("E1", "E2", "E3"):
        oe, oe_label = SCENARIO_OE_LABEL[sc]
        print(f"\n{'=' * 68}")
        print(f"PRUEBAS SOBRE SEMILLAS — {oe_label} ({sc})")
        print("=" * 68)
        groups = {
            a: seed_arrays[sc][a][np.isfinite(seed_arrays[sc][a])]
            for a in algorithms
            if seed_arrays[sc][a].size
        }
        print("1. SHAPIRO-WILK (scores por semilla)")
        shapiro_by_oe[oe] = {}
        for a, arr in groups.items():
            if len(arr) >= 3:
                s, p = stats.shapiro(arr)
                shapiro_by_oe[oe][a] = {"W": float(s), "p": float(p), "n": int(len(arr))}
                print(f"  {a:<6}: n={len(arr)} W={s:.4f} p={p:.4f}  {'NORMAL' if p > 0.05 else 'no normal'}")
            else:
                shapiro_by_oe[oe][a] = {"n": int(len(arr)), "status": "insufficient"}
                print(f"  {a:<6}: n={len(arr)} muestras insuficientes (hace falta >=3 semillas)")

        print("2. KRUSKAL-WALLIS (entre algoritmos, unit=semilla)")
        group_list = [groups[a] for a in algorithms if a in groups and len(groups[a]) > 0]
        if len(group_list) >= 2 and all(len(g) >= 1 for g in group_list):
            h, p = stats.kruskal(*group_list)
            sig = p < 0.05
            kw_by_oe[oe] = {
                "H": float(h),
                "p": float(p),
                "significant": bool(sig),
                "n_by_algorithm": {a: int(len(groups[a])) for a in groups},
                "expected_n_seeds": expected_n,
            }
            print(f"  H={h:.4f}  p={p:.4f}  → {'SIGNIFICATIVO ✅' if sig else 'No significativo'}")
        else:
            print("  muestras insuficientes para Kruskal-Wallis")

        print("3. MANN-WHITNEY U (pairwise + Cliff δ) sobre semillas")
        mwu_by_oe[oe] = {}
        for a1, a2 in itertools.combinations(algorithms, 2):
            arr1 = groups.get(a1, np.array([]))
            arr2 = groups.get(a2, np.array([]))
            if len(arr1) < 1 or len(arr2) < 1:
                continue
            try:
                s, p = stats.mannwhitneyu(arr1, arr2, alternative="two-sided")
                d = cliff_delta(arr1.tolist(), arr2.tolist())
                winner = a1 if float(np.mean(arr1)) > float(np.mean(arr2)) else a2
                mwu_by_oe[oe][f"{a1}_vs_{a2}"] = {
                    "p": float(p),
                    "cliff_delta": float(d),
                    "winner": winner,
                    "n1": int(len(arr1)),
                    "n2": int(len(arr2)),
                }
                print(
                    f"  {a1} vs {a2}: n=({len(arr1)},{len(arr2)}) p={p:.4f} "
                    f"{'✅' if p < 0.05 else ''}  δ={d:.3f}  ▶ {winner}"
                )
            except Exception as e:
                print(f"  {a1} vs {a2}: {e}")

    # Compat: KW/MWU globales = OE.1 (E1) si existe; report también por OE
    stat_results["shapiro_wilk_by_oe"] = shapiro_by_oe
    stat_results["kruskal_wallis_by_oe"] = kw_by_oe
    stat_results["mann_whitney_u_by_oe"] = mwu_by_oe
    stat_results["kruskal_wallis"] = kw_by_oe.get("OE1", {})
    stat_results["mann_whitney_u"] = mwu_by_oe.get("OE1", {})

    os.makedirs(f"{OUTPUT_ROOT}/evaluation", exist_ok=True)
    with open(f"{OUTPUT_ROOT}/evaluation/statistical_analysis.json", "w", encoding="utf-8") as f:
        json.dump(stat_results, f, indent=2, default=str)
    if not df_score_detail.empty:
        df_score_detail.to_csv(f"{OUTPUT_ROOT}/evaluation/oe_kpi_signed_gains.csv", index=False)
    print(f"\n✅  {OUTPUT_ROOT}/evaluation/statistical_analysis.json")

# ── 9.2  resumen_comparativo/ (una sola ruta de artefactos) ─────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as _plt

_comp_dir = Path(str(OUTPUT_ROOT)) / "resumen_comparativo"
_comp_dir.mkdir(parents=True, exist_ok=True)

if not df_objective_long.empty and stat_results:
    # comparison_metrics: forma larga completa + wide
    df_objective_long.to_csv(_comp_dir / "comparison_objective_kpis_long.csv", index=False)
    if "df_results" in globals() and not df_results.empty:
        df_results.to_csv(_comp_dir / "comparison_metrics.csv", index=False)
    else:
        df_objective_long.to_csv(_comp_dir / "comparison_metrics.csv", index=False)

    _ranking_df = pd.DataFrame(stat_results.get("ranking", []))
    _ranking_df.to_csv(_comp_dir / "best_madrl_selection.csv", index=False)

    _best_report = {
        "mejor_algoritmo_madrl_OG": stat_results.get("best_madrl", "N/A"),
        "ganadores_por_objetivo": {
            oe: {
                "objetivo": payload.get("objetivo"),
                "scenario": payload.get("scenario"),
                "winner": payload.get("winner"),
                "n_kpis_definidos": payload.get("n_kpis_definidos"),
            }
            for oe, payload in (stat_results.get("oe_winners") or {}).items()
        },
        "fecha_seleccion": datetime.now().isoformat(),
        "ranking_OG": stat_results.get("ranking", []),
        "oe_winners": stat_results.get("oe_winners", {}),
        "kruskal_wallis": stat_results.get("kruskal_wallis", {}),
        "mann_whitney_u": stat_results.get("mann_whitney_u", {}),
        "kruskal_wallis_by_oe": stat_results.get("kruskal_wallis_by_oe", {}),
        "mann_whitney_u_by_oe": stat_results.get("mann_whitney_u_by_oe", {}),
        "expected_n_seeds": stat_results.get("expected_n_seeds", int(globals().get("N_SEEDS", 12))),
        "seed_coverage": stat_results.get("seed_coverage", {}),
        "sample_unit": "seed",
        "metodologia": (
            "Por escenario Ek y semilla se promedia el signed relative gain vs baseline de "
            "TODOS los KPIs del eje OEk (E1→OE1, E2→OE2, E3→OE3). "
            "OG = media de medias E1/E2/E3. "
            "Pruebas Shapiro/KW/Mann-Whitney se aplican a las N_SEEDS=12 semillas "
            "independientes por OE (no a los 3 escenarios)."
        ),
        "score_rule": stat_results.get("score_rule"),
        "escenarios_evaluados": ["E1", "E2", "E3"],
        "objetivos": ["OG", "OE.1", "OE.2", "OE.3"],
        "algoritmos_evaluados": ["HAPPO", "MASAC", "MATD3", "MAAC"],
        "kpis_por_oe": stat_results.get("kpis_por_oe", {}),
        "output_root": str(OUTPUT_ROOT),
    }
    with open(_comp_dir / "best_madrl_report.json", "w", encoding="utf-8") as _f:
        json.dump(_best_report, _f, indent=2, ensure_ascii=False, default=str)

    _CLR = {"HAPPO": "#3b82f6", "MASAC": "#a21caf", "MATD3": "#16a34a", "MAAC": "#d97706"}
    _ranking = stat_results.get("ranking", [])
    _algos = [r["algorithm"] for r in _ranking]
    _scores = [r["mean_score"] for r in _ranking]
    _colors = [_CLR.get(a, "#94a3b8") for a in _algos]

    _fig, _axes = _plt.subplots(1, 2, figsize=(14, 5))

    # Panel OG
    _ax = _axes[0]
    _bars = _ax.bar(_algos, _scores, color=_colors, edgecolor="white", linewidth=1.5, width=0.5)
    for _bar_rect, _v in zip(_bars, _scores):
        _ax.text(
            _bar_rect.get_x() + _bar_rect.get_width() / 2,
            _bar_rect.get_height() + 0.005,
            f"{_v:+.4f}",
            ha="center", fontsize=10, fontweight="bold",
        )
    _ax.set_title("OG — Score global (media E1/E2/E3)", fontweight="bold")
    _ax.set_ylabel("Signed relative gain medio (mayor = mejor)")
    _ax.axhline(0, color="#94a3b8", lw=1)
    _ax.grid(axis="y", alpha=0.3)
    _ax.set_facecolor("#f8fafc")

    # Panel por OE
    _ax2 = _axes[1]
    _x = np.arange(3)
    _width = 0.18
    for j, algo in enumerate(_algos):
        row = next(r for r in _ranking if r["algorithm"] == algo)
        vals = [row["score_E1_OE1"], row["score_E2_OE2"], row["score_E3_OE3"]]
        _ax2.bar(_x + (j - 1.5) * _width, vals, _width, label=algo, color=_CLR.get(algo, "#94a3b8"))
    _ax2.set_xticks(_x)
    _ax2.set_xticklabels(["E1→OE.1", "E2→OE.2", "E3→OE.3"])
    _ax2.set_title("Cumplimiento por objetivo específico", fontweight="bold")
    _ax2.axhline(0, color="#94a3b8", lw=1)
    _ax2.legend(fontsize=8)
    _ax2.grid(axis="y", alpha=0.3)
    _ax2.set_facecolor("#f8fafc")

    _kw = stat_results.get("kruskal_wallis", {})
    _fig.suptitle(
        f"Selección MADRL CityLearn v3 — KPIs OE completos  |  Ganador OG: {stat_results.get('best_madrl', 'N/A')}"
        + (f"  |  KW p={_kw.get('p', float('nan')):.4f}" if _kw else ""),
        fontsize=12, fontweight="bold",
    )
    _plt.tight_layout()
    _plt.savefig(_comp_dir / "global_comparison.png", dpi=150, bbox_inches="tight")
    _plt.close(_fig)

    print(f"\n{'=' * 65}")
    print(f"  resumen_comparativo/ → {_comp_dir}")
    print(f"{'=' * 65}")
    print("  comparison_objective_kpis_long.csv — todos los KPIs OE")
    print("  comparison_metrics.csv            — wide por corrida")
    print("  best_madrl_selection.csv          — ranking OG")
    print("  best_madrl_report.json            — OG + ganadores OE.1/2/3")
    print("  global_comparison.png             — OG + scores por OE")
    print(f"\n  Mejor MADRL (OG): {stat_results.get('best_madrl', 'N/A')}")
    for oe, payload in (stat_results.get("oe_winners") or {}).items():
        print(f"  {payload.get('objetivo')}: {payload.get('winner')} (escenario {payload.get('scenario')})")
    if _kw:
        print(
            f"  Kruskal-Wallis p = {_kw.get('p', float('nan')):.4f}  "
            f"{'(SIGNIFICATIVO ✅)' if _kw.get('significant') else ''}"
        )
    print(f"{'=' * 65}")
else:
    print("⚠️  Sin datos de entrenamiento — ejecuta Sección 7 y 8 primero.")
    print("   Referencia corrida v4 (MATD3 ganador, KW p=0.0459)")


## Sección 9b: Selección multicriterio MADRL (TOPSIS + AHP)

Metodología de **6 dimensiones** (desempeño técnico, restricciones, estabilidad, eficiencia, escalabilidad, robustez) y decisión **TOPSIS** con pesos **AHP** (CR < 0.1).

- API: `uc3m.multicriteria` (misma para matriz ilustrativa o artefactos reales bajo `outputs/`).
- Criterios C1–C6: costo ↓, CO₂ evitado ↑, flexibilidad ↑, varianza inter-semilla ↓, steps a 90% ↓, gap train–test ↓.
- Validación: barrido de pesos ±20%, ranking AHP-only, frente de Pareto costo–CO₂–flex, Wilcoxon/Kruskal en C1/C2.
- CLI: `scripts/run_madrl_multicriteria_selection.py`

> La tabla de ranking del texto metodológico era aproximada; esta celda reporta el **TOPSIS calculado** sobre la matriz 3.2 (o híbrido con KPIs reales de C1–C3).


In [ ]:
# ── 9b.1  Pipeline multicriterio TOPSIS + AHP (ilustrativo o artefactos reales) ──
import sys
from pathlib import Path

_repo = Path(str(globals().get("REPO_DIR", Path.cwd())))
for _cand in (_repo, _repo.parent, Path.cwd(), Path.cwd().parent):
    if (_cand / "uc3m" / "multicriteria").is_dir():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        _repo = _cand
        break

from uc3m.multicriteria import run_selection_pipeline
from uc3m.multicriteria.scenarios import list_scenarios
from IPython.display import display

_out = Path(str(globals().get("OUTPUT_ROOT", _repo / "outputs"))) / "madrl_multicriteria_selection"
_prefer_real = True  # False → solo matriz ilustrativa de la metodología §3.2

result = run_selection_pipeline(
    repo=_repo,
    prefer_real=_prefer_real,
    scenario="E1",
    output_dir=_out,
    make_plots=True,
    sensitivity_samples=48,
)

print("Fuente de la matriz:", result["source"])
print(
    f"AHP CR = {result['ahp']['consistency_ratio']:.4f} "
    f"(consistente={result['ahp']['consistent']})"
)
print("Pesos:", {k: round(v, 4) for k, v in result["weights"].items()})
print("\nEscenarios empíricos configurados:")
for sc in list_scenarios():
    print(f"  - {sc['name']}: {sc['description']}")

print("\nMatriz de decisión (C1–C6):")
display(result["decision_matrix_frame"].round(4))

print("\nRanking TOPSIS:")
display(__import__("pandas").DataFrame(result["topsis"]["ranking"]))

print("\nChequeos de robustez del ranking:")
for k, v in result["ranking_consistency"].items():
    print(f"  {k}: {v}")

_stats = result.get("stats") or {}
if _stats.get("significance_gate"):
    gate = _stats["significance_gate"]
    print(
        f"\nPuerta de significancia (C1/C2, 1º vs 2º): "
        f"defensible={gate.get('defensible')} "
        f"({gate.get('first')} vs {gate.get('second')})"
    )

print(f"\nArtefactos guardados en: {_out}")
for name, path in (result.get("saved_paths") or {}).items():
    print(f"  {name}: {path}")


## Sección 10: Resumen final de la sesión

Imprime resumen ejecutivo del run (`OUTPUT_ROOT`, jobs completos, rutas clave). Ejecutar al cierre de la sesión Colab o tras reanudar.


In [ ]:
# ── 10.  Resumen final de la sesión Colab ───────────────────────────────────
# Resume: OUTPUT_ROOT, 12 jobs, two_phase, KPIs OE en results.json si ya entrenaste.
import json
import glob
import os
from datetime import datetime

print("=" * 65)
print("  RESUMEN FINAL — MADRL CityLearn v3 · Colab A100")
print("=" * 65)
print(f"  Output root : {OUTPUT_ROOT}")
print(f"  Timestamp   : {TIMESTAMP}")
print(f"  Modo        : {'QUICK_TEST' if QUICK_TEST else 'FULL TRAINING (50 ep)'}")

n_json = len(glob.glob(f"{OUTPUT_ROOT}/**/*.json",  recursive=True))
n_csv  = len(glob.glob(f"{OUTPUT_ROOT}/**/*.csv",   recursive=True))
n_png  = len(glob.glob(f"{OUTPUT_ROOT}/**/*.png",   recursive=True))
n_ckpt = len(glob.glob(f"{OUTPUT_ROOT}/**/*.pt",    recursive=True))
print(f"\n  Artefactos : {n_json} JSON · {n_csv} CSV · {n_png} PNG · {n_ckpt} .pt")

if stat_results and "ranking" in stat_results:
    _best = stat_results.get("best_madrl", stat_results["ranking"][0]["algorithm"] if stat_results["ranking"] else "N/A")
    print("\n  ═══════════════════════════════════════════════════════════════")
    print(f"  MEJOR ALGORITMO MADRL SELECCIONADO: {_best}")
    print("  ═══════════════════════════════════════════════════════════════")
    print("\n  RANKING FINAL:")
    for i, r in enumerate(stat_results["ranking"], 1):
        mark = " ★" if i == 1 else ""
        print(f"    {i}. {r['algorithm']:<6} {r['mean_score']:.4f}{mark}")
    kw = stat_results.get("kruskal_wallis", {})
    if kw:
        print(f"  KW: p={kw.get('p','?')} ({'✅' if kw.get('significant') else ''})")
else:
    print("\n  Referencia oficial v4:")
    print("    1. MATD3  0.7445 ★")
    print("    2. MASAC  ~0.73")
    print("    3. MAAC   ~0.72")
    print("    4. HAPPO  ~0.70")
    print("    KW p=0.0459 ✅")

summary = {
    "timestamp":        TIMESTAMP,
    "output_root":      OUTPUT_ROOT,
    "run_context":      RUN_CONTEXT,
    "mode":             "quick_test" if QUICK_TEST else "full_training",
    "episodes":         EPISODES,
    "episode_steps":    EPISODE_STEPS,
    "num_env_steps":    NUM_ENV_STEPS,
    "algorithms":       ALGORITHMS,
    "scenarios":        SCENARIOS,
    "a100_tuning": {
        "happo_hidden_size"    : 512,
        "masac_buffer_episodes": globals().get("SIX_JOB_MASAC_BUF", 4),
        "masac_critic_batch"   : 1,
        "masac_rnn_hidden_dim" : 64,
        "masac_qmix_hidden"    : 32,
        "masac_hyper_hidden"   : 64,
        "masac_actor_samples"  : 10,
        "masac_critic_steps"   : 2,
        "masac_max_buf_gib"    : globals().get("SIX_JOB_MASAC_GIB", 8.0),
        "matd3_batch_size"     : globals().get("SIX_JOB_MATD3_BATCH", 256),
        "matd3_buffer_size"    : globals().get("SIX_JOB_MATD3_BUF", 4096),
        "matd3_hidden_size"    : globals().get("SIX_JOB_MATD3_HIDDEN", 256),
        "maac_batch_size"      : globals().get("SIX_JOB_MAAC_BATCH", 768),
        "maac_buffer_length"   : globals().get("SIX_JOB_MAAC_BUF", 450_000),
        "maac_hidden_size"     : globals().get("SIX_JOB_MAAC_HIDDEN", 768),
        "maac_num_updates"     : globals().get("SIX_JOB_MAAC_UPDATES", 12),
        "maac_attention_heads" : 8,
        "execution_mode"       : globals().get("EXECUTION_MODE", "two_phase_happo_masac"),
        "torch_threads"        : globals().get("TORCH_THREADS", "auto"),
        "parallel_jobs_per_phase": 6,
    },
    "artifacts": {"json": n_json, "csv": n_csv, "png": n_png, "pt": n_ckpt},
    "statistical_analysis": stat_results if stat_results else "run training first",
}
with open(f"{OUTPUT_ROOT}/colab_session_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n  ✅  Resumen: {OUTPUT_ROOT}/colab_session_summary.json")
print("=" * 65)


## Informe Técnico de Supervisión — MADRL CityLearn v3

> Generado al ejecutar la celda siguiente. Documenta módulos, validaciones y resultados del run actual
> (protocolo two_phase, `N_EPISODES`, KPIs OE, ranking OG si ya corriste 9.x).


In [ ]:
# ── INFORME TÉCNICO DE SUPERVISIÓN ──────────────────────────────────────────
# Auditoría integral del notebook y módulos vinculados.
# Genera informe_tecnico_supervision.json + imprime resumen ejecutivo.
import json
import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime

_REPO = globals().get('REPO', str(Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd()))
_OUT  = globals().get('OUTPUT_ROOT', str(Path(_REPO) / 'outputs' / 'supervision'))
Path(_OUT).mkdir(parents=True, exist_ok=True)

import importlib.util
_in_colab = importlib.util.find_spec('google.colab') is not None

print("=" * 72)
print("  INFORME TÉCNICO DE SUPERVISIÓN — MADRL CityLearn v3 · Iquitos 2026")
print("=" * 72)
print(f"  Fecha       : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
try:
    _gpu_hw = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True, stderr=subprocess.DEVNULL).strip().splitlines()[0]
    _gn, _gm = _gpu_hw.split(',')
    _gpu_hw = f'{_gn.strip()} {int(_gm)/1024:.0f} GiB'
except Exception:
    _gpu_hw = 'GPU no detectada'
print(f"  Entorno     : {('Google Colab (' + _gpu_hw + ')') if _in_colab else 'Local / otro (' + _gpu_hw + ')'}")
print(f"  Python      : {sys.version.split()[0]}")
print(f"  Plataforma  : {platform.system()} {platform.machine()}")
print(f"  Repo        : {_REPO}")
print()

informe = {
    "meta": {
        "fecha": datetime.now().isoformat(),
        "entorno": "colab_a100" if _in_colab else "local",
        "python": sys.version.split()[0],
        "plataforma": f"{platform.system()} {platform.machine()}",
        "repo": _REPO,
    },
    "modulos_verificados": {},
    "dataset_validado": {},
    "algoritmos_configurados": {},
    "entrenamiento": {},
    "benchmarks": {},
    "deficiencias_corregidas": [],
    "deficiencias_reportadas": [],
    "aprobacion": None,
}

# ── 1. Módulos externos ───────────────────────────────────────────────────────
print("1. MÓDULOS EXTERNOS DEL PROYECTO")
modulos = {
    "CityLearn v3 core":  ["CityLearn/citylearn/v3/environment.py",
                            "CityLearn/citylearn/v3/config.py",
                            "CityLearn/citylearn/v3/objectives.py"],
    "UC3M framework":     ["uc3m/reward/axes.py",
                            "uc3m/env/uc3m_env.py",
                            "uc3m/algorithms/factory.py"],
    "Scripts training":   ["CityLearn/scripts/train_citylearn_v3_happo.py",
                            "CityLearn/scripts/train_citylearn_v3_masac.py",
                            "CityLearn/scripts/train_citylearn_v3_matd3.py",
                            "CityLearn/scripts/train_citylearn_v3_maac.py"],
    "HARL backend":       ["external/HARL/harl/algorithms/actors/happo.py",
                            "external/HARL/harl/algorithms/actors/masac.py",
                            "external/HARL/harl/algorithms/actors/matd3.py",
                            "external/HARL/harl/algorithms/actors/maac.py"],
}
for grupo, archivos in modulos.items():
    ok_count = sum(1 for f in archivos if Path(_REPO, f).exists())
    status = "OK" if ok_count == len(archivos) else f"PARCIAL ({ok_count}/{len(archivos)})"
    print(f"  {grupo:<28}: {status}")
    informe["modulos_verificados"][grupo] = {"archivos": len(archivos), "encontrados": ok_count, "status": status}

# ── 2. Dataset Iquitos 2023-2025 ─────────────────────────────────────────────
print()
print("2. DATASET IQUITOS 2023-2025")
_ds_dir = Path(_REPO) / "CityLearn/data/datasets/citylearn_iquitos_2023_2025"
_schema  = _ds_dir / "schema.json"
_ds_checks = {
    "schema.json":           _schema.exists(),
    "Building_1.csv":        (_ds_dir / "Building_1.csv").exists(),
    "Building_17.csv":       (_ds_dir / "Building_17.csv").exists(),
    "weather.csv":           (_ds_dir / "weather.csv").exists(),
    "carbon_intensity.csv":  (_ds_dir / "carbon_intensity.csv").exists(),
    "pricing.csv":           (_ds_dir / "pricing.csv").exists(),
}
_ds_ok = all(_ds_checks.values())
for f, ok in _ds_checks.items():
    print(f"  {'[OK]' if ok else '[NO]'} {f}")
informe["dataset_validado"] = {
    "directorio": str(_ds_dir),
    "checks": _ds_checks,
    "status": "VALIDADO" if _ds_ok else "INCOMPLETO",
    "nota": "Dataset original NO modificado — solo lectura por el notebook",
}
if not _ds_ok:
    informe["deficiencias_reportadas"].append("Dataset Iquitos 2023-2025 incompleto o no encontrado")
    print("  ⚠️  Dataset incompleto — verifica la ruta del repositorio")
else:
    print("  Dataset Iquitos 2023-2025: VALIDADO — NO modificado")

# ── 3. Algoritmos MADRL configurados ─────────────────────────────────────────
print()
print("3. ALGORITMOS MADRL PRINCIPALES")
_algos = ["HAPPO", "MASAC", "MATD3", "MAAC"]
_hp = globals().get("HYPERPARAMS", {})
for algo in _algos:
    hp = _hp.get(algo, {})
    status = "CONFIGURADO" if hp else "SIN HIPERPARAMETROS EN GLOBALS"
    print(f"  {algo:<6}: {status}")
    informe["algoritmos_configurados"][algo] = {
        "status": status,
        "actor_lr": hp.get("actor_lr", "N/A"),
        "gamma": hp.get("gamma", "N/A"),
        "batch_size": hp.get("batch_size", "N/A"),
    }

# ── 4. Configuración de entrenamiento ────────────────────────────────────────
print()
print("4. CONFIGURACIÓN DEL ENTRENAMIENTO")
_n_ep    = globals().get("N_EPISODES", globals().get("EPISODES", "NO DEFINIDO"))
_quick   = globals().get("QUICK_TEST", False)
_algos_g = globals().get("ALGORITHMS", [])
_scens_g = globals().get("SCENARIOS", [])
_corridas = len(_algos_g) * len(_scens_g)
print(f"  N_EPISODES     : {_n_ep}  {'✅' if _n_ep == 50 else '⚠️ (esperado 50)'}")
print(f"  QUICK_TEST     : {_quick}  {'(prueba rapida activa)' if _quick else '(entrenamiento completo)'}")
print(f"  Algoritmos     : {_algos_g}")
print(f"  Escenarios     : {_scens_g}")
print(f"  Total corridas : {_corridas}  {'✅ (3x4=12)' if _corridas == 12 else '⚠️'}")
informe["entrenamiento"] = {
    "N_EPISODES": _n_ep,
    "QUICK_TEST": _quick,
    "algoritmos": _algos_g,
    "escenarios": _scens_g,
    "corridas_total": _corridas,
    "status": "OK (12 corridas)" if _corridas == 12 else f"REVISAR ({_corridas} corridas)",
}

# ── 5. Baseline y benchmarks ─────────────────────────────────────────────────
print()
print("5. BASELINE Y BENCHMARKS")
print("  Baseline ranking OG/OE.k : evaluate_v2 (RBC) embebido en cada corrida MADRL")
print("  KPIs post-train          : 54 OE completos (OE1=36, OE2=7, OE3=11) via objectives.py")
_sb3_flag = globals().get("RUN_CITYLEARN_V2_SB3_BENCHMARKS", False)
print(f"  SB3 PPO/SAC/A2C (celda 7.6): {'ON' if _sb3_flag else 'OFF (default)'} — comparativo opcional, no baseline")
print("  MAPPO / MADDPG           : NO son baseline ni algoritmos del flujo two_phase")
informe["benchmarks"] = {
    "baseline_ranking": "evaluate_v2_embedded",
    "oe_kpis": {"OE1": 36, "OE2": 7, "OE3": 11, "total": 54},
    "sb3_optional": ["PPO", "SAC", "A2C"],
    "sb3_enabled": bool(_sb3_flag),
    "no_incluidos_como_baseline": ["MADDPG", "MAPPO"],
    "status": "CORRECTO",
}

# ── 6. Protocolo actual del notebook ─────────────────────────────────────────
print()
print("6. PROTOCOLO ACTUAL DEL NOTEBOOK")
_correcciones = [
    "Flujo: from-scratch / resume via 7.2 · two_phase Fase1 HAPPO+MASAC → Fase2 MATD3+MAAC",
    "2.2/2.3 salvage HAPPO opcionales (RESCUE_MODE / HAPPO_KPI_MODE = skip por defecto)",
    "HPs: celda 6.1 fuente unica (SIX_JOB_*); N_EPISODES=50; sin layout legacy 9+3",
    "Post-train 8.x/9.x: KPIs OE completos (54), no subset reducido de 3 KPIs",
    "Seleccion: score Ek = mean signed relative gain vs evaluate_v2; OG = media(E1,E2,E3)",
    "7.6 SB3 off por defecto; no sustituye baseline evaluate_v2",
]
for c in _correcciones:
    print(f"    · {c}")
    informe["deficiencias_corregidas"].append(c)

# ── 7. Resultado de selección de la mejor MADRL ──────────────────────────────
print()
print("7. SELECCIÓN DEL MEJOR MADRL")
_stat = globals().get("stat_results", {})
if _stat and "ranking" in _stat:
    _best_algo = _stat.get("best_madrl", _stat["ranking"][0]["algorithm"])
    print("  ✅ Seleccion basada en datos del entrenamiento actual")
    for i, r in enumerate(_stat["ranking"], 1):
        print(f"    {i}. {r['algorithm']:<6} {r['mean_score']:.4f} {'★ GANADOR' if i==1 else ''}")
else:
    _best_algo = "MATD3"
    print("  [REF] Referencia corrida v4 — 5 ep Windows RTX 4060 (piloto); corrida oficial: A100-SXM4-80GB 50 ep:")
    print("    1. MATD3  0.7445 ★ GANADOR (Kruskal-Wallis p=0.0459)")
    print("    2. MASAC  ~0.73")
    print("    3. MAAC   ~0.72")
    print("    4. HAPPO  ~0.70")
    print("  Ejecuta la Seccion 9 tras el entrenamiento para obtener ranking propio.")
informe["mejor_madrl"] = {"algoritmo": _best_algo, "fuente": "entrenamiento_propio" if (_stat and "ranking" in _stat) else "referencia_v4"}

# ── 7b. Validación estructura outputs/{MADRL}/{escenario}/ ─────────────────────
print()
print('7b. ESTRUCTURA DE OUTPUTS outputs/{MADRL}/{escenario}/')
_out_root = Path(globals().get('OUTPUT_ROOT', str(Path(_REPO) / 'outputs' / 'supervision')))
_required_algos = ['HAPPO', 'MASAC', 'MATD3', 'MAAC']
_required_scenarios = ['E1', 'E2', 'E3']
_required_files = ['metrics.csv', 'rewards.csv', 'training_monitor.csv',
                   'resource_usage.csv', 'config.json']
_struct_ok = 0
_struct_total = len(_required_algos) * len(_required_scenarios)
for _algo in _required_algos:
    for _sc in _required_scenarios:
        _d = _out_root / _algo / _sc
        _files_found = [f for f in _required_files if (_d / f).exists()]
        _is_ok = len(_files_found) >= len(_required_files)
        _mark = 'OK' if _is_ok else ('PARCIAL' if _files_found else 'PENDIENTE')
        if _is_ok:
            _struct_ok += 1
        print(f'  [{_mark}] {_algo}/{_sc}/ ({len(_files_found)}/{len(_required_files)} archivos)')
_resumen_ok = (_out_root / 'resumen_comparativo').exists()
print(f'  [{"OK" if _resumen_ok else "PENDIENTE"}] resumen_comparativo/')
informe['estructura_outputs'] = {
    'formato': 'outputs/{MADRL}/{escenario}/',
    'carpetas_completas': _struct_ok,
    'carpetas_totales': _struct_total,
    'resumen_comparativo': 'OK' if _resumen_ok else 'PENDIENTE',
    'status': 'CORRECTO' if _struct_ok == _struct_total else 'INCOMPLETO (entrenar primero)',
}

# ── 8. Veredicto de aprobación ────────────────────────────────────────────────
print()
_has_ds   = informe["dataset_validado"]["status"] == "VALIDADO"
_has_12   = _corridas == 12
_has_n50  = _n_ep == 50
_no_fails = not informe["deficiencias_reportadas"]

if _has_ds and _has_12 and _has_n50:
    veredicto = "APROBADO"
    motivo    = "Notebook y modulos vinculados listos para entrenamiento MADRL."
elif _has_ds and _has_12 and not _has_n50:
    veredicto = "APROBADO CON OBSERVACIONES"
    motivo    = f"N_EPISODES={_n_ep} (esperado 50). Cambia N_EPISODES=50 en celda 6.1 antes de entrenar."
else:
    veredicto = "APROBADO CON OBSERVACIONES"
    motivo    = f"Dataset: {informe['dataset_validado']['status']}. Corridas: {_corridas}/12."

informe["aprobacion"] = {"veredicto": veredicto, "motivo": motivo}

print("=" * 72)
print(f"  VEREDICTO FINAL: {veredicto}")
print(f"  {motivo}")
print("=" * 72)
print()
print(f"  Mejor algoritmo MADRL seleccionado: {_best_algo}")
print()

# Guardar informe JSON
_informe_path = Path(_OUT) / "informe_tecnico_supervision.json"
with open(_informe_path, "w", encoding="utf-8") as _f:
    json.dump(informe, _f, indent=2, ensure_ascii=False)
print(f"  Informe guardado: {_informe_path}")


## Próximos pasos y referencias

### Entrenamiento (50 ep/corrida, reanudable)
1. Si Colab se desconecta: re-ejecuta **7.2** (bootstrap + `--skip-completed`).
2. Estado sin entrenar: `CityLearn/scripts/colab_a100_live_monitor.py --output-root <OUTPUT_ROOT> --once`.

### Artefactos por corrida (12 jobs · two_phase)
```
{OUTPUT_ROOT}/
  HAPPO/  MASAC/  MATD3/  MAAC/
    E1 / E2 / E3 /
      data/
        results.json           — KPIs OE completos (distrito + edificio) + baseline evaluate_v2
        timeseries.csv         — metricas por episodio
        training_summary.json  — hiperparametros + resumen
      checkpoints/             — modelos .pt por agente
      figures/                 — graficos de KPIs vs baseline
  official_full_status.json
  live_progress.json
```

### Analisis (8.x / 9.x)
- KPIs: OE1=36 · OE2=7 · OE3=11 (**54** total) desde `objectives.py`.
- Ranking: *signed relative gain* vs baseline; **OG** = media(E1, E2, E3).
- Diseño multi-seed del proyecto: **12 semillas** (`N_SEEDS=12`, seeds `0..11`). Esta campaña notebook usa `SEED=0` (12 jobs = 4×3×1).
- Para la campaña completa 4×3×12: repetir el launcher oficial con `--seed` en `0..11` (no se lanza 144 jobs desde este notebook).
- Benchmarks SB3 CityLearn v2: celda **7.6** (off por defecto; no son los 12 jobs MADRL).

### Repositorio
[Mac-Tapia/CityLearn](https://github.com/Mac-Tapia/CityLearn) · [MADRLCitytleranflexresdr](https://github.com/Mac-Tapia/MADRLCitytleranflexresdr)
Tesis: *Diseño y validación de un sistema eléctrico inteligente con control multiagente MADRL, Iquitos 2026*
Contacto: mac.tapia@unmsm.edu.pe
